# Quanvolutional Experiments Notebook

This notebook is organized into focused experiment blocks so results are easier to reproduce and compare.

## Sections
1. Experiment A — Classical vs Hybrid baseline
2. Experiment B — Patch strategy study
3. Experiment C — Measurement function study
4. Experiment D — Entanglement topology study
5. Sanity-check test run

> Recommendation: run cells top-to-bottom in a fresh kernel for reproducible outputs.



In [19]:
# ============================================================
# Quanvolution with AMPLITUDE ENCODING (2 qubits, 2x2 patch -> 4 features)
# + multiple filters + H2-style classifier
# ============================================================

from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
import pennylane as qml

# -----------------------
# Reproducibility and runtime setup
# -----------------------
def set_global_seed(seed: int) -> None:
    """Set deterministic seeds for NumPy and PyTorch."""
    np.random.seed(seed)
    torch.manual_seed(seed)


seed = 0
set_global_seed(seed)

device_torch = torch.device("cuda" if torch.cuda.is_available() else "cpu")

n_epochs = 100
n_layers = 1
n_filters = 2

n_train = 1200
n_test = 300

batch_size = 4
adam_lr = 1e-3
weight_decay = 1e-4

SAVE_DIR = Path("quanv_cache")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

PREPROCESS = True
circuit_name = "ampenc_2q_cry_star"

# -----------------------
# Load MNIST
# -----------------------
transform = T.Compose([T.ToTensor()])

train_ds_full = torchvision.datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_ds_full  = torchvision.datasets.MNIST(root="data", train=False, download=True, transform=transform)

# (Recommended) shuffle indices so you don't take the first samples
rng = np.random.RandomState(seed)
train_idx = rng.permutation(len(train_ds_full))[:n_train]
test_idx  = rng.permutation(len(test_ds_full))[:n_test]

train_images = torch.stack([train_ds_full[i][0] for i in train_idx], dim=0)
train_labels = torch.tensor([train_ds_full[i][1] for i in train_idx], dtype=torch.long)

test_images = torch.stack([test_ds_full[i][0] for i in test_idx], dim=0)
test_labels = torch.tensor([test_ds_full[i][1] for i in test_idx], dtype=torch.long)

train_images_nhwc = train_images.permute(0, 2, 3, 1).contiguous()
test_images_nhwc  = test_images.permute(0, 2, 3, 1).contiguous()

# -----------------------
# Quantum device (CHANGED: 2 qubits)
# -----------------------
n_wires = 2  # NEW
qdev = qml.device("default.qubit", wires=n_wires)  # CHANGED

# -----------------------
# Parameters per filter (example: simple 2-qubit CRY entangler)
# We'll use 1 parameter per layer: CRY(theta) with control=0 target=1
# Shape: (n_layers, 1)
# -----------------------
rand_params_list = []
for f in range(n_filters):
    rng_f = np.random.RandomState(seed + 1000 * f)
    rp = rng_f.uniform(0, 2 * np.pi, size=(n_layers, 1)).astype(np.float32)
    rand_params_list.append(torch.tensor(rp, dtype=torch.float32))

# -----------------------
# Circuit: AmplitudeEmbedding + (optional) entangling layers
# -----------------------
@qml.qnode(qdev, interface="torch")
def circuit(phi4, params):
    # phi4 must have length 4 for 2 qubits amplitude encoding
    qml.AmplitudeEmbedding(phi4, wires=[0, 1], normalize=True)  # NEW

    # simple "star" doesn't make sense for 2 qubits, so use CRY(0->1)
    for l in range(params.shape[0]):
        qml.CRY(params[l, 0], wires=[0, 1])

    # CHANGED: only 2 outputs now
    return [qml.expval(qml.PauliZ(0)), qml.expval(qml.PauliZ(1))]

# -----------------------
# Quanvolution on one image (CHANGED output channels: 2*n_filters)
# -----------------------
def quanv_one_image(image_hwc: torch.Tensor) -> torch.Tensor:
    out = torch.zeros((14, 14, 2 * n_filters), dtype=torch.float32)  # CHANGED

    for j in range(0, 28, 2):
        for k in range(0, 28, 2):
            phi4 = torch.tensor(
                [
                    image_hwc[j, k, 0],
                    image_hwc[j, k + 1, 0],
                    image_hwc[j + 1, k, 0],
                    image_hwc[j + 1, k + 1, 0],
                ],
                dtype=torch.float32,
            )

            feats = []
            for params_f in rand_params_list:
                q_res = circuit(phi4, params_f)              # 2 expvals now
                feats.append(torch.stack(q_res).float())     # [2]
            out[j // 2, k // 2, :] = torch.cat(feats, dim=0)  # [2*K]

    return out

def preprocess_dataset(images_nhwc: torch.Tensor, name: str) -> np.ndarray:
    q_images = []
    N = images_nhwc.shape[0]
    print(f"Quantum pre-processing of {name} images:")
    for idx in range(N):
        print(f"{idx + 1}/{N}", end="\r")
        q_img = quanv_one_image(images_nhwc[idx])
        q_images.append(q_img.numpy())
    print()
    return np.asarray(q_images, dtype=np.float32)

# -----------------------
# Cache
# -----------------------
q_train_path = SAVE_DIR / f"q_train_{n_train}_{circuit_name}_L{n_layers}_K{n_filters}_seed{seed}.npy"
q_test_path  = SAVE_DIR / f"q_test_{n_test}_{circuit_name}_L{n_layers}_K{n_filters}_seed{seed}.npy"

if PREPROCESS:
    q_train_images = preprocess_dataset(train_images_nhwc, "train")
    q_test_images  = preprocess_dataset(test_images_nhwc, "test")
    np.save(q_train_path, q_train_images)
    np.save(q_test_path, q_test_images)
else:
    q_train_images = np.load(q_train_path)
    q_test_images  = np.load(q_test_path)

# Flatten
q_train_x = torch.from_numpy(q_train_images).view(n_train, -1)  # [N, 14*14*(2*K)]
q_test_x  = torch.from_numpy(q_test_images).view(n_test, -1)

# Normalize with train stats
mean = q_train_x.mean(dim=0, keepdim=True)
std  = q_train_x.std(dim=0, keepdim=True).clamp_min(1e-6)
q_train_x = (q_train_x - mean) / std
q_test_x  = (q_test_x  - mean) / std

train_loader = DataLoader(TensorDataset(q_train_x, train_labels), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(q_test_x, test_labels), batch_size=batch_size, shuffle=False)

print("Quanv feature shape (flattened):", q_train_x.shape)

# -----------------------
# H2-style classifier (same pattern, input dim changed automatically)
# -----------------------
class H2Classifier(nn.Module):
    def __init__(self, in_dim, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        return self.net(x)

in_dim = q_train_x.shape[1]
model = H2Classifier(in_dim=in_dim).to(device_torch)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=adam_lr, weight_decay=weight_decay)

def evaluate(loader):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)
    return loss_sum / total, correct / total

history = {"epoch": [], "train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

for epoch in range(1, n_epochs + 1):
    model.train()
    for xb, yb in train_loader:
        xb = xb.to(device_torch)
        yb = yb.to(device_torch)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    train_loss, train_acc = evaluate(train_loader)
    test_loss, test_acc = evaluate(test_loader)

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(
        f"Epoch {epoch:03d}/{n_epochs} | "
        f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
        f"test loss {test_loss:.4f} acc {test_acc:.3f}"
    )

plt.figure()
plt.plot(history["epoch"], history["train_acc"], label="Train accuracy")
plt.plot(history["epoch"], history["test_acc"], label="Test accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.show()

plt.figure()
plt.plot(history["epoch"], history["train_loss"], label="Train loss")
plt.plot(history["epoch"], history["test_loss"], label="Test loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.show()

Quantum pre-processing of train images:
12/1200

KeyboardInterrupt: 

# Experiment A — Classical vs Hybrid (Baseline)

This section compares a classical baseline against a hybrid quantum-classical pipeline under the same data split and training protocol.

**What to look at**
- Validation and test accuracy
- Training stability across epochs
- Relative compute cost


In [1]:
import math
import random
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as T

import pennylane as qml

# ============================================================
# DATA-EFFICIENCY EXPERIMENT WITH CACHED PREPROCESSING
#
# Models:
#   1) Hybrid quantum random patch features + MLP head
#   2) Classical random patch features + same MLP head
#
# Main idea:
#   - For each seed, create ONE shared raw split using MAX_TRAIN.
#   - Preprocess ONCE per model per seed.
#   - Save cached features to disk.
#   - For each n_train in TRAIN_SIZES, slice the cached train pool.
#
# Measures:
#   - best validation accuracy
#   - test accuracy
#   - train accuracy at best epoch
#   - preprocessing time
#   - training time
#   - mean ± std across seeds
# ============================================================

# -----------------------------
# Global config
# -----------------------------
BASE_SEED = 246
device_torch = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN_SIZES = [50, 100, 250, 500]
MAX_TRAIN = max(TRAIN_SIZES)
N_VAL = 100
N_TEST = 100
EXPERIMENT_SEEDS = [246, 247, 248]

RUN_HYBRID = True
RUN_CLASSICAL_RANDOM = True

batch_size = 4
n_epochs = 100
lr = 1e-3
weight_decay = 1e-4

patch_size = 4
num_output_channels = 32

L2_NORMALIZE_HYBRID_FEATURES = False
L2_NORMALIZE_CLASSICAL_FEATURES = False
CLASSICAL_NONLINEARITY = "tanh"  # "tanh" or "relu"

CACHE_DIR = Path("feature_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# -----------------------------
# Helpers
# -----------------------------
def get_num_output_channels(kernel_size: int) -> int:
    return kernel_size**2 if num_output_channels == -1 else num_output_channels

def add_padding(matrix: np.ndarray, padding: Tuple[int, int]) -> np.ndarray:
    matrix = np.squeeze(matrix)
    n, m = matrix.shape
    r, c = padding
    padded = np.zeros((n + 2 * r, m + 2 * c), dtype=matrix.dtype)
    padded[r:n + r, c:m + c] = matrix
    return padded

def pad_to_divisible(img: np.ndarray, f: int) -> np.ndarray:
    img = np.squeeze(img)
    n = img.shape[0]
    if n % f != 0:
        padding_size = f - (n % f)
        pad = int(np.ceil(padding_size / 2))
        img = add_padding(img, (pad, pad))
    return img

def patch_to_nqubits_and_feature_len(patch_len: int) -> Tuple[int, int]:
    n_qubits = int(math.ceil(math.log2(patch_len)))
    feature_len = n_qubits
    return n_qubits, feature_len

def mean_std(values: List[float]) -> Tuple[float, float]:
    arr = np.array(values, dtype=np.float64)
    return float(arr.mean()), float(arr.std(ddof=0))

def config_tag() -> str:
    return (
        f"p{patch_size}_c{get_num_output_channels(patch_size)}"
        f"_maxtr{MAX_TRAIN}_val{N_VAL}_te{N_TEST}"
    )

# ============================================================
# HYBRID QUANTUM FEATURE EXTRACTOR
# ============================================================
_QNODE_CACHE: Dict[int, qml.QNode] = {}

def get_hybrid_qnode(n_qubits: int):
    if n_qubits in _QNODE_CACHE:
        return _QNODE_CACHE[n_qubits]

    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch")
    def qnode(state_vec, thetas):
        wires = list(range(n_qubits))

        qml.AmplitudeEmbedding(state_vec, wires=wires, normalize=True)

        for i in range(n_qubits):
            qml.RY(thetas[i], wires=i)

        if n_qubits > 1:
            for i in range(n_qubits):
                qml.CZ(wires=[i, (i + 1) % n_qubits])

        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    _QNODE_CACHE[n_qubits] = qnode
    return qnode

_HYBRID_THETA_BANK: Optional[np.ndarray] = None
_HYBRID_THETA_META: Optional[Tuple[int, int, int]] = None
# meta = (seed, num_circuits, n_qubits)

def get_hybrid_theta_bank(num_circuits: int, n_qubits: int, seed: int) -> np.ndarray:
    global _HYBRID_THETA_BANK, _HYBRID_THETA_META

    if _HYBRID_THETA_BANK is not None and _HYBRID_THETA_META == (seed, num_circuits, n_qubits):
        return _HYBRID_THETA_BANK

    rng = np.random.default_rng(seed)
    bank = rng.uniform(0.0, 2 * np.pi, size=(num_circuits, n_qubits)).astype(np.float32)

    _HYBRID_THETA_BANK = bank
    _HYBRID_THETA_META = (seed, num_circuits, n_qubits)
    return bank

def hybrid_connector(vector: np.ndarray, thetas: np.ndarray) -> np.ndarray:
    vec = vector.astype(np.float32)

    n_qubits = int(math.ceil(np.log2(vec.shape[0])))
    target_len = 2 ** n_qubits

    if vec.shape[0] < target_len:
        vec = np.concatenate(
            [vec, np.zeros(target_len - vec.shape[0], dtype=np.float32)],
            axis=0
        )

    if thetas.shape[0] != n_qubits:
        raise ValueError(f"thetas has length {thetas.shape[0]} but n_qubits={n_qubits}.")

    qnode = get_hybrid_qnode(n_qubits)
    feats_t = qnode(torch.tensor(vec), torch.tensor(thetas))
    feats = np.asarray(feats_t, dtype=np.float32)

    if L2_NORMALIZE_HYBRID_FEATURES:
        norm = np.linalg.norm(feats)
        if norm > 1e-12:
            feats = feats / norm

    return feats

def hybrid_quanv(image: np.ndarray, seed: int) -> np.ndarray:
    img = np.squeeze(image).astype(np.float32)
    img = pad_to_divisible(img, patch_size)
    n_image = img.shape[0]
    f = patch_size

    num_deep = get_num_output_channels(f)
    patch_len = f * f
    n_qubits, feature_len = patch_to_nqubits_and_feature_len(patch_len)
    num_circuits = int(math.ceil(num_deep / feature_len))

    theta_bank = get_hybrid_theta_bank(num_circuits=num_circuits, n_qubits=n_qubits, seed=seed)

    out = np.zeros((n_image // f, n_image // f, num_deep), dtype=np.float32)

    for i in range(0, n_image, f):
        for j in range(0, n_image, f):
            sub = img[i:i + f, j:j + f].copy()

            if np.all(sub == 0):
                sub.flat[0] = 1.0

            flat = sub.flatten()
            pnorm = np.linalg.norm(flat)
            if pnorm < 1e-12:
                pnorm = 1.0
            flat = flat / pnorm

            feats = []
            for c in range(num_circuits):
                feats.append(hybrid_connector(flat, theta_bank[c]))

            all_feats = np.concatenate(feats, axis=0)
            out[i // f, j // f, :num_deep] = all_feats[:num_deep]

    return out

def hybrid_converter(data: np.ndarray, seed: int) -> np.ndarray:
    out = []
    N = len(data)
    print("\n=== Hybrid quantum preprocessing started ===")
    for idx, x in enumerate(data):
        print(f"Processing image {idx+1}/{N}", end="\r")
        out.append(hybrid_quanv(x, seed=seed))
    print("\n=== Hybrid quantum preprocessing finished ===\n")
    return np.array(out, dtype=np.float32)

# ============================================================
# CLASSICAL RANDOM FEATURE EXTRACTOR
# ============================================================
_CLASSICAL_W: Optional[np.ndarray] = None
_CLASSICAL_B: Optional[np.ndarray] = None
_CLASSICAL_META: Optional[Tuple[int, int, int]] = None
# meta = (seed, num_features, patch_len)

def get_classical_random_bank(num_features: int, patch_len: int, seed: int):
    global _CLASSICAL_W, _CLASSICAL_B, _CLASSICAL_META

    if _CLASSICAL_W is not None and _CLASSICAL_META == (seed, num_features, patch_len):
        return _CLASSICAL_W, _CLASSICAL_B

    rng = np.random.default_rng(seed)
    W = rng.normal(0.0, 1.0 / np.sqrt(patch_len), size=(num_features, patch_len)).astype(np.float32)
    b = rng.normal(0.0, 0.1, size=(num_features,)).astype(np.float32)

    _CLASSICAL_W = W
    _CLASSICAL_B = b
    _CLASSICAL_META = (seed, num_features, patch_len)
    return W, b

def classical_connector(vector: np.ndarray, W: np.ndarray, b: np.ndarray) -> np.ndarray:
    vec = vector.astype(np.float32)
    feats = W @ vec + b

    if CLASSICAL_NONLINEARITY == "relu":
        feats = np.maximum(feats, 0.0)
    else:
        feats = np.tanh(feats)

    feats = feats.astype(np.float32)

    if L2_NORMALIZE_CLASSICAL_FEATURES:
        norm = np.linalg.norm(feats)
        if norm > 1e-12:
            feats = feats / norm

    return feats

def classical_patch_map(image: np.ndarray, seed: int) -> np.ndarray:
    img = np.squeeze(image).astype(np.float32)
    img = pad_to_divisible(img, patch_size)
    n_image = img.shape[0]
    f = patch_size

    num_features = get_num_output_channels(f)
    patch_len = f * f
    W, b = get_classical_random_bank(num_features=num_features, patch_len=patch_len, seed=seed)

    out = np.zeros((n_image // f, n_image // f, num_features), dtype=np.float32)

    for i in range(0, n_image, f):
        for j in range(0, n_image, f):
            sub = img[i:i + f, j:j + f].copy()

            if np.all(sub == 0):
                sub.flat[0] = 1.0

            flat = sub.flatten()
            pnorm = np.linalg.norm(flat)
            if pnorm < 1e-12:
                pnorm = 1.0
            flat = flat / pnorm

            feats = classical_connector(flat, W, b)
            out[i // f, j // f, :] = feats

    return out

def classical_converter(data: np.ndarray, seed: int) -> np.ndarray:
    out = []
    N = len(data)
    print("\n=== Classical random preprocessing started ===")
    for idx, x in enumerate(data):
        print(f"Processing image {idx+1}/{N}", end="\r")
        out.append(classical_patch_map(x, seed=seed))
    print("\n=== Classical random preprocessing finished ===\n")
    return np.array(out, dtype=np.float32)

# ============================================================
# DATA LOADING WITH SHARED RAW SPLITS
# ============================================================
def load_fashion_mnist_shared_pool(seed: int, max_train: int, n_val: int, n_test: int):
    rng = random.Random(seed)
    tfm = T.Compose([T.ToTensor()])

    train_ds = torchvision.datasets.FashionMNIST(root="data", train=True, download=True, transform=tfm)
    test_ds = torchvision.datasets.FashionMNIST(root="data", train=False, download=True, transform=tfm)

    train_idx = rng.sample(range(len(train_ds)), max_train + n_val)
    test_idx = rng.sample(range(len(test_ds)), n_test)

    x_train_pool = np.asarray(
        [train_ds[i][0].numpy().transpose(1, 2, 0) for i in train_idx[:max_train]],
        dtype=np.float32
    )
    y_train_pool = np.asarray(
        [train_ds[i][1] for i in train_idx[:max_train]],
        dtype=np.int64
    )

    x_val = np.asarray(
        [train_ds[i][0].numpy().transpose(1, 2, 0) for i in train_idx[max_train:max_train + n_val]],
        dtype=np.float32
    )
    y_val = np.asarray(
        [train_ds[i][1] for i in train_idx[max_train:max_train + n_val]],
        dtype=np.int64
    )

    x_test = np.asarray(
        [test_ds[i][0].numpy().transpose(1, 2, 0) for i in test_idx],
        dtype=np.float32
    )
    y_test = np.asarray(
        [test_ds[i][1] for i in test_idx],
        dtype=np.int64
    )

    return x_train_pool, y_train_pool, x_val, y_val, x_test, y_test

# ============================================================
# CACHE HELPERS
# ============================================================
def raw_cache_path(seed: int) -> Path:
    return CACHE_DIR / f"raw_seed{seed}_{config_tag()}.npz"

def feature_cache_path(model_name: str, seed: int) -> Path:
    return CACHE_DIR / f"{model_name}_seed{seed}_{config_tag()}.npz"

def save_npz(path: Path, **arrays):
    np.savez_compressed(path, **arrays)

def load_or_build_raw_pool(seed: int):
    path = raw_cache_path(seed)
    if path.exists():
        data = np.load(path)
        return (
            data["x_train_pool"], data["y_train_pool"],
            data["x_val"], data["y_val"],
            data["x_test"], data["y_test"],
        )

    x_train_pool, y_train_pool, x_val, y_val, x_test, y_test = load_fashion_mnist_shared_pool(
        seed=seed, max_train=MAX_TRAIN, n_val=N_VAL, n_test=N_TEST
    )
    save_npz(
        path,
        x_train_pool=x_train_pool, y_train_pool=y_train_pool,
        x_val=x_val, y_val=y_val,
        x_test=x_test, y_test=y_test
    )
    return x_train_pool, y_train_pool, x_val, y_val, x_test, y_test

def load_or_build_feature_cache(model_name: str, seed: int):
    path = feature_cache_path(model_name, seed)
    if path.exists():
        data = np.load(path, allow_pickle=True)
        return {
            "x_train_pool": data["x_train_pool"],
            "y_train_pool": data["y_train_pool"],
            "x_val": data["x_val"],
            "y_val": data["y_val"],
            "x_test": data["x_test"],
            "y_test": data["y_test"],
            "preprocess_time_sec": float(data["preprocess_time_sec"]),
            "feature_shape": tuple(data["feature_shape"]),
        }

    x_train_pool, y_train_pool, x_val, y_val, x_test, y_test = load_or_build_raw_pool(seed)

    start = time.perf_counter()
    if model_name == "hybrid":
        fx_train_pool = hybrid_converter(x_train_pool, seed=seed)
        fx_val = hybrid_converter(x_val, seed=seed)
        fx_test = hybrid_converter(x_test, seed=seed)
    elif model_name == "classical_random":
        fx_train_pool = classical_converter(x_train_pool, seed=seed)
        fx_val = classical_converter(x_val, seed=seed)
        fx_test = classical_converter(x_test, seed=seed)
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    preprocess_time = time.perf_counter() - start
    feature_shape = np.array(fx_train_pool.shape[1:], dtype=np.int64)

    save_npz(
        path,
        x_train_pool=fx_train_pool, y_train_pool=y_train_pool,
        x_val=fx_val, y_val=y_val,
        x_test=fx_test, y_test=y_test,
        preprocess_time_sec=np.array(preprocess_time, dtype=np.float64),
        feature_shape=feature_shape,
    )

    return {
        "x_train_pool": fx_train_pool,
        "y_train_pool": y_train_pool,
        "x_val": fx_val,
        "y_val": y_val,
        "x_test": fx_test,
        "y_test": y_test,
        "preprocess_time_sec": preprocess_time,
        "feature_shape": tuple(fx_train_pool.shape[1:]),
    }

# ============================================================
# SHARED MLP HEAD
# ============================================================
class HybridModel(nn.Module):
    def __init__(self, in_dim: int, num_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        return self.net(x)

def eval_loader(model, loader, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)
    return loss_sum / total, correct / total

def train_model(Xtr, Ytr, Xva, Yva, Xte, Yte):
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(Xva, Yva), batch_size=64, shuffle=False)
    test_loader = DataLoader(TensorDataset(Xte, Yte), batch_size=64, shuffle=False)

    in_dim = int(np.prod(Xtr.shape[1:]))
    model = HybridModel(in_dim=in_dim).to(device_torch)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_acc = -1.0
    best_state = None
    best_train_acc = 0.0
    best_train_loss = 0.0
    best_val_loss = 0.0

    train_start = time.perf_counter()

    for epoch in range(1, n_epochs + 1):
        model.train()
        total, correct, loss_sum = 0, 0, 0.0

        for xb, yb in train_loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)

        train_loss = loss_sum / total
        train_acc = correct / total
        val_loss, val_acc = eval_loader(model, val_loader, criterion)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_train_loss = train_loss
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:03d}/{n_epochs} | "
            f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.3f}"
        )

    training_time = time.perf_counter() - train_start

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_acc = eval_loader(model, test_loader, criterion)

    return {
        "best_train_loss": best_train_loss,
        "best_train_acc": best_train_acc,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "training_time_sec": training_time,
    }

# ============================================================
# EXPERIMENT RUNNERS
# ============================================================
def run_one_from_cache(model_name: str, seed: int, n_train: int):
    set_seed(seed)

    cache = load_or_build_feature_cache(model_name, seed)

    Xtr_np = cache["x_train_pool"][:n_train]
    Ytr_np = cache["y_train_pool"][:n_train]
    Xva_np = cache["x_val"]
    Yva_np = cache["y_val"]
    Xte_np = cache["x_test"]
    Yte_np = cache["y_test"]

    Xtr = torch.tensor(Xtr_np, dtype=torch.float32)
    Ytr = torch.tensor(Ytr_np, dtype=torch.long)
    Xva = torch.tensor(Xva_np, dtype=torch.float32)
    Yva = torch.tensor(Yva_np, dtype=torch.long)
    Xte = torch.tensor(Xte_np, dtype=torch.float32)
    Yte = torch.tensor(Yte_np, dtype=torch.long)

    metrics = train_model(Xtr, Ytr, Xva, Yva, Xte, Yte)
    metrics["preprocess_time_sec"] = cache["preprocess_time_sec"]
    metrics["feature_shape"] = cache["feature_shape"]
    return metrics

def summarize_results(results: List[Dict], label: str, n_train: int):
    print(f"\n===== SUMMARY | {label} | n_train={n_train} =====")

    keys = [
        "best_train_acc",
        "best_val_acc",
        "test_acc",
        "best_train_loss",
        "best_val_loss",
        "test_loss",
        "preprocess_time_sec",
        "training_time_sec",
    ]

    for k in keys:
        mu, sd = mean_std([r[k] for r in results])
        print(f"{k:20s}: {mu:.6f} ± {sd:.6f}")

    print(f"feature_shape        : {results[0]['feature_shape']}")

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    print(f"Using device: {device_torch}")
    print(f"Train sizes: {TRAIN_SIZES}")
    print(f"Seeds: {EXPERIMENT_SEEDS}")
    print(f"Patch size: {patch_size}")
    print(f"Output channels: {get_num_output_channels(patch_size)}")
    print(f"Cache dir: {CACHE_DIR.resolve()}")

    patch_len = patch_size * patch_size
    n_qubits, features_per_circuit = patch_to_nqubits_and_feature_len(patch_len)
    num_circuits = int(math.ceil(get_num_output_channels(patch_size) / features_per_circuit))
    print(
        f"Hybrid config => n_qubits={n_qubits}, "
        f"features_per_circuit={features_per_circuit}, "
        f"num_circuits={num_circuits}"
    )

    # Precompute/load cache once
    for seed in EXPERIMENT_SEEDS:
        print(f"\nPreparing shared raw pool for seed={seed}")
        load_or_build_raw_pool(seed)

        if RUN_HYBRID:
            print(f"Preparing/loading hybrid cache for seed={seed}")
            load_or_build_feature_cache("hybrid", seed)

        if RUN_CLASSICAL_RANDOM:
            print(f"Preparing/loading classical cache for seed={seed}")
            load_or_build_feature_cache("classical_random", seed)

    all_results = {"hybrid": {}, "classical_random": {}}

    for n_train in TRAIN_SIZES:
        if RUN_HYBRID:
            hybrid_results = []
            for seed in EXPERIMENT_SEEDS:
                print(f"\n\n########## HYBRID | n_train={n_train} | seed={seed} ##########")
                result = run_one_from_cache("hybrid", seed=seed, n_train=n_train)
                hybrid_results.append(result)

                print("\n--- Run result ---")
                for k, v in result.items():
                    print(f"{k}: {v}")

            all_results["hybrid"][n_train] = hybrid_results
            summarize_results(hybrid_results, "HYBRID", n_train)

        if RUN_CLASSICAL_RANDOM:
            classical_results = []
            for seed in EXPERIMENT_SEEDS:
                print(f"\n\n########## CLASSICAL_RANDOM | n_train={n_train} | seed={seed} ##########")
                result = run_one_from_cache("classical_random", seed=seed, n_train=n_train)
                classical_results.append(result)

                print("\n--- Run result ---")
                for k, v in result.items():
                    print(f"{k}: {v}")

            all_results["classical_random"][n_train] = classical_results
            summarize_results(classical_results, "CLASSICAL_RANDOM", n_train)

    print("\n\n================ FINAL AGGREGATED SUMMARY ================")
    for model_name, model_results in all_results.items():
        if not model_results:
            continue
        print(f"\nMODEL: {model_name}")
        for n_train, runs in model_results.items():
            test_mu, test_sd = mean_std([r["test_acc"] for r in runs])
            val_mu, val_sd = mean_std([r["best_val_acc"] for r in runs])
            prep_mu, prep_sd = mean_std([r["preprocess_time_sec"] for r in runs])
            train_mu, train_sd = mean_std([r["training_time_sec"] for r in runs])

            print(
                f"n_train={n_train:4d} | "
                f"val_acc={val_mu:.4f}±{val_sd:.4f} | "
                f"test_acc={test_mu:.4f}±{test_sd:.4f} | "
                f"prep_time={prep_mu:.2f}±{prep_sd:.2f}s | "
                f"train_time={train_mu:.2f}±{train_sd:.2f}s"
            )

Using device: cpu
Train sizes: [50, 100, 250, 500]
Seeds: [246, 247, 248]
Patch size: 4
Output channels: 32
Cache dir: C:\Users\Asus\qml\coursework\feature_cache
Hybrid config => n_qubits=4, features_per_circuit=4, num_circuits=8

Preparing shared raw pool for seed=246
Preparing/loading hybrid cache for seed=246

=== Hybrid quantum preprocessing started ===
Processing image 500/500
=== Hybrid quantum preprocessing finished ===


=== Hybrid quantum preprocessing started ===
Processing image 100/100
=== Hybrid quantum preprocessing finished ===


=== Hybrid quantum preprocessing started ===
Processing image 100/100
=== Hybrid quantum preprocessing finished ===

Preparing/loading classical cache for seed=246

=== Classical random preprocessing started ===
Processing image 500/500
=== Classical random preprocessing finished ===


=== Classical random preprocessing started ===
Processing image 100/100
=== Classical random preprocessing finished ===


=== Classical random preprocessing start

# Experiment B — Patch Strategy Study

This section evaluates how patch configuration impacts learned representations and downstream performance.

**Focus**
- Patch size
- Stride / overlap behavior
- Impact on feature dimensionality


In [4]:
import math
import random
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as T
import pennylane as qml

# ============================================================
# HYBRID PATCH EXPERIMENTS ONLY + PCA ANALYSIS
#
# Experiments:
#   1) Non-stride (non-overlapping): patch_size in {2, 4}
#      -> stride = patch_size
#
#   2) Stride experiments:
#      -> patch_size = 4, stride = 2
#      -> patch_size = 7, stride = 3
#
# No padding is used.
# We enforce exact tiling / valid sliding:
#   - non-stride: 28 % patch_size == 0
#   - stride: (28 - patch_size) % stride == 0
#
# Model:
#   Hybrid quantum patch features + MLP head
#
# Cached preprocessing:
#   - one shared raw split per seed
#   - one feature cache per experiment config per seed
#
# PCA metrics:
#   - variance explained by top 5 PCs
#   - variance explained by top 10 PCs
#   - variance explained by top 20 PCs
#   - number of PCs needed for 90% variance
#   - effective rank
# ============================================================

# -----------------------------
# Global config
# -----------------------------
device_torch = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_SEED = 246
EXPERIMENT_SEEDS = [246, 247, 248]

TRAIN_SIZES = [50, 100, 250, 500]
MAX_TRAIN = max(TRAIN_SIZES)
N_VAL = 100
N_TEST = 100

batch_size = 4
n_epochs = 100
lr = 1e-3
weight_decay = 1e-4

num_output_channels = 4
L2_NORMALIZE_HYBRID_FEATURES = False

CACHE_DIR = Path("feature_cache_hybrid_patch_experiments")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Patch experiments
# -----------------------------
NON_STRIDE_EXPERIMENTS = [
    {"name": "nonstride_p2_s2", "patch_size": 2, "stride": 2},
    {"name": "nonstride_p7_s7", "patch_size": 7, "stride": 7},
    {"name": "nonstride_p14_s14", "patch_size": 14, "stride": 14}
]

STRIDE_EXPERIMENTS = [
    {"name": "stride_p2_s1", "patch_size": 2, "stride": 1},
    {"name": "stride_p4_s2", "patch_size": 4, "stride": 2},
    {"name": "stride_p7_s3", "patch_size": 7, "stride": 3},
]

ALL_EXPERIMENTS = NON_STRIDE_EXPERIMENTS + STRIDE_EXPERIMENTS

# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# -----------------------------
# Helpers
# -----------------------------
def get_num_output_channels(kernel_size: int) -> int:
    return kernel_size**2 if num_output_channels == -1 else num_output_channels

def patch_to_nqubits_and_feature_len(patch_len: int) -> Tuple[int, int]:
    n_qubits = int(math.ceil(math.log2(patch_len)))
    feature_len = n_qubits  # expval(PauliZ(i)) for each qubit
    return n_qubits, feature_len

def mean_std(values: List[float]) -> Tuple[float, float]:
    arr = np.array(values, dtype=np.float64)
    return float(arr.mean()), float(arr.std(ddof=0))

def validate_patch_config(image_size: int, patch_size: int, stride: int):
    if stride == patch_size:
        if image_size % patch_size != 0:
            raise ValueError(
                f"Non-stride config invalid: image_size={image_size}, patch_size={patch_size}. "
                f"patch_size must divide image_size exactly."
            )
    else:
        if (image_size - patch_size) % stride != 0:
            raise ValueError(
                f"Stride config invalid: image_size={image_size}, patch_size={patch_size}, stride={stride}. "
                f"(image_size - patch_size) must be divisible by stride."
            )

def output_shape_for_patching(image_size: int, patch_size: int, stride: int) -> Tuple[int, int]:
    out_h = (image_size - patch_size) // stride + 1
    out_w = (image_size - patch_size) // stride + 1
    return out_h, out_w

def config_tag(exp_cfg: Dict) -> str:
    return (
        f"{exp_cfg['name']}"
        f"_c{get_num_output_channels(exp_cfg['patch_size'])}"
        f"_maxtr{MAX_TRAIN}_val{N_VAL}_te{N_TEST}"
    )

# -----------------------------
# PCA helpers
# -----------------------------
def flatten_feature_maps(x: np.ndarray) -> np.ndarray:
    """
    x: shape (N, H, W, C)
    returns: shape (N, D)
    """
    return x.reshape(x.shape[0], -1).astype(np.float64)

def compute_pca_metrics(
    x: np.ndarray,
    max_components: int = 20,
    variance_threshold: float = 0.90
) -> Dict[str, float]:
    """
    PCA metrics using SVD on cached feature maps.
    """
    X = flatten_feature_maps(x)
    X = X - X.mean(axis=0, keepdims=True)

    if X.shape[0] < 2 or np.allclose(X, 0.0):
        return {
            "pca_top5_var": 0.0,
            "pca_top10_var": 0.0,
            "pca_top20_var": 0.0,
            "pca_num_for_90": 0.0,
            "pca_effective_rank": 0.0,
        }

    _, s, _ = np.linalg.svd(X, full_matrices=False)

    eigvals = (s ** 2) / max(X.shape[0] - 1, 1)
    total_var = eigvals.sum()

    if total_var <= 1e-12:
        return {
            "pca_top5_var": 0.0,
            "pca_top10_var": 0.0,
            "pca_top20_var": 0.0,
            "pca_num_for_90": 0.0,
            "pca_effective_rank": 0.0,
        }

    explained = eigvals / total_var
    cumsum = np.cumsum(explained)

    def topk_var(k: int) -> float:
        k = min(k, len(explained))
        return float(explained[:k].sum())

    num_for_threshold = int(np.searchsorted(cumsum, variance_threshold) + 1)

    eps = 1e-12
    p = explained[explained > eps]
    entropy = -np.sum(p * np.log(p))
    effective_rank = float(np.exp(entropy))

    return {
        "pca_top5_var": topk_var(5),
        "pca_top10_var": topk_var(10),
        "pca_top20_var": topk_var(max_components),
        "pca_num_for_90": float(num_for_threshold),
        "pca_effective_rank": effective_rank,
    }

# ============================================================
# HYBRID QUANTUM FEATURE EXTRACTOR
# ============================================================
_QNODE_CACHE: Dict[int, qml.QNode] = {}

def get_hybrid_qnode(n_qubits: int):
    if n_qubits in _QNODE_CACHE:
        return _QNODE_CACHE[n_qubits]

    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch")
    def qnode(state_vec, thetas):
        wires = list(range(n_qubits))

        qml.AmplitudeEmbedding(state_vec, wires=wires, normalize=True)

        for i in range(n_qubits):
            qml.RY(thetas[i], wires=i)

        if n_qubits > 1:
            for i in range(n_qubits):
                qml.CZ(wires=[i, (i + 1) % n_qubits])

        # measurement: single-qubit Pauli-Z expectation values
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    _QNODE_CACHE[n_qubits] = qnode
    return qnode

_HYBRID_THETA_BANK: Optional[np.ndarray] = None
_HYBRID_THETA_META: Optional[Tuple[int, int, int]] = None
# meta = (seed, num_circuits, n_qubits)

def get_hybrid_theta_bank(num_circuits: int, n_qubits: int, seed: int) -> np.ndarray:
    global _HYBRID_THETA_BANK, _HYBRID_THETA_META

    if _HYBRID_THETA_BANK is not None and _HYBRID_THETA_META == (seed, num_circuits, n_qubits):
        return _HYBRID_THETA_BANK

    rng = np.random.default_rng(seed)
    bank = rng.uniform(0.0, 2 * np.pi, size=(num_circuits, n_qubits)).astype(np.float32)

    _HYBRID_THETA_BANK = bank
    _HYBRID_THETA_META = (seed, num_circuits, n_qubits)
    return bank

def hybrid_connector(vector: np.ndarray, thetas: np.ndarray) -> np.ndarray:
    vec = vector.astype(np.float32)

    n_qubits = int(math.ceil(np.log2(vec.shape[0])))
    target_len = 2 ** n_qubits

    if vec.shape[0] < target_len:
        vec = np.concatenate(
            [vec, np.zeros(target_len - vec.shape[0], dtype=np.float32)],
            axis=0
        )

    if thetas.shape[0] != n_qubits:
        raise ValueError(f"thetas has length {thetas.shape[0]} but n_qubits={n_qubits}.")

    qnode = get_hybrid_qnode(n_qubits)
    feats_t = qnode(torch.tensor(vec), torch.tensor(thetas))
    feats = np.asarray(feats_t, dtype=np.float32)

    if L2_NORMALIZE_HYBRID_FEATURES:
        norm = np.linalg.norm(feats)
        if norm > 1e-12:
            feats = feats / norm

    return feats

def hybrid_patch_features(image: np.ndarray, seed: int, patch_size: int, stride: int) -> np.ndarray:
    img = np.squeeze(image).astype(np.float32)
    image_size = img.shape[0]

    validate_patch_config(image_size=image_size, patch_size=patch_size, stride=stride)

    num_deep = get_num_output_channels(patch_size)
    patch_len = patch_size * patch_size
    n_qubits, feature_len = patch_to_nqubits_and_feature_len(patch_len)
    num_circuits = int(math.ceil(num_deep / feature_len))

    theta_bank = get_hybrid_theta_bank(
        num_circuits=num_circuits,
        n_qubits=n_qubits,
        seed=seed
    )

    out_h, out_w = output_shape_for_patching(
        image_size=image_size,
        patch_size=patch_size,
        stride=stride
    )
    out = np.zeros((out_h, out_w, num_deep), dtype=np.float32)

    out_i = 0
    for i in range(0, image_size - patch_size + 1, stride):
        out_j = 0
        for j in range(0, image_size - patch_size + 1, stride):
            sub = img[i:i + patch_size, j:j + patch_size].copy()

            if np.all(sub == 0):
                sub.flat[0] = 1.0

            flat = sub.flatten()
            pnorm = np.linalg.norm(flat)
            if pnorm < 1e-12:
                pnorm = 1.0
            flat = flat / pnorm

            feats = []
            for c in range(num_circuits):
                feats.append(hybrid_connector(flat, theta_bank[c]))

            all_feats = np.concatenate(feats, axis=0)
            out[out_i, out_j, :num_deep] = all_feats[:num_deep]
            out_j += 1
        out_i += 1

    return out

def hybrid_converter(data: np.ndarray, seed: int, patch_size: int, stride: int) -> np.ndarray:
    out = []
    N = len(data)
    print(f"\n=== Hybrid preprocessing started | patch_size={patch_size}, stride={stride} ===")
    for idx, x in enumerate(data):
        print(f"Processing image {idx+1}/{N}", end="\r")
        out.append(hybrid_patch_features(x, seed=seed, patch_size=patch_size, stride=stride))
    print(f"\n=== Hybrid preprocessing finished | patch_size={patch_size}, stride={stride} ===\n")
    return np.array(out, dtype=np.float32)

# ============================================================
# DATA LOADING WITH SHARED RAW SPLITS
# ============================================================
def load_fashion_mnist_shared_pool(seed: int, max_train: int, n_val: int, n_test: int):
    rng = random.Random(seed)
    tfm = T.Compose([T.ToTensor()])

    train_ds = torchvision.datasets.FashionMNIST(root="data", train=True, download=True, transform=tfm)
    test_ds = torchvision.datasets.FashionMNIST(root="data", train=False, download=True, transform=tfm)

    train_idx = rng.sample(range(len(train_ds)), max_train + n_val)
    test_idx = rng.sample(range(len(test_ds)), n_test)

    x_train_pool = np.asarray(
        [train_ds[i][0].numpy().transpose(1, 2, 0) for i in train_idx[:max_train]],
        dtype=np.float32
    )
    y_train_pool = np.asarray(
        [train_ds[i][1] for i in train_idx[:max_train]],
        dtype=np.int64
    )

    x_val = np.asarray(
        [train_ds[i][0].numpy().transpose(1, 2, 0) for i in train_idx[max_train:max_train + n_val]],
        dtype=np.float32
    )
    y_val = np.asarray(
        [train_ds[i][1] for i in train_idx[max_train:max_train + n_val]],
        dtype=np.int64
    )

    x_test = np.asarray(
        [test_ds[i][0].numpy().transpose(1, 2, 0) for i in test_idx],
        dtype=np.float32
    )
    y_test = np.asarray(
        [test_ds[i][1] for i in test_idx],
        dtype=np.int64
    )

    return x_train_pool, y_train_pool, x_val, y_val, x_test, y_test

# ============================================================
# CACHE HELPERS
# ============================================================
def raw_cache_path(seed: int) -> Path:
    return CACHE_DIR / f"raw_seed{seed}_maxtr{MAX_TRAIN}_val{N_VAL}_te{N_TEST}.npz"

def feature_cache_path(seed: int, exp_cfg: Dict) -> Path:
    return CACHE_DIR / f"hybrid_seed{seed}_{config_tag(exp_cfg)}.npz"

def save_npz(path: Path, **arrays):
    np.savez_compressed(path, **arrays)

def load_or_build_raw_pool(seed: int):
    path = raw_cache_path(seed)
    if path.exists():
        data = np.load(path)
        return (
            data["x_train_pool"], data["y_train_pool"],
            data["x_val"], data["y_val"],
            data["x_test"], data["y_test"],
        )

    x_train_pool, y_train_pool, x_val, y_val, x_test, y_test = load_fashion_mnist_shared_pool(
        seed=seed, max_train=MAX_TRAIN, n_val=N_VAL, n_test=N_TEST
    )
    save_npz(
        path,
        x_train_pool=x_train_pool, y_train_pool=y_train_pool,
        x_val=x_val, y_val=y_val,
        x_test=x_test, y_test=y_test
    )
    return x_train_pool, y_train_pool, x_val, y_val, x_test, y_test

def load_or_build_feature_cache(seed: int, exp_cfg: Dict):
    path = feature_cache_path(seed, exp_cfg)
    if path.exists():
        data = np.load(path, allow_pickle=True)
        return {
            "x_train_pool": data["x_train_pool"],
            "y_train_pool": data["y_train_pool"],
            "x_val": data["x_val"],
            "y_val": data["y_val"],
            "x_test": data["x_test"],
            "y_test": data["y_test"],
            "preprocess_time_sec": float(data["preprocess_time_sec"]),
            "feature_shape": tuple(data["feature_shape"]),
        }

    x_train_pool, y_train_pool, x_val, y_val, x_test, y_test = load_or_build_raw_pool(seed)

    start = time.perf_counter()
    fx_train_pool = hybrid_converter(
        x_train_pool,
        seed=seed,
        patch_size=exp_cfg["patch_size"],
        stride=exp_cfg["stride"],
    )
    fx_val = hybrid_converter(
        x_val,
        seed=seed,
        patch_size=exp_cfg["patch_size"],
        stride=exp_cfg["stride"],
    )
    fx_test = hybrid_converter(
        x_test,
        seed=seed,
        patch_size=exp_cfg["patch_size"],
        stride=exp_cfg["stride"],
    )
    preprocess_time = time.perf_counter() - start

    feature_shape = np.array(fx_train_pool.shape[1:], dtype=np.int64)

    save_npz(
        path,
        x_train_pool=fx_train_pool, y_train_pool=y_train_pool,
        x_val=fx_val, y_val=y_val,
        x_test=fx_test, y_test=y_test,
        preprocess_time_sec=np.array(preprocess_time, dtype=np.float64),
        feature_shape=feature_shape,
    )

    return {
        "x_train_pool": fx_train_pool,
        "y_train_pool": y_train_pool,
        "x_val": fx_val,
        "y_val": y_val,
        "x_test": fx_test,
        "y_test": y_test,
        "preprocess_time_sec": preprocess_time,
        "feature_shape": tuple(fx_train_pool.shape[1:]),
    }

# ============================================================
# SHARED MLP HEAD
# ============================================================
class HybridModel(nn.Module):
    def __init__(self, in_dim: int, num_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        return self.net(x)

def eval_loader(model, loader, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)
    return loss_sum / total, correct / total

def train_model(Xtr, Ytr, Xva, Yva, Xte, Yte):
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(Xva, Yva), batch_size=64, shuffle=False)
    test_loader = DataLoader(TensorDataset(Xte, Yte), batch_size=64, shuffle=False)

    in_dim = int(np.prod(Xtr.shape[1:]))
    model = HybridModel(in_dim=in_dim).to(device_torch)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_acc = -1.0
    best_state = None
    best_train_acc = 0.0
    best_train_loss = 0.0
    best_val_loss = 0.0

    train_start = time.perf_counter()

    for epoch in range(1, n_epochs + 1):
        model.train()
        total, correct, loss_sum = 0, 0, 0.0

        for xb, yb in train_loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)

        train_loss = loss_sum / total
        train_acc = correct / total
        val_loss, val_acc = eval_loader(model, val_loader, criterion)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_train_loss = train_loss
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:03d}/{n_epochs} | "
            f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.3f}"
        )

    training_time = time.perf_counter() - train_start

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_acc = eval_loader(model, test_loader, criterion)

    return {
        "best_train_loss": best_train_loss,
        "best_train_acc": best_train_acc,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "training_time_sec": training_time,
    }

# ============================================================
# EXPERIMENT RUNNERS
# ============================================================
def run_one_from_cache(seed: int, n_train: int, exp_cfg: Dict):
    set_seed(seed)

    cache = load_or_build_feature_cache(seed=seed, exp_cfg=exp_cfg)

    Xtr_np = cache["x_train_pool"][:n_train]
    Ytr_np = cache["y_train_pool"][:n_train]
    Xva_np = cache["x_val"]
    Yva_np = cache["y_val"]
    Xte_np = cache["x_test"]
    Yte_np = cache["y_test"]

    # PCA on train features
    pca_metrics = compute_pca_metrics(Xtr_np, max_components=20, variance_threshold=0.90)

    Xtr = torch.tensor(Xtr_np, dtype=torch.float32)
    Ytr = torch.tensor(Ytr_np, dtype=torch.long)
    Xva = torch.tensor(Xva_np, dtype=torch.float32)
    Yva = torch.tensor(Yva_np, dtype=torch.long)
    Xte = torch.tensor(Xte_np, dtype=torch.float32)
    Yte = torch.tensor(Yte_np, dtype=torch.long)

    metrics = train_model(Xtr, Ytr, Xva, Yva, Xte, Yte)
    metrics.update(pca_metrics)

    metrics["preprocess_time_sec"] = cache["preprocess_time_sec"]
    metrics["feature_shape"] = cache["feature_shape"]
    metrics["patch_size"] = exp_cfg["patch_size"]
    metrics["stride"] = exp_cfg["stride"]

    image_size = 28
    patch_len = exp_cfg["patch_size"] * exp_cfg["patch_size"]
    n_qubits, features_per_circuit = patch_to_nqubits_and_feature_len(patch_len)
    num_circuits = int(math.ceil(get_num_output_channels(exp_cfg["patch_size"]) / features_per_circuit))
    out_h, out_w = output_shape_for_patching(image_size, exp_cfg["patch_size"], exp_cfg["stride"])

    metrics["n_qubits"] = n_qubits
    metrics["features_per_circuit"] = features_per_circuit
    metrics["num_circuits"] = num_circuits
    metrics["patches_per_image"] = out_h * out_w

    return metrics

def summarize_results(results: List[Dict], exp_cfg: Dict, n_train: int):
    print(f"\n===== SUMMARY | {exp_cfg['name']} | n_train={n_train} =====")

    keys = [
        "best_train_acc",
        "best_val_acc",
        "test_acc",
        "best_train_loss",
        "best_val_loss",
        "test_loss",
        "preprocess_time_sec",
        "training_time_sec",
        "pca_top5_var",
        "pca_top10_var",
        "pca_top20_var",
        "pca_num_for_90",
        "pca_effective_rank",
    ]

    for k in keys:
        mu, sd = mean_std([r[k] for r in results])
        print(f"{k:20s}: {mu:.6f} ± {sd:.6f}")

    print(f"feature_shape        : {results[0]['feature_shape']}")
    print(f"patch_size           : {results[0]['patch_size']}")
    print(f"stride               : {results[0]['stride']}")
    print(f"n_qubits             : {results[0]['n_qubits']}")
    print(f"features_per_circuit : {results[0]['features_per_circuit']}")
    print(f"num_circuits         : {results[0]['num_circuits']}")
    print(f"patches_per_image    : {results[0]['patches_per_image']}")

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    print(f"Using device: {device_torch}")
    print(f"Train sizes: {TRAIN_SIZES}")
    print(f"Seeds: {EXPERIMENT_SEEDS}")
    print(f"Output channels: {num_output_channels}")
    print(f"Cache dir: {CACHE_DIR.resolve()}")

    print("\nExperiments:")
    for exp_cfg in ALL_EXPERIMENTS:
        validate_patch_config(28, exp_cfg["patch_size"], exp_cfg["stride"])
        out_h, out_w = output_shape_for_patching(28, exp_cfg["patch_size"], exp_cfg["stride"])
        patch_len = exp_cfg["patch_size"] * exp_cfg["patch_size"]
        n_qubits, features_per_circuit = patch_to_nqubits_and_feature_len(patch_len)
        num_circuits = int(math.ceil(get_num_output_channels(exp_cfg["patch_size"]) / features_per_circuit))

        print(
            f"  {exp_cfg['name']}: "
            f"patch={exp_cfg['patch_size']}, stride={exp_cfg['stride']}, "
            f"out=({out_h},{out_w}), patches/image={out_h*out_w}, "
            f"n_qubits={n_qubits}, features/circuit={features_per_circuit}, circuits/patch={num_circuits}"
        )

    # Precompute / load cache once per seed per experiment
    for seed in EXPERIMENT_SEEDS:
        print(f"\nPreparing shared raw pool for seed={seed}")
        load_or_build_raw_pool(seed)

        for exp_cfg in ALL_EXPERIMENTS:
            print(f"Preparing/loading hybrid cache for seed={seed}, exp={exp_cfg['name']}")
            load_or_build_feature_cache(seed=seed, exp_cfg=exp_cfg)

    all_results = {}

    for exp_cfg in ALL_EXPERIMENTS:
        all_results[exp_cfg["name"]] = {}

        for n_train in TRAIN_SIZES:
            exp_results = []

            for seed in EXPERIMENT_SEEDS:
                print(f"\n\n########## {exp_cfg['name']} | n_train={n_train} | seed={seed} ##########")
                result = run_one_from_cache(seed=seed, n_train=n_train, exp_cfg=exp_cfg)
                exp_results.append(result)

                print("\n--- Run result ---")
                for k, v in result.items():
                    print(f"{k}: {v}")

            all_results[exp_cfg["name"]][n_train] = exp_results
            summarize_results(exp_results, exp_cfg, n_train)

    print("\n\n================ FINAL AGGREGATED SUMMARY ================")
    for exp_name, exp_results in all_results.items():
        print(f"\nEXPERIMENT: {exp_name}")
        for n_train, runs in exp_results.items():
            test_mu, test_sd = mean_std([r["test_acc"] for r in runs])
            val_mu, val_sd = mean_std([r["best_val_acc"] for r in runs])
            prep_mu, prep_sd = mean_std([r["preprocess_time_sec"] for r in runs])
            train_mu, train_sd = mean_std([r["training_time_sec"] for r in runs])
            pca_rank_mu, pca_rank_sd = mean_std([r["pca_effective_rank"] for r in runs])
            pca90_mu, pca90_sd = mean_std([r["pca_num_for_90"] for r in runs])

            meta = runs[0]
            print(
                f"n_train={n_train:4d} | "
                f"patch={meta['patch_size']} stride={meta['stride']} | "
                f"val_acc={val_mu:.4f}±{val_sd:.4f} | "
                f"test_acc={test_mu:.4f}±{test_sd:.4f} | "
                f"pca_rank={pca_rank_mu:.2f}±{pca_rank_sd:.2f} | "
                f"pca90={pca90_mu:.2f}±{pca90_sd:.2f} | "
                f"prep_time={prep_mu:.2f}±{prep_sd:.2f}s | "
                f"train_time={train_mu:.2f}±{train_sd:.2f}s"
            )

Using device: cpu
Train sizes: [50, 100, 250, 500]
Seeds: [246, 247, 248]
Output channels: 4
Cache dir: C:\Users\Asus\qml\coursework\feature_cache_hybrid_patch_experiments

Experiments:
  nonstride_p2_s2: patch=2, stride=2, out=(14,14), patches/image=196, n_qubits=2, features/circuit=2, circuits/patch=2
  nonstride_p4_s4: patch=7, stride=7, out=(4,4), patches/image=16, n_qubits=6, features/circuit=6, circuits/patch=1
  stride_p4_s2: patch=4, stride=2, out=(13,13), patches/image=169, n_qubits=4, features/circuit=4, circuits/patch=1
  stride_p7_s3: patch=7, stride=3, out=(8,8), patches/image=64, n_qubits=6, features/circuit=6, circuits/patch=1

Preparing shared raw pool for seed=246
Preparing/loading hybrid cache for seed=246, exp=nonstride_p2_s2

=== Hybrid preprocessing started | patch_size=2, stride=2 ===
Processing image 500/500
=== Hybrid preprocessing finished | patch_size=2, stride=2 ===


=== Hybrid preprocessing started | patch_size=2, stride=2 ===
Processing image 100/100
=== H

# Experiment C — Measurement Function Study

This section compares different measurement strategies in the quantum feature extractor and their effect on model quality.


## C1 — Pauli X / Y / Z Measurements

Evaluate a full Pauli measurement set for each patch representation.


In [5]:
import math
import random
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as T
import pennylane as qml

# ============================================================
# HYBRID MEASUREMENT EXPERIMENTS ONLY + PCA ANALYSIS
#
# Goal:
#   Investigate measurement choice while keeping the circuit fixed.
#
# Fixed circuit:
#   AmplitudeEmbedding
#   -> random RY layer
#   -> CZ ring entanglement
#
# Measurements compared (same number of features per circuit):
#   - Z
#   - X
#   - Y
#   - ZZ  (nearest-neighbor PauliZ correlations)
#
# Important:
#   - same circuit for all experiments
#   - same number of output channels
#   - same train/val/test splits
#   - only measurement changes
#
# Run only for n_train = 500
# ============================================================

# -----------------------------
# Global config
# -----------------------------
device_torch = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_SEED = 246
EXPERIMENT_SEEDS = [246, 247, 248]

N_TRAIN = 500
N_VAL = 100
N_TEST = 100

batch_size = 4
n_epochs = 100
lr = 1e-3
weight_decay = 1e-4

# Fixed patch setup for measurement study
patch_size = 4
stride = 4

num_output_channels = 4
L2_NORMALIZE_HYBRID_FEATURES = False

CACHE_DIR = Path("feature_cache_hybrid_measurement_experiments")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Measurement experiments
# -----------------------------
MEASUREMENT_EXPERIMENTS = [
    {"name": "measure_Z", "measurement_type": "Z"},
    {"name": "measure_X", "measurement_type": "X"},
    {"name": "measure_Y", "measurement_type": "Y"},
    {"name": "measure_ZZ", "measurement_type": "ZZ"},
]

# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# -----------------------------
# Helpers
# -----------------------------
def get_num_output_channels(kernel_size: int) -> int:
    return kernel_size**2 if num_output_channels == -1 else num_output_channels

def patch_to_nqubits_and_feature_len(patch_len: int) -> Tuple[int, int]:
    n_qubits = int(math.ceil(math.log2(patch_len)))
    feature_len = n_qubits  # Z, X, Y, ZZ all return n_qubits features/circuit
    return n_qubits, feature_len

def mean_std(values):
    arr = np.array(values, dtype=np.float64)
    return float(arr.mean()), float(arr.std(ddof=0))

def output_shape_for_patching(image_size: int, patch_size: int, stride: int) -> Tuple[int, int]:
    out_h = (image_size - patch_size) // stride + 1
    out_w = (image_size - patch_size) // stride + 1
    return out_h, out_w

def validate_patch_config(image_size: int, patch_size: int, stride: int):
    if (image_size - patch_size) % stride != 0:
        raise ValueError(
            f"Invalid config: image_size={image_size}, patch_size={patch_size}, stride={stride}"
        )

def config_tag(exp_cfg: Dict) -> str:
    return (
        f"{exp_cfg['name']}"
        f"_p{patch_size}_s{stride}"
        f"_c{get_num_output_channels(patch_size)}"
        f"_tr{N_TRAIN}_val{N_VAL}_te{N_TEST}"
    )

# -----------------------------
# PCA helpers
# -----------------------------
def flatten_feature_maps(x: np.ndarray) -> np.ndarray:
    return x.reshape(x.shape[0], -1).astype(np.float64)

def compute_pca_metrics(
    x: np.ndarray,
    max_components: int = 20,
    variance_threshold: float = 0.90
) -> Dict[str, float]:
    X = flatten_feature_maps(x)
    X = X - X.mean(axis=0, keepdims=True)

    if X.shape[0] < 2 or np.allclose(X, 0.0):
        return {
            "pca_top5_var": 0.0,
            "pca_top10_var": 0.0,
            "pca_top20_var": 0.0,
            "pca_num_for_90": 0.0,
            "pca_effective_rank": 0.0,
        }

    _, s, _ = np.linalg.svd(X, full_matrices=False)
    eigvals = (s ** 2) / max(X.shape[0] - 1, 1)
    total_var = eigvals.sum()

    if total_var <= 1e-12:
        return {
            "pca_top5_var": 0.0,
            "pca_top10_var": 0.0,
            "pca_top20_var": 0.0,
            "pca_num_for_90": 0.0,
            "pca_effective_rank": 0.0,
        }

    explained = eigvals / total_var
    cumsum = np.cumsum(explained)

    def topk_var(k: int) -> float:
        k = min(k, len(explained))
        return float(explained[:k].sum())

    num_for_threshold = int(np.searchsorted(cumsum, variance_threshold) + 1)

    eps = 1e-12
    p = explained[explained > eps]
    entropy = -np.sum(p * np.log(p))
    effective_rank = float(np.exp(entropy))

    return {
        "pca_top5_var": topk_var(5),
        "pca_top10_var": topk_var(10),
        "pca_top20_var": topk_var(max_components),
        "pca_num_for_90": float(num_for_threshold),
        "pca_effective_rank": effective_rank,
    }

# ============================================================
# HYBRID QUANTUM FEATURE EXTRACTOR
# ============================================================
_QNODE_CACHE: Dict[Tuple[int, str], qml.QNode] = {}

def get_hybrid_qnode(n_qubits: int, measurement_type: str):
    key = (n_qubits, measurement_type)
    if key in _QNODE_CACHE:
        return _QNODE_CACHE[key]

    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch")
    def qnode(state_vec, thetas):
        wires = list(range(n_qubits))

        # Fixed circuit for all measurement experiments
        qml.AmplitudeEmbedding(state_vec, wires=wires, normalize=True)

        for i in range(n_qubits):
            qml.RY(thetas[i], wires=i)

        if n_qubits > 1:
            for i in range(n_qubits):
                qml.CZ(wires=[i, (i + 1) % n_qubits])

        # Only the measurement changes
        if measurement_type == "Z":
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
        elif measurement_type == "X":
            return [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]
        elif measurement_type == "Y":
            return [qml.expval(qml.PauliY(i)) for i in range(n_qubits)]
        elif measurement_type == "ZZ":
            if n_qubits == 1:
                return [qml.expval(qml.PauliZ(0))]
            return [
                qml.expval(qml.PauliZ(i) @ qml.PauliZ((i + 1) % n_qubits))
                for i in range(n_qubits)
            ]
        else:
            raise ValueError(f"Unknown measurement_type: {measurement_type}")

    _QNODE_CACHE[key] = qnode
    return qnode

_HYBRID_THETA_BANK: Optional[np.ndarray] = None
_HYBRID_THETA_META: Optional[Tuple[int, int, int]] = None
# meta = (seed, num_circuits, n_qubits)

def get_hybrid_theta_bank(num_circuits: int, n_qubits: int, seed: int) -> np.ndarray:
    global _HYBRID_THETA_BANK, _HYBRID_THETA_META

    if _HYBRID_THETA_BANK is not None and _HYBRID_THETA_META == (seed, num_circuits, n_qubits):
        return _HYBRID_THETA_BANK

    rng = np.random.default_rng(seed)
    bank = rng.uniform(0.0, 2 * np.pi, size=(num_circuits, n_qubits)).astype(np.float32)

    _HYBRID_THETA_BANK = bank
    _HYBRID_THETA_META = (seed, num_circuits, n_qubits)
    return bank

def hybrid_connector(vector: np.ndarray, thetas: np.ndarray, measurement_type: str) -> np.ndarray:
    vec = vector.astype(np.float32)

    n_qubits = int(math.ceil(np.log2(vec.shape[0])))
    target_len = 2 ** n_qubits

    if vec.shape[0] < target_len:
        vec = np.concatenate(
            [vec, np.zeros(target_len - vec.shape[0], dtype=np.float32)],
            axis=0
        )

    if thetas.shape[0] != n_qubits:
        raise ValueError(f"thetas has length {thetas.shape[0]} but n_qubits={n_qubits}.")

    qnode = get_hybrid_qnode(n_qubits, measurement_type)
    feats_t = qnode(torch.tensor(vec), torch.tensor(thetas))
    feats = np.asarray(feats_t, dtype=np.float32)

    if L2_NORMALIZE_HYBRID_FEATURES:
        norm = np.linalg.norm(feats)
        if norm > 1e-12:
            feats = feats / norm

    return feats

def hybrid_patch_features(
    image: np.ndarray,
    seed: int,
    patch_size: int,
    stride: int,
    measurement_type: str
) -> np.ndarray:
    img = np.squeeze(image).astype(np.float32)
    image_size = img.shape[0]

    validate_patch_config(image_size=image_size, patch_size=patch_size, stride=stride)

    num_deep = get_num_output_channels(patch_size)
    patch_len = patch_size * patch_size
    n_qubits, feature_len = patch_to_nqubits_and_feature_len(patch_len)
    num_circuits = int(math.ceil(num_deep / feature_len))

    theta_bank = get_hybrid_theta_bank(
        num_circuits=num_circuits,
        n_qubits=n_qubits,
        seed=seed
    )

    out_h, out_w = output_shape_for_patching(image_size, patch_size, stride)
    out = np.zeros((out_h, out_w, num_deep), dtype=np.float32)

    out_i = 0
    for i in range(0, image_size - patch_size + 1, stride):
        out_j = 0
        for j in range(0, image_size - patch_size + 1, stride):
            sub = img[i:i + patch_size, j:j + patch_size].copy()

            if np.all(sub == 0):
                sub.flat[0] = 1.0

            flat = sub.flatten()
            pnorm = np.linalg.norm(flat)
            if pnorm < 1e-12:
                pnorm = 1.0
            flat = flat / pnorm

            feats = []
            for c in range(num_circuits):
                feats.append(hybrid_connector(flat, theta_bank[c], measurement_type))

            all_feats = np.concatenate(feats, axis=0)
            out[out_i, out_j, :num_deep] = all_feats[:num_deep]
            out_j += 1
        out_i += 1

    return out

def hybrid_converter(data: np.ndarray, seed: int, measurement_type: str) -> np.ndarray:
    out = []
    N = len(data)
    print(f"\n=== Hybrid preprocessing started | measurement={measurement_type} ===")
    for idx, x in enumerate(data):
        print(f"Processing image {idx+1}/{N}", end="\r")
        out.append(
            hybrid_patch_features(
                x,
                seed=seed,
                patch_size=patch_size,
                stride=stride,
                measurement_type=measurement_type
            )
        )
    print(f"\n=== Hybrid preprocessing finished | measurement={measurement_type} ===\n")
    return np.array(out, dtype=np.float32)

# ============================================================
# DATA LOADING WITH SHARED RAW SPLITS
# ============================================================
def load_fashion_mnist_shared_pool(seed: int, n_train: int, n_val: int, n_test: int):
    rng = random.Random(seed)
    tfm = T.Compose([T.ToTensor()])

    train_ds = torchvision.datasets.FashionMNIST(root="data", train=True, download=True, transform=tfm)
    test_ds = torchvision.datasets.FashionMNIST(root="data", train=False, download=True, transform=tfm)

    train_idx = rng.sample(range(len(train_ds)), n_train + n_val)
    test_idx = rng.sample(range(len(test_ds)), n_test)

    x_train = np.asarray(
        [train_ds[i][0].numpy().transpose(1, 2, 0) for i in train_idx[:n_train]],
        dtype=np.float32
    )
    y_train = np.asarray(
        [train_ds[i][1] for i in train_idx[:n_train]],
        dtype=np.int64
    )

    x_val = np.asarray(
        [train_ds[i][0].numpy().transpose(1, 2, 0) for i in train_idx[n_train:n_train + n_val]],
        dtype=np.float32
    )
    y_val = np.asarray(
        [train_ds[i][1] for i in train_idx[n_train:n_train + n_val]],
        dtype=np.int64
    )

    x_test = np.asarray(
        [test_ds[i][0].numpy().transpose(1, 2, 0) for i in test_idx],
        dtype=np.float32
    )
    y_test = np.asarray(
        [test_ds[i][1] for i in test_idx],
        dtype=np.int64
    )

    return x_train, y_train, x_val, y_val, x_test, y_test

# ============================================================
# CACHE HELPERS
# ============================================================
def raw_cache_path(seed: int) -> Path:
    return CACHE_DIR / f"raw_seed{seed}_tr{N_TRAIN}_val{N_VAL}_te{N_TEST}.npz"

def feature_cache_path(seed: int, exp_cfg: Dict) -> Path:
    return CACHE_DIR / f"hybrid_seed{seed}_{config_tag(exp_cfg)}.npz"

def save_npz(path: Path, **arrays):
    np.savez_compressed(path, **arrays)

def load_or_build_raw_pool(seed: int):
    path = raw_cache_path(seed)
    if path.exists():
        data = np.load(path)
        return (
            data["x_train"], data["y_train"],
            data["x_val"], data["y_val"],
            data["x_test"], data["y_test"],
        )

    x_train, y_train, x_val, y_val, x_test, y_test = load_fashion_mnist_shared_pool(
        seed=seed, n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST
    )
    save_npz(
        path,
        x_train=x_train, y_train=y_train,
        x_val=x_val, y_val=y_val,
        x_test=x_test, y_test=y_test
    )
    return x_train, y_train, x_val, y_val, x_test, y_test

def load_or_build_feature_cache(seed: int, exp_cfg: Dict):
    path = feature_cache_path(seed, exp_cfg)
    if path.exists():
        data = np.load(path, allow_pickle=True)
        return {
            "x_train": data["x_train"],
            "y_train": data["y_train"],
            "x_val": data["x_val"],
            "y_val": data["y_val"],
            "x_test": data["x_test"],
            "y_test": data["y_test"],
            "preprocess_time_sec": float(data["preprocess_time_sec"]),
            "feature_shape": tuple(data["feature_shape"]),
        }

    x_train, y_train, x_val, y_val, x_test, y_test = load_or_build_raw_pool(seed)

    start = time.perf_counter()
    fx_train = hybrid_converter(x_train, seed=seed, measurement_type=exp_cfg["measurement_type"])
    fx_val = hybrid_converter(x_val, seed=seed, measurement_type=exp_cfg["measurement_type"])
    fx_test = hybrid_converter(x_test, seed=seed, measurement_type=exp_cfg["measurement_type"])
    preprocess_time = time.perf_counter() - start

    feature_shape = np.array(fx_train.shape[1:], dtype=np.int64)

    save_npz(
        path,
        x_train=fx_train, y_train=y_train,
        x_val=fx_val, y_val=y_val,
        x_test=fx_test, y_test=y_test,
        preprocess_time_sec=np.array(preprocess_time, dtype=np.float64),
        feature_shape=feature_shape,
    )

    return {
        "x_train": fx_train,
        "y_train": y_train,
        "x_val": fx_val,
        "y_val": y_val,
        "x_test": fx_test,
        "y_test": y_test,
        "preprocess_time_sec": preprocess_time,
        "feature_shape": tuple(fx_train.shape[1:]),
    }

# ============================================================
# SHARED MLP HEAD
# ============================================================
class HybridModel(nn.Module):
    def __init__(self, in_dim: int, num_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        return self.net(x)

def eval_loader(model, loader, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)
    return loss_sum / total, correct / total

def train_model(Xtr, Ytr, Xva, Yva, Xte, Yte):
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(Xva, Yva), batch_size=64, shuffle=False)
    test_loader = DataLoader(TensorDataset(Xte, Yte), batch_size=64, shuffle=False)

    in_dim = int(np.prod(Xtr.shape[1:]))
    model = HybridModel(in_dim=in_dim).to(device_torch)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_acc = -1.0
    best_state = None
    best_train_acc = 0.0
    best_train_loss = 0.0
    best_val_loss = 0.0

    train_start = time.perf_counter()

    for epoch in range(1, n_epochs + 1):
        model.train()
        total, correct, loss_sum = 0, 0, 0.0

        for xb, yb in train_loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)

        train_loss = loss_sum / total
        train_acc = correct / total
        val_loss, val_acc = eval_loader(model, val_loader, criterion)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_train_loss = train_loss
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:03d}/{n_epochs} | "
            f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.3f}"
        )

    training_time = time.perf_counter() - train_start

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_acc = eval_loader(model, test_loader, criterion)

    return {
        "best_train_loss": best_train_loss,
        "best_train_acc": best_train_acc,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "training_time_sec": training_time,
    }

# ============================================================
# EXPERIMENT RUNNERS
# ============================================================
def run_one_from_cache(seed: int, exp_cfg: Dict):
    set_seed(seed)

    cache = load_or_build_feature_cache(seed=seed, exp_cfg=exp_cfg)

    Xtr_np = cache["x_train"]
    Ytr_np = cache["y_train"]
    Xva_np = cache["x_val"]
    Yva_np = cache["y_val"]
    Xte_np = cache["x_test"]
    Yte_np = cache["y_test"]

    pca_metrics = compute_pca_metrics(Xtr_np, max_components=20, variance_threshold=0.90)

    Xtr = torch.tensor(Xtr_np, dtype=torch.float32)
    Ytr = torch.tensor(Ytr_np, dtype=torch.long)
    Xva = torch.tensor(Xva_np, dtype=torch.float32)
    Yva = torch.tensor(Yva_np, dtype=torch.long)
    Xte = torch.tensor(Xte_np, dtype=torch.float32)
    Yte = torch.tensor(Yte_np, dtype=torch.long)

    metrics = train_model(Xtr, Ytr, Xva, Yva, Xte, Yte)
    metrics.update(pca_metrics)

    metrics["preprocess_time_sec"] = cache["preprocess_time_sec"]
    metrics["feature_shape"] = cache["feature_shape"]
    metrics["measurement_type"] = exp_cfg["measurement_type"]

    image_size = 28
    patch_len = patch_size * patch_size
    n_qubits, features_per_circuit = patch_to_nqubits_and_feature_len(patch_len)
    num_circuits = int(math.ceil(get_num_output_channels(patch_size) / features_per_circuit))
    out_h, out_w = output_shape_for_patching(image_size, patch_size, stride)

    metrics["patch_size"] = patch_size
    metrics["stride"] = stride
    metrics["n_qubits"] = n_qubits
    metrics["features_per_circuit"] = features_per_circuit
    metrics["num_circuits"] = num_circuits
    metrics["patches_per_image"] = out_h * out_w

    return metrics

def summarize_results(results, exp_cfg):
    print(f"\n===== SUMMARY | {exp_cfg['name']} =====")

    keys = [
        "best_train_acc",
        "best_val_acc",
        "test_acc",
        "best_train_loss",
        "best_val_loss",
        "test_loss",
        "preprocess_time_sec",
        "training_time_sec",
        "pca_top5_var",
        "pca_top10_var",
        "pca_top20_var",
        "pca_num_for_90",
        "pca_effective_rank",
    ]

    for k in keys:
        mu, sd = mean_std([r[k] for r in results])
        print(f"{k:20s}: {mu:.6f} ± {sd:.6f}")

    print(f"measurement_type     : {results[0]['measurement_type']}")
    print(f"feature_shape        : {results[0]['feature_shape']}")
    print(f"patch_size           : {results[0]['patch_size']}")
    print(f"stride               : {results[0]['stride']}")
    print(f"n_qubits             : {results[0]['n_qubits']}")
    print(f"features_per_circuit : {results[0]['features_per_circuit']}")
    print(f"num_circuits         : {results[0]['num_circuits']}")
    print(f"patches_per_image    : {results[0]['patches_per_image']}")

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    print(f"Using device: {device_torch}")
    print(f"Seeds: {EXPERIMENT_SEEDS}")
    print(f"n_train: {N_TRAIN}")
    print(f"Patch size: {patch_size}")
    print(f"Stride: {stride}")
    print(f"Output channels: {num_output_channels}")
    print(f"Cache dir: {CACHE_DIR.resolve()}")

    validate_patch_config(28, patch_size, stride)

    out_h, out_w = output_shape_for_patching(28, patch_size, stride)
    patch_len = patch_size * patch_size
    n_qubits, features_per_circuit = patch_to_nqubits_and_feature_len(patch_len)
    num_circuits = int(math.ceil(get_num_output_channels(patch_size) / features_per_circuit))

    print(
        f"\nFixed config: out=({out_h},{out_w}), patches/image={out_h*out_w}, "
        f"n_qubits={n_qubits}, features/circuit={features_per_circuit}, circuits/patch={num_circuits}"
    )

    print("\nMeasurements:")
    for exp_cfg in MEASUREMENT_EXPERIMENTS:
        print(f"  {exp_cfg['name']}: measurement={exp_cfg['measurement_type']}")

    # Build raw cache once per seed
    for seed in EXPERIMENT_SEEDS:
        print(f"\nPreparing shared raw pool for seed={seed}")
        load_or_build_raw_pool(seed)

        for exp_cfg in MEASUREMENT_EXPERIMENTS:
            print(f"Preparing/loading hybrid cache for seed={seed}, measurement={exp_cfg['measurement_type']}")
            load_or_build_feature_cache(seed=seed, exp_cfg=exp_cfg)

    all_results = {}

    for exp_cfg in MEASUREMENT_EXPERIMENTS:
        exp_results = []

        for seed in EXPERIMENT_SEEDS:
            print(f"\n\n########## {exp_cfg['name']} | seed={seed} ##########")
            result = run_one_from_cache(seed=seed, exp_cfg=exp_cfg)
            exp_results.append(result)

            print("\n--- Run result ---")
            for k, v in result.items():
                print(f"{k}: {v}")

        all_results[exp_cfg["name"]] = exp_results
        summarize_results(exp_results, exp_cfg)

    print("\n\n================ FINAL AGGREGATED SUMMARY ================")
    for exp_name, runs in all_results.items():
        test_mu, test_sd = mean_std([r["test_acc"] for r in runs])
        val_mu, val_sd = mean_std([r["best_val_acc"] for r in runs])
        prep_mu, prep_sd = mean_std([r["preprocess_time_sec"] for r in runs])
        train_mu, train_sd = mean_std([r["training_time_sec"] for r in runs])
        pca_rank_mu, pca_rank_sd = mean_std([r["pca_effective_rank"] for r in runs])
        pca90_mu, pca90_sd = mean_std([r["pca_num_for_90"] for r in runs])

        meta = runs[0]
        print(
            f"{exp_name:12s} | "
            f"meas={meta['measurement_type']} | "
            f"val_acc={val_mu:.4f}±{val_sd:.4f} | "
            f"test_acc={test_mu:.4f}±{test_sd:.4f} | "
            f"pca_rank={pca_rank_mu:.2f}±{pca_rank_sd:.2f} | "
            f"pca90={pca90_mu:.2f}±{pca90_sd:.2f} | "
            f"prep_time={prep_mu:.2f}±{prep_sd:.2f}s | "
            f"train_time={train_mu:.2f}±{train_sd:.2f}s"
        )

Using device: cpu
Seeds: [246, 247, 248]
n_train: 500
Patch size: 4
Stride: 4
Output channels: 4
Cache dir: C:\Users\Asus\qml\coursework\feature_cache_hybrid_measurement_experiments

Fixed config: out=(7,7), patches/image=49, n_qubits=4, features/circuit=4, circuits/patch=1

Measurements:
  measure_Z: measurement=Z
  measure_X: measurement=X
  measure_Y: measurement=Y
  measure_ZZ: measurement=ZZ

Preparing shared raw pool for seed=246
Preparing/loading hybrid cache for seed=246, measurement=Z

=== Hybrid preprocessing started | measurement=Z ===
Processing image 500/500
=== Hybrid preprocessing finished | measurement=Z ===


=== Hybrid preprocessing started | measurement=Z ===
Processing image 100/100
=== Hybrid preprocessing finished | measurement=Z ===


=== Hybrid preprocessing started | measurement=Z ===
Processing image 100/100
=== Hybrid preprocessing finished | measurement=Z ===

Preparing/loading hybrid cache for seed=246, measurement=X

=== Hybrid preprocessing started | meas

## C2 — Pauli X / Z Measurements

Reduced measurement set used to test whether lower measurement complexity retains performance.


In [2]:
import math
import random
import time
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as T
import pennylane as qml

# ============================================================
# HYBRID MEASUREMENT EXPERIMENT: XXZZ
#
# Fixed circuit:
#   AmplitudeEmbedding
#   -> random RY layer
#   -> CZ ring entanglement
#
# Only change:
#   measurement = [X on first half of qubits, Z on second half]
#
# For patch_size = 4:
#   patch_len = 16
#   n_qubits = 4
#   measurement returns:
#       [<X0>, <X1>, <Z2>, <Z3>]
#
# Same number of features as Z / X / Y / ZZ:
#   features_per_circuit = n_qubits = 4
#
# So with num_output_channels = 4:
#   num_circuits = 1
# ============================================================

# -----------------------------
# Global config
# -----------------------------
device_torch = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_SEED = 246
EXPERIMENT_SEEDS = [246, 247, 248]

N_TRAIN = 1200
N_VAL = 300
N_TEST = 300

batch_size = 4
n_epochs = 100
lr = 1e-3
weight_decay = 1e-4

# Fixed patch setup for measurement study
patch_size = 4
stride = 4

num_output_channels = 16
L2_NORMALIZE_HYBRID_FEATURES = False

CACHE_DIR = Path("feature_cache_hybrid_measurement_experiments_xxzz")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Measurement experiments
# -----------------------------
MEASUREMENT_EXPERIMENTS = [
    {"name": "measure_XXZZ", "measurement_type": "XXZZ"},
]

# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# -----------------------------
# Helpers
# -----------------------------
def get_num_output_channels(kernel_size: int) -> int:
    return kernel_size**2 if num_output_channels == -1 else num_output_channels

def get_features_per_circuit(patch_len: int, measurement_type: str) -> Tuple[int, int]:
    n_qubits = int(math.ceil(math.log2(patch_len)))

    if measurement_type in {"Z", "X", "Y", "ZZ", "XXZZ"}:
        feature_len = n_qubits
    elif measurement_type == "XZ":
        feature_len = 2 * n_qubits
    else:
        raise ValueError(f"Unknown measurement_type: {measurement_type}")

    return n_qubits, feature_len

def mean_std(values):
    arr = np.array(values, dtype=np.float64)
    return float(arr.mean()), float(arr.std(ddof=0))

def output_shape_for_patching(image_size: int, patch_size: int, stride: int) -> Tuple[int, int]:
    out_h = (image_size - patch_size) // stride + 1
    out_w = (image_size - patch_size) // stride + 1
    return out_h, out_w

def validate_patch_config(image_size: int, patch_size: int, stride: int):
    if (image_size - patch_size) % stride != 0:
        raise ValueError(
            f"Invalid config: image_size={image_size}, patch_size={patch_size}, stride={stride}"
        )

def config_tag(exp_cfg: Dict) -> str:
    return (
        f"{exp_cfg['name']}"
        f"_p{patch_size}_s{stride}"
        f"_c{get_num_output_channels(patch_size)}"
        f"_tr{N_TRAIN}_val{N_VAL}_te{N_TEST}"
    )

# -----------------------------
# PCA helpers
# -----------------------------
def flatten_feature_maps(x: np.ndarray) -> np.ndarray:
    return x.reshape(x.shape[0], -1).astype(np.float64)

def compute_pca_metrics(
    x: np.ndarray,
    max_components: int = 20,
    variance_threshold: float = 0.90
) -> Dict[str, float]:
    X = flatten_feature_maps(x)
    X = X - X.mean(axis=0, keepdims=True)

    if X.shape[0] < 2 or np.allclose(X, 0.0):
        return {
            "pca_top5_var": 0.0,
            "pca_top10_var": 0.0,
            "pca_top20_var": 0.0,
            "pca_num_for_90": 0.0,
            "pca_effective_rank": 0.0,
        }

    _, s, _ = np.linalg.svd(X, full_matrices=False)
    eigvals = (s ** 2) / max(X.shape[0] - 1, 1)
    total_var = eigvals.sum()

    if total_var <= 1e-12:
        return {
            "pca_top5_var": 0.0,
            "pca_top10_var": 0.0,
            "pca_top20_var": 0.0,
            "pca_num_for_90": 0.0,
            "pca_effective_rank": 0.0,
        }

    explained = eigvals / total_var
    cumsum = np.cumsum(explained)

    def topk_var(k: int) -> float:
        k = min(k, len(explained))
        return float(explained[:k].sum())

    num_for_threshold = int(np.searchsorted(cumsum, variance_threshold) + 1)

    eps = 1e-12
    p = explained[explained > eps]
    entropy = -np.sum(p * np.log(p))
    effective_rank = float(np.exp(entropy))

    return {
        "pca_top5_var": topk_var(5),
        "pca_top10_var": topk_var(10),
        "pca_top20_var": topk_var(max_components),
        "pca_num_for_90": float(num_for_threshold),
        "pca_effective_rank": effective_rank,
    }

# ============================================================
# HYBRID QUANTUM FEATURE EXTRACTOR
# ============================================================
_QNODE_CACHE: Dict[Tuple[int, str], qml.QNode] = {}

def get_hybrid_qnode(n_qubits: int, measurement_type: str):
    key = (n_qubits, measurement_type)
    if key in _QNODE_CACHE:
        return _QNODE_CACHE[key]

    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch")
    def qnode(state_vec, thetas):
        wires = list(range(n_qubits))

        # Fixed circuit for all measurement experiments
        qml.AmplitudeEmbedding(state_vec, wires=wires, normalize=True)

        for i in range(n_qubits):
            qml.RY(thetas[i], wires=i)

        if n_qubits > 1:
            for i in range(n_qubits):
                qml.CZ(wires=[i, (i + 1) % n_qubits])

        # Only the measurement changes
        if measurement_type == "Z":
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
        elif measurement_type == "X":
            return [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]
        elif measurement_type == "Y":
            return [qml.expval(qml.PauliY(i)) for i in range(n_qubits)]
        elif measurement_type == "ZZ":
            if n_qubits == 1:
                return [qml.expval(qml.PauliZ(0))]
            return [
                qml.expval(qml.PauliZ(i) @ qml.PauliZ((i + 1) % n_qubits))
                for i in range(n_qubits)
            ]
        elif measurement_type == "XXZZ":
            half = n_qubits // 2
            x_part = [qml.expval(qml.PauliX(i)) for i in range(half)]
            z_part = [qml.expval(qml.PauliZ(i)) for i in range(half, n_qubits)]
            return x_part + z_part
        elif measurement_type == "XZ":
            x_vals = [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]
            z_vals = [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
            return x_vals + z_vals
        else:
            raise ValueError(f"Unknown measurement_type: {measurement_type}")

    _QNODE_CACHE[key] = qnode
    return qnode

_HYBRID_THETA_BANK: Optional[np.ndarray] = None
_HYBRID_THETA_META: Optional[Tuple[int, int, int]] = None
# meta = (seed, num_circuits, n_qubits)

def get_hybrid_theta_bank(num_circuits: int, n_qubits: int, seed: int) -> np.ndarray:
    global _HYBRID_THETA_BANK, _HYBRID_THETA_META

    if _HYBRID_THETA_BANK is not None and _HYBRID_THETA_META == (seed, num_circuits, n_qubits):
        return _HYBRID_THETA_BANK

    rng = np.random.default_rng(seed)
    bank = rng.uniform(0.0, 2 * np.pi, size=(num_circuits, n_qubits)).astype(np.float32)

    _HYBRID_THETA_BANK = bank
    _HYBRID_THETA_META = (seed, num_circuits, n_qubits)
    return bank

def hybrid_connector(vector: np.ndarray, thetas: np.ndarray, measurement_type: str) -> np.ndarray:
    vec = vector.astype(np.float32)

    n_qubits = int(math.ceil(np.log2(vec.shape[0])))
    target_len = 2 ** n_qubits

    if vec.shape[0] < target_len:
        vec = np.concatenate(
            [vec, np.zeros(target_len - vec.shape[0], dtype=np.float32)],
            axis=0
        )

    if thetas.shape[0] != n_qubits:
        raise ValueError(f"thetas has length {thetas.shape[0]} but n_qubits={n_qubits}.")

    qnode = get_hybrid_qnode(n_qubits, measurement_type)
    feats_t = qnode(torch.tensor(vec), torch.tensor(thetas))
    feats = np.asarray(feats_t, dtype=np.float32)

    if L2_NORMALIZE_HYBRID_FEATURES:
        norm = np.linalg.norm(feats)
        if norm > 1e-12:
            feats = feats / norm

    return feats

def hybrid_patch_features(
    image: np.ndarray,
    seed: int,
    patch_size: int,
    stride: int,
    measurement_type: str
) -> np.ndarray:
    img = np.squeeze(image).astype(np.float32)
    image_size = img.shape[0]

    validate_patch_config(image_size=image_size, patch_size=patch_size, stride=stride)

    num_deep = get_num_output_channels(patch_size)
    patch_len = patch_size * patch_size
    n_qubits, feature_len = get_features_per_circuit(patch_len, measurement_type)
    num_circuits = int(math.ceil(num_deep / feature_len))

    theta_bank = get_hybrid_theta_bank(
        num_circuits=num_circuits,
        n_qubits=n_qubits,
        seed=seed
    )

    out_h, out_w = output_shape_for_patching(image_size, patch_size, stride)
    out = np.zeros((out_h, out_w, num_deep), dtype=np.float32)

    out_i = 0
    for i in range(0, image_size - patch_size + 1, stride):
        out_j = 0
        for j in range(0, image_size - patch_size + 1, stride):
            sub = img[i:i + patch_size, j:j + patch_size].copy()

            if np.all(sub == 0):
                sub.flat[0] = 1.0

            flat = sub.flatten()
            pnorm = np.linalg.norm(flat)
            if pnorm < 1e-12:
                pnorm = 1.0
            flat = flat / pnorm

            feats = []
            for c in range(num_circuits):
                feats.append(hybrid_connector(flat, theta_bank[c], measurement_type))

            all_feats = np.concatenate(feats, axis=0)
            out[out_i, out_j, :num_deep] = all_feats[:num_deep]
            out_j += 1
        out_i += 1

    return out

def hybrid_converter(data: np.ndarray, seed: int, measurement_type: str) -> np.ndarray:
    out = []
    N = len(data)
    print(f"\n=== Hybrid preprocessing started | measurement={measurement_type} ===")
    for idx, x in enumerate(data):
        print(f"Processing image {idx+1}/{N}", end="\r")
        out.append(
            hybrid_patch_features(
                x,
                seed=seed,
                patch_size=patch_size,
                stride=stride,
                measurement_type=measurement_type
            )
        )
    print(f"\n=== Hybrid preprocessing finished | measurement={measurement_type} ===\n")
    return np.array(out, dtype=np.float32)

# ============================================================
# DATA LOADING WITH SHARED RAW SPLITS
# ============================================================
def load_fashion_mnist_shared_pool(seed: int, n_train: int, n_val: int, n_test: int):
    rng = random.Random(seed)
    tfm = T.Compose([T.ToTensor()])

    train_ds = torchvision.datasets.FashionMNIST(root="data", train=True, download=True, transform=tfm)
    test_ds = torchvision.datasets.FashionMNIST(root="data", train=False, download=True, transform=tfm)

    train_idx = rng.sample(range(len(train_ds)), n_train + n_val)
    test_idx = rng.sample(range(len(test_ds)), n_test)

    x_train = np.asarray(
        [train_ds[i][0].numpy().transpose(1, 2, 0) for i in train_idx[:n_train]],
        dtype=np.float32
    )
    y_train = np.asarray(
        [train_ds[i][1] for i in train_idx[:n_train]],
        dtype=np.int64
    )

    x_val = np.asarray(
        [train_ds[i][0].numpy().transpose(1, 2, 0) for i in train_idx[n_train:n_train + n_val]],
        dtype=np.float32
    )
    y_val = np.asarray(
        [train_ds[i][1] for i in train_idx[n_train:n_train + n_val]],
        dtype=np.int64
    )

    x_test = np.asarray(
        [test_ds[i][0].numpy().transpose(1, 2, 0) for i in test_idx],
        dtype=np.float32
    )
    y_test = np.asarray(
        [test_ds[i][1] for i in test_idx],
        dtype=np.int64
    )

    return x_train, y_train, x_val, y_val, x_test, y_test

# ============================================================
# CACHE HELPERS
# ============================================================
def raw_cache_path(seed: int) -> Path:
    return CACHE_DIR / f"raw_seed{seed}_tr{N_TRAIN}_val{N_VAL}_te{N_TEST}.npz"

def feature_cache_path(seed: int, exp_cfg: Dict) -> Path:
    return CACHE_DIR / f"hybrid_seed{seed}_{config_tag(exp_cfg)}.npz"

def save_npz(path: Path, **arrays):
    np.savez_compressed(path, **arrays)

def load_or_build_raw_pool(seed: int):
    path = raw_cache_path(seed)
    if path.exists():
        data = np.load(path)
        return (
            data["x_train"], data["y_train"],
            data["x_val"], data["y_val"],
            data["x_test"], data["y_test"],
        )

    x_train, y_train, x_val, y_val, x_test, y_test = load_fashion_mnist_shared_pool(
        seed=seed, n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST
    )
    save_npz(
        path,
        x_train=x_train, y_train=y_train,
        x_val=x_val, y_val=y_val,
        x_test=x_test, y_test=y_test
    )
    return x_train, y_train, x_val, y_val, x_test, y_test

def load_or_build_feature_cache(seed: int, exp_cfg: Dict):
    path = feature_cache_path(seed, exp_cfg)
    if path.exists():
        data = np.load(path, allow_pickle=True)
        return {
            "x_train": data["x_train"],
            "y_train": data["y_train"],
            "x_val": data["x_val"],
            "y_val": data["y_val"],
            "x_test": data["x_test"],
            "y_test": data["y_test"],
            "preprocess_time_sec": float(data["preprocess_time_sec"]),
            "feature_shape": tuple(data["feature_shape"]),
        }

    x_train, y_train, x_val, y_val, x_test, y_test = load_or_build_raw_pool(seed)

    start = time.perf_counter()
    fx_train = hybrid_converter(x_train, seed=seed, measurement_type=exp_cfg["measurement_type"])
    fx_val = hybrid_converter(x_val, seed=seed, measurement_type=exp_cfg["measurement_type"])
    fx_test = hybrid_converter(x_test, seed=seed, measurement_type=exp_cfg["measurement_type"])
    preprocess_time = time.perf_counter() - start

    feature_shape = np.array(fx_train.shape[1:], dtype=np.int64)

    save_npz(
        path,
        x_train=fx_train, y_train=y_train,
        x_val=fx_val, y_val=y_val,
        x_test=fx_test, y_test=y_test,
        preprocess_time_sec=np.array(preprocess_time, dtype=np.float64),
        feature_shape=feature_shape,
    )

    return {
        "x_train": fx_train,
        "y_train": y_train,
        "x_val": fx_val,
        "y_val": y_val,
        "x_test": fx_test,
        "y_test": y_test,
        "preprocess_time_sec": preprocess_time,
        "feature_shape": tuple(fx_train.shape[1:]),
    }

# ============================================================
# SHARED MLP HEAD
# ============================================================
class HybridModel(nn.Module):
    def __init__(self, in_dim: int, num_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        return self.net(x)

def eval_loader(model, loader, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)
    return loss_sum / total, correct / total

def train_model(Xtr, Ytr, Xva, Yva, Xte, Yte):
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(Xva, Yva), batch_size=64, shuffle=False)
    test_loader = DataLoader(TensorDataset(Xte, Yte), batch_size=64, shuffle=False)

    in_dim = int(np.prod(Xtr.shape[1:]))
    model = HybridModel(in_dim=in_dim).to(device_torch)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_acc = -1.0
    best_state = None
    best_train_acc = 0.0
    best_train_loss = 0.0
    best_val_loss = 0.0

    train_start = time.perf_counter()

    for epoch in range(1, n_epochs + 1):
        model.train()
        total, correct, loss_sum = 0, 0, 0.0

        for xb, yb in train_loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)

        train_loss = loss_sum / total
        train_acc = correct / total
        val_loss, val_acc = eval_loader(model, val_loader, criterion)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_train_loss = train_loss
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:03d}/{n_epochs} | "
            f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.3f}"
        )

    training_time = time.perf_counter() - train_start

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_acc = eval_loader(model, test_loader, criterion)

    return {
        "best_train_loss": best_train_loss,
        "best_train_acc": best_train_acc,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "training_time_sec": training_time,
    }

# ============================================================
# EXPERIMENT RUNNERS
# ============================================================
def run_one_from_cache(seed: int, exp_cfg: Dict):
    set_seed(seed)

    cache = load_or_build_feature_cache(seed=seed, exp_cfg=exp_cfg)

    Xtr_np = cache["x_train"]
    Ytr_np = cache["y_train"]
    Xva_np = cache["x_val"]
    Yva_np = cache["y_val"]
    Xte_np = cache["x_test"]
    Yte_np = cache["y_test"]

    pca_metrics = compute_pca_metrics(Xtr_np, max_components=20, variance_threshold=0.90)

    Xtr = torch.tensor(Xtr_np, dtype=torch.float32)
    Ytr = torch.tensor(Ytr_np, dtype=torch.long)
    Xva = torch.tensor(Xva_np, dtype=torch.float32)
    Yva = torch.tensor(Yva_np, dtype=torch.long)
    Xte = torch.tensor(Xte_np, dtype=torch.float32)
    Yte = torch.tensor(Yte_np, dtype=torch.long)

    metrics = train_model(Xtr, Ytr, Xva, Yva, Xte, Yte)
    metrics.update(pca_metrics)

    metrics["preprocess_time_sec"] = cache["preprocess_time_sec"]
    metrics["feature_shape"] = cache["feature_shape"]
    metrics["measurement_type"] = exp_cfg["measurement_type"]

    image_size = 28
    patch_len = patch_size * patch_size
    n_qubits, features_per_circuit = get_features_per_circuit(patch_len, exp_cfg["measurement_type"])
    num_circuits = int(math.ceil(get_num_output_channels(patch_size) / features_per_circuit))
    out_h, out_w = output_shape_for_patching(image_size, patch_size, stride)

    metrics["patch_size"] = patch_size
    metrics["stride"] = stride
    metrics["n_qubits"] = n_qubits
    metrics["features_per_circuit"] = features_per_circuit
    metrics["num_circuits"] = num_circuits
    metrics["patches_per_image"] = out_h * out_w

    return metrics

def summarize_results(results, exp_cfg):
    print(f"\n===== SUMMARY | {exp_cfg['name']} =====")

    keys = [
        "best_train_acc",
        "best_val_acc",
        "test_acc",
        "best_train_loss",
        "best_val_loss",
        "test_loss",
        "preprocess_time_sec",
        "training_time_sec",
        "pca_top5_var",
        "pca_top10_var",
        "pca_top20_var",
        "pca_num_for_90",
        "pca_effective_rank",
    ]

    for k in keys:
        mu, sd = mean_std([r[k] for r in results])
        print(f"{k:20s}: {mu:.6f} ± {sd:.6f}")

    print(f"measurement_type     : {results[0]['measurement_type']}")
    print(f"feature_shape        : {results[0]['feature_shape']}")
    print(f"patch_size           : {results[0]['patch_size']}")
    print(f"stride               : {results[0]['stride']}")
    print(f"n_qubits             : {results[0]['n_qubits']}")
    print(f"features_per_circuit : {results[0]['features_per_circuit']}")
    print(f"num_circuits         : {results[0]['num_circuits']}")
    print(f"patches_per_image    : {results[0]['patches_per_image']}")

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    print(f"Using device: {device_torch}")
    print(f"Seeds: {EXPERIMENT_SEEDS}")
    print(f"n_train: {N_TRAIN}")
    print(f"Patch size: {patch_size}")
    print(f"Stride: {stride}")
    print(f"Output channels: {num_output_channels}")
    print(f"Cache dir: {CACHE_DIR.resolve()}")

    validate_patch_config(28, patch_size, stride)

    out_h, out_w = output_shape_for_patching(28, patch_size, stride)
    patch_len = patch_size * patch_size
    print(
        f"\nFixed config: out=({out_h},{out_w}), patches/image={out_h*out_w}"
    )

    print("\nMeasurements:")
    for exp_cfg in MEASUREMENT_EXPERIMENTS:
        n_qubits, features_per_circuit = get_features_per_circuit(patch_len, exp_cfg["measurement_type"])
        num_circuits = int(math.ceil(get_num_output_channels(patch_size) / features_per_circuit))
        print(
            f"  {exp_cfg['name']}: measurement={exp_cfg['measurement_type']}, "
            f"n_qubits={n_qubits}, features/circuit={features_per_circuit}, circuits/patch={num_circuits}"
        )

    # Build raw cache once per seed
    for seed in EXPERIMENT_SEEDS:
        print(f"\nPreparing shared raw pool for seed={seed}")
        load_or_build_raw_pool(seed)

        for exp_cfg in MEASUREMENT_EXPERIMENTS:
            print(f"Preparing/loading hybrid cache for seed={seed}, measurement={exp_cfg['measurement_type']}")
            load_or_build_feature_cache(seed=seed, exp_cfg=exp_cfg)

    all_results = {}

    for exp_cfg in MEASUREMENT_EXPERIMENTS:
        exp_results = []

        for seed in EXPERIMENT_SEEDS:
            print(f"\n\n########## {exp_cfg['name']} | seed={seed} ##########")
            result = run_one_from_cache(seed=seed, exp_cfg=exp_cfg)
            exp_results.append(result)

            print("\n--- Run result ---")
            for k, v in result.items():
                print(f"{k}: {v}")

        all_results[exp_cfg["name"]] = exp_results
        summarize_results(exp_results, exp_cfg)

    print("\n\n================ FINAL AGGREGATED SUMMARY ================")
    for exp_name, runs in all_results.items():
        test_mu, test_sd = mean_std([r["test_acc"] for r in runs])
        val_mu, val_sd = mean_std([r["best_val_acc"] for r in runs])
        prep_mu, prep_sd = mean_std([r["preprocess_time_sec"] for r in runs])
        train_mu, train_sd = mean_std([r["training_time_sec"] for r in runs])
        pca_rank_mu, pca_rank_sd = mean_std([r["pca_effective_rank"] for r in runs])
        pca90_mu, pca90_sd = mean_std([r["pca_num_for_90"] for r in runs])

        meta = runs[0]
        print(
            f"{exp_name:12s} | "
            f"meas={meta['measurement_type']} | "
            f"val_acc={val_mu:.4f}±{val_sd:.4f} | "
            f"test_acc={test_mu:.4f}±{test_sd:.4f} | "
            f"pca_rank={pca_rank_mu:.2f}±{pca_rank_sd:.2f} | "
            f"pca90={pca90_mu:.2f}±{pca90_sd:.2f} | "
            f"prep_time={prep_mu:.2f}±{prep_sd:.2f}s | "
            f"train_time={train_mu:.2f}±{train_sd:.2f}s"
        )

Using device: cpu
Seeds: [246, 247, 248]
n_train: 500
Patch size: 4
Stride: 4
Output channels: 4
Cache dir: C:\Users\Asus\qml\coursework\feature_cache_hybrid_measurement_experiments_xxzz

Fixed config: out=(7,7), patches/image=49

Measurements:
  measure_XXZZ: measurement=XXZZ, n_qubits=4, features/circuit=4, circuits/patch=1

Preparing shared raw pool for seed=246
Preparing/loading hybrid cache for seed=246, measurement=XXZZ

=== Hybrid preprocessing started | measurement=XXZZ ===
Processing image 500/500
=== Hybrid preprocessing finished | measurement=XXZZ ===


=== Hybrid preprocessing started | measurement=XXZZ ===
Processing image 100/100
=== Hybrid preprocessing finished | measurement=XXZZ ===


=== Hybrid preprocessing started | measurement=XXZZ ===
Processing image 100/100
=== Hybrid preprocessing finished | measurement=XXZZ ===


Preparing shared raw pool for seed=247
Preparing/loading hybrid cache for seed=247, measurement=XXZZ

=== Hybrid preprocessing started | measurement=

# Experiment D — Entanglement Topology Study

Compare circuit entanglement structures (e.g., chain vs circular) while holding other settings fixed.


In [22]:
import math
import random
import time
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as T
import pennylane as qml

# ============================================================
# HYBRID ENTANGLEMENT EXPERIMENT
#
# Fixed circuit:
#   AmplitudeEmbedding
#   -> random RY layer
#   -> entanglement pattern (THIS is what changes)
#   -> Z measurement
#
# Entanglement experiments:
#   NONE  : no entanglement
#   CHAIN : CZ(0,1), CZ(1,2), ..., CZ(n-2,n-1)
#   RING  : CHAIN + CZ(n-1,0)
#   FULL  : CZ on all qubit pairs
#
# Everything else is held fixed so you can isolate the effect
# of entanglement on representation quality and downstream accuracy.
# ============================================================

# -----------------------------
# Global config
# -----------------------------
device_torch = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_SEED = 246
EXPERIMENT_SEEDS = [246, 247, 248]

N_TRAIN = 1200
N_VAL = 300
N_TEST = 300

batch_size = 4
n_epochs = 100
lr = 1e-3
weight_decay = 1e-4

# Fixed patch setup
patch_size = 4
stride = 4

num_output_channels = 16
L2_NORMALIZE_HYBRID_FEATURES = False

CACHE_DIR = Path("feature_cache_hybrid_entanglement_experiments")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Entanglement experiments
# -----------------------------
ENTANGLEMENT_EXPERIMENTS = [
    {"name": "ent_none", "measurement_type": "Z", "entanglement": "NONE"},
    {"name": "ent_chain", "measurement_type": "Z", "entanglement": "CHAIN"},
    {"name": "ent_ring", "measurement_type": "Z", "entanglement": "RING"},
    {"name": "ent_full", "measurement_type": "Z", "entanglement": "FULL"},
]

# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# -----------------------------
# Helpers
# -----------------------------
def get_num_output_channels(kernel_size: int) -> int:
    return kernel_size**2 if num_output_channels == -1 else num_output_channels

def get_features_per_circuit(patch_len: int, measurement_type: str) -> Tuple[int, int]:
    n_qubits = int(math.ceil(math.log2(patch_len)))

    if measurement_type in {"Z", "X", "Y", "ZZ", "XXZZ"}:
        feature_len = n_qubits
    elif measurement_type == "XZ":
        feature_len = 2 * n_qubits
    else:
        raise ValueError(f"Unknown measurement_type: {measurement_type}")

    return n_qubits, feature_len

def mean_std(values):
    arr = np.array(values, dtype=np.float64)
    return float(arr.mean()), float(arr.std(ddof=0))

def output_shape_for_patching(image_size: int, patch_size: int, stride: int) -> Tuple[int, int]:
    out_h = (image_size - patch_size) // stride + 1
    out_w = (image_size - patch_size) // stride + 1
    return out_h, out_w

def validate_patch_config(image_size: int, patch_size: int, stride: int):
    if (image_size - patch_size) % stride != 0:
        raise ValueError(
            f"Invalid config: image_size={image_size}, patch_size={patch_size}, stride={stride}"
        )

def config_tag(exp_cfg: Dict) -> str:
    return (
        f"{exp_cfg['name']}"
        f"_meas{exp_cfg['measurement_type']}"
        f"_ent{exp_cfg['entanglement']}"
        f"_p{patch_size}_s{stride}"
        f"_c{get_num_output_channels(patch_size)}"
        f"_tr{N_TRAIN}_val{N_VAL}_te{N_TEST}"
    )

# -----------------------------
# PCA helpers
# -----------------------------
def flatten_feature_maps(x: np.ndarray) -> np.ndarray:
    return x.reshape(x.shape[0], -1).astype(np.float64)

def compute_pca_metrics(
    x: np.ndarray,
    max_components: int = 20,
    variance_threshold: float = 0.90
) -> Dict[str, float]:
    X = flatten_feature_maps(x)
    X = X - X.mean(axis=0, keepdims=True)

    if X.shape[0] < 2 or np.allclose(X, 0.0):
        return {
            "pca_top5_var": 0.0,
            "pca_top10_var": 0.0,
            "pca_top20_var": 0.0,
            "pca_num_for_90": 0.0,
            "pca_effective_rank": 0.0,
        }

    _, s, _ = np.linalg.svd(X, full_matrices=False)
    eigvals = (s ** 2) / max(X.shape[0] - 1, 1)
    total_var = eigvals.sum()

    if total_var <= 1e-12:
        return {
            "pca_top5_var": 0.0,
            "pca_top10_var": 0.0,
            "pca_top20_var": 0.0,
            "pca_num_for_90": 0.0,
            "pca_effective_rank": 0.0,
        }

    explained = eigvals / total_var
    cumsum = np.cumsum(explained)

    def topk_var(k: int) -> float:
        k = min(k, len(explained))
        return float(explained[:k].sum())

    num_for_threshold = int(np.searchsorted(cumsum, variance_threshold) + 1)

    eps = 1e-12
    p = explained[explained > eps]
    entropy = -np.sum(p * np.log(p))
    effective_rank = float(np.exp(entropy))

    return {
        "pca_top5_var": topk_var(5),
        "pca_top10_var": topk_var(10),
        "pca_top20_var": topk_var(max_components),
        "pca_num_for_90": float(num_for_threshold),
        "pca_effective_rank": effective_rank,
    }

# ============================================================
# HYBRID QUANTUM FEATURE EXTRACTOR
# ============================================================
_QNODE_CACHE: Dict[Tuple[int, str, str], qml.QNode] = {}

def apply_entanglement(n_qubits: int, entanglement: str):
    if n_qubits <= 1 or entanglement == "NONE":
        return

    if entanglement == "CHAIN":
        for i in range(n_qubits - 1):
            qml.CZ(wires=[i, i + 1])

    elif entanglement == "RING":
        for i in range(n_qubits):
            qml.CZ(wires=[i, (i + 1) % n_qubits])

    elif entanglement == "FULL":
        for i in range(n_qubits):
            for j in range(i + 1, n_qubits):
                qml.CZ(wires=[i, j])

    else:
        raise ValueError(f"Unknown entanglement: {entanglement}")

def get_hybrid_qnode(n_qubits: int, measurement_type: str, entanglement: str):
    key = (n_qubits, measurement_type, entanglement)
    if key in _QNODE_CACHE:
        return _QNODE_CACHE[key]

    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch")
    def qnode(state_vec, thetas):
        wires = list(range(n_qubits))

        qml.AmplitudeEmbedding(state_vec, wires=wires, normalize=True)

        for i in range(n_qubits):
            qml.RY(thetas[i], wires=i)

        apply_entanglement(n_qubits, entanglement)

        if measurement_type == "Z":
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
        elif measurement_type == "X":
            return [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]
        elif measurement_type == "Y":
            return [qml.expval(qml.PauliY(i)) for i in range(n_qubits)]
        elif measurement_type == "ZZ":
            if n_qubits == 1:
                return [qml.expval(qml.PauliZ(0))]
            return [
                qml.expval(qml.PauliZ(i) @ qml.PauliZ((i + 1) % n_qubits))
                for i in range(n_qubits)
            ]
        elif measurement_type == "XXZZ":
            half = n_qubits // 2
            x_part = [qml.expval(qml.PauliX(i)) for i in range(half)]
            z_part = [qml.expval(qml.PauliZ(i)) for i in range(half, n_qubits)]
            return x_part + z_part
        elif measurement_type == "XZ":
            x_vals = [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]
            z_vals = [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
            return x_vals + z_vals
        else:
            raise ValueError(f"Unknown measurement_type: {measurement_type}")

    _QNODE_CACHE[key] = qnode
    return qnode

_HYBRID_THETA_BANK: Optional[np.ndarray] = None
_HYBRID_THETA_META: Optional[Tuple[int, int, int]] = None
# meta = (seed, num_circuits, n_qubits)

def get_hybrid_theta_bank(num_circuits: int, n_qubits: int, seed: int) -> np.ndarray:
    global _HYBRID_THETA_BANK, _HYBRID_THETA_META

    if _HYBRID_THETA_BANK is not None and _HYBRID_THETA_META == (seed, num_circuits, n_qubits):
        return _HYBRID_THETA_BANK

    rng = np.random.default_rng(seed)
    bank = rng.uniform(0.0, 2 * np.pi, size=(num_circuits, n_qubits)).astype(np.float32)

    _HYBRID_THETA_BANK = bank
    _HYBRID_THETA_META = (seed, num_circuits, n_qubits)
    return bank

def hybrid_connector(
    vector: np.ndarray,
    thetas: np.ndarray,
    measurement_type: str,
    entanglement: str
) -> np.ndarray:
    vec = vector.astype(np.float32)

    n_qubits = int(math.ceil(np.log2(vec.shape[0])))
    target_len = 2 ** n_qubits

    if vec.shape[0] < target_len:
        vec = np.concatenate(
            [vec, np.zeros(target_len - vec.shape[0], dtype=np.float32)],
            axis=0
        )

    if thetas.shape[0] != n_qubits:
        raise ValueError(f"thetas has length {thetas.shape[0]} but n_qubits={n_qubits}.")

    qnode = get_hybrid_qnode(n_qubits, measurement_type, entanglement)
    feats_t = qnode(torch.tensor(vec), torch.tensor(thetas))
    feats = np.asarray(feats_t, dtype=np.float32)

    if L2_NORMALIZE_HYBRID_FEATURES:
        norm = np.linalg.norm(feats)
        if norm > 1e-12:
            feats = feats / norm

    return feats

def hybrid_patch_features(
    image: np.ndarray,
    seed: int,
    patch_size: int,
    stride: int,
    measurement_type: str,
    entanglement: str
) -> np.ndarray:
    img = np.squeeze(image).astype(np.float32)
    image_size = img.shape[0]

    validate_patch_config(image_size=image_size, patch_size=patch_size, stride=stride)

    num_deep = get_num_output_channels(patch_size)
    patch_len = patch_size * patch_size
    n_qubits, feature_len = get_features_per_circuit(patch_len, measurement_type)
    num_circuits = int(math.ceil(num_deep / feature_len))

    theta_bank = get_hybrid_theta_bank(
        num_circuits=num_circuits,
        n_qubits=n_qubits,
        seed=seed
    )

    out_h, out_w = output_shape_for_patching(image_size, patch_size, stride)
    out = np.zeros((out_h, out_w, num_deep), dtype=np.float32)

    out_i = 0
    for i in range(0, image_size - patch_size + 1, stride):
        out_j = 0
        for j in range(0, image_size - patch_size + 1, stride):
            sub = img[i:i + patch_size, j:j + patch_size].copy()

            if np.all(sub == 0):
                sub.flat[0] = 1.0

            flat = sub.flatten()
            pnorm = np.linalg.norm(flat)
            if pnorm < 1e-12:
                pnorm = 1.0
            flat = flat / pnorm

            feats = []
            for c in range(num_circuits):
                feats.append(
                    hybrid_connector(
                        flat,
                        theta_bank[c],
                        measurement_type=measurement_type,
                        entanglement=entanglement
                    )
                )

            all_feats = np.concatenate(feats, axis=0)
            out[out_i, out_j, :num_deep] = all_feats[:num_deep]
            out_j += 1
        out_i += 1

    return out

def hybrid_converter(data: np.ndarray, seed: int, measurement_type: str, entanglement: str) -> np.ndarray:
    out = []
    N = len(data)
    print(
        f"\n=== Hybrid preprocessing started | "
        f"measurement={measurement_type} | entanglement={entanglement} ==="
    )
    for idx, x in enumerate(data):
        print(f"Processing image {idx+1}/{N}", end="\r")
        out.append(
            hybrid_patch_features(
                x,
                seed=seed,
                patch_size=patch_size,
                stride=stride,
                measurement_type=measurement_type,
                entanglement=entanglement
            )
        )
    print(
        f"\n=== Hybrid preprocessing finished | "
        f"measurement={measurement_type} | entanglement={entanglement} ===\n"
    )
    return np.array(out, dtype=np.float32)

# ============================================================
# DATA LOADING WITH SHARED RAW SPLITS
# ============================================================
def load_fashion_mnist_shared_pool(seed: int, n_train: int, n_val: int, n_test: int):
    rng = random.Random(seed)
    tfm = T.Compose([T.ToTensor()])

    train_ds = torchvision.datasets.FashionMNIST(
        root="data",
        train=True,
        download=True,
        transform=tfm
    )
    test_ds = torchvision.datasets.FashionMNIST(
        root="data",
        train=False,
        download=True,
        transform=tfm
    )

    train_idx = rng.sample(range(len(train_ds)), n_train + n_val)
    test_idx = rng.sample(range(len(test_ds)), n_test)

    x_train = np.asarray(
        [train_ds[i][0].numpy().transpose(1, 2, 0) for i in train_idx[:n_train]],
        dtype=np.float32
    )
    y_train = np.asarray(
        [train_ds[i][1] for i in train_idx[:n_train]],
        dtype=np.int64
    )

    x_val = np.asarray(
        [train_ds[i][0].numpy().transpose(1, 2, 0) for i in train_idx[n_train:n_train + n_val]],
        dtype=np.float32
    )
    y_val = np.asarray(
        [train_ds[i][1] for i in train_idx[n_train:n_train + n_val]],
        dtype=np.int64
    )

    x_test = np.asarray(
        [test_ds[i][0].numpy().transpose(1, 2, 0) for i in test_idx],
        dtype=np.float32
    )
    y_test = np.asarray(
        [test_ds[i][1] for i in test_idx],
        dtype=np.int64
    )

    return x_train, y_train, x_val, y_val, x_test, y_test

# ============================================================
# CACHE HELPERS
# ============================================================
def raw_cache_path(seed: int) -> Path:
    return CACHE_DIR / f"raw_seed{seed}_tr{N_TRAIN}_val{N_VAL}_te{N_TEST}.npz"

def feature_cache_path(seed: int, exp_cfg: Dict) -> Path:
    return CACHE_DIR / f"hybrid_seed{seed}_{config_tag(exp_cfg)}.npz"

def save_npz(path: Path, **arrays):
    np.savez_compressed(path, **arrays)

def load_or_build_raw_pool(seed: int):
    path = raw_cache_path(seed)
    if path.exists():
        data = np.load(path)
        return (
            data["x_train"], data["y_train"],
            data["x_val"], data["y_val"],
            data["x_test"], data["y_test"],
        )

    x_train, y_train, x_val, y_val, x_test, y_test = load_fashion_mnist_shared_pool(
        seed=seed, n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST
    )
    save_npz(
        path,
        x_train=x_train, y_train=y_train,
        x_val=x_val, y_val=y_val,
        x_test=x_test, y_test=y_test
    )
    return x_train, y_train, x_val, y_val, x_test, y_test

def load_or_build_feature_cache(seed: int, exp_cfg: Dict):
    path = feature_cache_path(seed, exp_cfg)
    if path.exists():
        data = np.load(path, allow_pickle=True)
        return {
            "x_train": data["x_train"],
            "y_train": data["y_train"],
            "x_val": data["x_val"],
            "y_val": data["y_val"],
            "x_test": data["x_test"],
            "y_test": data["y_test"],
            "preprocess_time_sec": float(data["preprocess_time_sec"]),
            "feature_shape": tuple(data["feature_shape"]),
        }

    x_train, y_train, x_val, y_val, x_test, y_test = load_or_build_raw_pool(seed)

    start = time.perf_counter()
    fx_train = hybrid_converter(
        x_train,
        seed=seed,
        measurement_type=exp_cfg["measurement_type"],
        entanglement=exp_cfg["entanglement"]
    )
    fx_val = hybrid_converter(
        x_val,
        seed=seed,
        measurement_type=exp_cfg["measurement_type"],
        entanglement=exp_cfg["entanglement"]
    )
    fx_test = hybrid_converter(
        x_test,
        seed=seed,
        measurement_type=exp_cfg["measurement_type"],
        entanglement=exp_cfg["entanglement"]
    )
    preprocess_time = time.perf_counter() - start

    feature_shape = np.array(fx_train.shape[1:], dtype=np.int64)

    save_npz(
        path,
        x_train=fx_train, y_train=y_train,
        x_val=fx_val, y_val=y_val,
        x_test=fx_test, y_test=y_test,
        preprocess_time_sec=np.array(preprocess_time, dtype=np.float64),
        feature_shape=feature_shape,
    )

    return {
        "x_train": fx_train,
        "y_train": y_train,
        "x_val": fx_val,
        "y_val": y_val,
        "x_test": fx_test,
        "y_test": y_test,
        "preprocess_time_sec": preprocess_time,
        "feature_shape": tuple(fx_train.shape[1:]),
    }

# ============================================================
# SHARED MLP HEAD
# ============================================================
class HybridModel(nn.Module):
    def __init__(self, in_dim: int, num_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        return self.net(x)

def eval_loader(model, loader, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)
    return loss_sum / total, correct / total

def train_model(Xtr, Ytr, Xva, Yva, Xte, Yte):
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(Xva, Yva), batch_size=64, shuffle=False)
    test_loader = DataLoader(TensorDataset(Xte, Yte), batch_size=64, shuffle=False)

    in_dim = int(np.prod(Xtr.shape[1:]))
    model = HybridModel(in_dim=in_dim).to(device_torch)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_acc = -1.0
    best_state = None
    best_train_acc = 0.0
    best_train_loss = 0.0
    best_val_loss = 0.0

    train_start = time.perf_counter()

    for epoch in range(1, n_epochs + 1):
        model.train()
        total, correct, loss_sum = 0, 0, 0.0

        for xb, yb in train_loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)

        train_loss = loss_sum / total
        train_acc = correct / total
        val_loss, val_acc = eval_loader(model, val_loader, criterion)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_train_loss = train_loss
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:03d}/{n_epochs} | "
            f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.3f}"
        )

    training_time = time.perf_counter() - train_start

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_acc = eval_loader(model, test_loader, criterion)

    return {
        "best_train_loss": best_train_loss,
        "best_train_acc": best_train_acc,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "training_time_sec": training_time,
    }

# ============================================================
# EXPERIMENT RUNNERS
# ============================================================
def run_one_from_cache(seed: int, exp_cfg: Dict):
    set_seed(seed)

    cache = load_or_build_feature_cache(seed=seed, exp_cfg=exp_cfg)

    Xtr_np = cache["x_train"]
    Ytr_np = cache["y_train"]
    Xva_np = cache["x_val"]
    Yva_np = cache["y_val"]
    Xte_np = cache["x_test"]
    Yte_np = cache["y_test"]

    pca_metrics = compute_pca_metrics(Xtr_np, max_components=20, variance_threshold=0.90)

    Xtr = torch.tensor(Xtr_np, dtype=torch.float32)
    Ytr = torch.tensor(Ytr_np, dtype=torch.long)
    Xva = torch.tensor(Xva_np, dtype=torch.float32)
    Yva = torch.tensor(Yva_np, dtype=torch.long)
    Xte = torch.tensor(Xte_np, dtype=torch.float32)
    Yte = torch.tensor(Yte_np, dtype=torch.long)

    metrics = train_model(Xtr, Ytr, Xva, Yva, Xte, Yte)
    metrics.update(pca_metrics)

    metrics["preprocess_time_sec"] = cache["preprocess_time_sec"]
    metrics["feature_shape"] = cache["feature_shape"]
    metrics["measurement_type"] = exp_cfg["measurement_type"]
    metrics["entanglement"] = exp_cfg["entanglement"]

    image_size = 28
    patch_len = patch_size * patch_size
    n_qubits, features_per_circuit = get_features_per_circuit(patch_len, exp_cfg["measurement_type"])
    num_circuits = int(math.ceil(get_num_output_channels(patch_size) / features_per_circuit))
    out_h, out_w = output_shape_for_patching(image_size, patch_size, stride)

    metrics["patch_size"] = patch_size
    metrics["stride"] = stride
    metrics["n_qubits"] = n_qubits
    metrics["features_per_circuit"] = features_per_circuit
    metrics["num_circuits"] = num_circuits
    metrics["patches_per_image"] = out_h * out_w

    return metrics

def summarize_results(results, exp_cfg):
    print(f"\n===== SUMMARY | {exp_cfg['name']} =====")

    keys = [
        "best_train_acc",
        "best_val_acc",
        "test_acc",
        "best_train_loss",
        "best_val_loss",
        "test_loss",
        "preprocess_time_sec",
        "training_time_sec",
        "pca_top5_var",
        "pca_top10_var",
        "pca_top20_var",
        "pca_num_for_90",
        "pca_effective_rank",
    ]

    for k in keys:
        mu, sd = mean_std([r[k] for r in results])
        print(f"{k:20s}: {mu:.6f} ± {sd:.6f}")

    print(f"measurement_type     : {results[0]['measurement_type']}")
    print(f"entanglement         : {results[0]['entanglement']}")
    print(f"feature_shape        : {results[0]['feature_shape']}")
    print(f"patch_size           : {results[0]['patch_size']}")
    print(f"stride               : {results[0]['stride']}")
    print(f"n_qubits             : {results[0]['n_qubits']}")
    print(f"features_per_circuit : {results[0]['features_per_circuit']}")
    print(f"num_circuits         : {results[0]['num_circuits']}")
    print(f"patches_per_image    : {results[0]['patches_per_image']}")

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    print(f"Using device: {device_torch}")
    print(f"Seeds: {EXPERIMENT_SEEDS}")
    print(f"n_train: {N_TRAIN}")
    print(f"Patch size: {patch_size}")
    print(f"Stride: {stride}")
    print(f"Output channels: {num_output_channels}")
    print(f"Cache dir: {CACHE_DIR.resolve()}")

    validate_patch_config(28, patch_size, stride)

    out_h, out_w = output_shape_for_patching(28, patch_size, stride)
    patch_len = patch_size * patch_size
    print(f"\nFixed config: out=({out_h},{out_w}), patches/image={out_h*out_w}")

    print("\nEntanglement experiments:")
    for exp_cfg in ENTANGLEMENT_EXPERIMENTS:
        n_qubits, features_per_circuit = get_features_per_circuit(
            patch_len, exp_cfg["measurement_type"]
        )
        num_circuits = int(math.ceil(get_num_output_channels(patch_size) / features_per_circuit))
        print(
            f"  {exp_cfg['name']}: "
            f"measurement={exp_cfg['measurement_type']}, "
            f"entanglement={exp_cfg['entanglement']}, "
            f"n_qubits={n_qubits}, "
            f"features/circuit={features_per_circuit}, "
            f"circuits/patch={num_circuits}"
        )

    # Build raw cache once per seed
    for seed in EXPERIMENT_SEEDS:
        print(f"\nPreparing shared raw pool for seed={seed}")
        load_or_build_raw_pool(seed)

        for exp_cfg in ENTANGLEMENT_EXPERIMENTS:
            print(
                f"Preparing/loading hybrid cache for seed={seed}, "
                f"measurement={exp_cfg['measurement_type']}, "
                f"entanglement={exp_cfg['entanglement']}"
            )
            load_or_build_feature_cache(seed=seed, exp_cfg=exp_cfg)

    all_results = {}

    for exp_cfg in ENTANGLEMENT_EXPERIMENTS:
        exp_results = []

        for seed in EXPERIMENT_SEEDS:
            print(f"\n\n########## {exp_cfg['name']} | seed={seed} ##########")
            result = run_one_from_cache(seed=seed, exp_cfg=exp_cfg)
            exp_results.append(result)

            print("\n--- Run result ---")
            for k, v in result.items():
                print(f"{k}: {v}")

        all_results[exp_cfg["name"]] = exp_results
        summarize_results(exp_results, exp_cfg)

    print("\n\n================ FINAL AGGREGATED SUMMARY ================")
    for exp_name, runs in all_results.items():
        test_mu, test_sd = mean_std([r["test_acc"] for r in runs])
        val_mu, val_sd = mean_std([r["best_val_acc"] for r in runs])
        prep_mu, prep_sd = mean_std([r["preprocess_time_sec"] for r in runs])
        train_mu, train_sd = mean_std([r["training_time_sec"] for r in runs])
        pca_rank_mu, pca_rank_sd = mean_std([r["pca_effective_rank"] for r in runs])
        pca90_mu, pca90_sd = mean_std([r["pca_num_for_90"] for r in runs])

        meta = runs[0]
        print(
            f"{exp_name:12s} | "
            f"meas={meta['measurement_type']} | "
            f"ent={meta['entanglement']} | "
            f"val_acc={val_mu:.4f}±{val_sd:.4f} | "
            f"test_acc={test_mu:.4f}±{test_sd:.4f} | "
            f"pca_rank={pca_rank_mu:.2f}±{pca_rank_sd:.2f} | "
            f"pca90={pca90_mu:.2f}±{pca90_sd:.2f} | "
            f"prep_time={prep_mu:.2f}±{prep_sd:.2f}s | "
            f"train_time={train_mu:.2f}±{train_sd:.2f}s"
        )

Using device: cpu
Seeds: [246, 247, 248]
n_train: 500
Patch size: 4
Stride: 4
Output channels: 4
Cache dir: C:\Users\Asus\qml\coursework\feature_cache_hybrid_entanglement_experiments

Fixed config: out=(7,7), patches/image=49

Entanglement experiments:
  ent_none: measurement=Z, entanglement=NONE, n_qubits=4, features/circuit=4, circuits/patch=1
  ent_chain: measurement=Z, entanglement=CHAIN, n_qubits=4, features/circuit=4, circuits/patch=1
  ent_ring: measurement=Z, entanglement=RING, n_qubits=4, features/circuit=4, circuits/patch=1
  ent_full: measurement=Z, entanglement=FULL, n_qubits=4, features/circuit=4, circuits/patch=1

Preparing shared raw pool for seed=246
Preparing/loading hybrid cache for seed=246, measurement=Z, entanglement=NONE

=== Hybrid preprocessing started | measurement=Z | entanglement=NONE ===
Processing image 500/500
=== Hybrid preprocessing finished | measurement=Z | entanglement=NONE ===


=== Hybrid preprocessing started | measurement=Z | entanglement=NONE ===

## D1 — Entanglement Depth Sweep (0–3)

Analyze whether increasing entanglement depth improves representation quality or overfits.


In [24]:
import math
import random
import time
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as T
import pennylane as qml

# ============================================================
# HYBRID PROGRESSIVE-ENTANGLEMENT EXPERIMENT
#
# Fixed circuit:
#   AmplitudeEmbedding
#   -> random RY layer
#   -> progressive CZ entanglement
#   -> Z measurement
#
# What changes:
#   entanglement_layers in {0, 1, 2, 3, 4}
#
# Meaning for 4 qubits:
#   0_ent : no entanglement
#   1_ent : CZ(0,1)
#   2_ent : CZ(0,1), CZ(1,2)
#   3_ent : CZ(0,1), CZ(1,2), CZ(2,3)
#   4_ent : CZ(0,1), CZ(1,2), CZ(2,3), CZ(3,0)
#
# Everything else is fixed so you isolate the effect
# of progressively adding entangling pairs.
# ============================================================

# -----------------------------
# Global config
# -----------------------------
device_torch = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_SEED = 246
EXPERIMENT_SEEDS = [246, 247, 248]

N_TRAIN = 500
N_VAL = 100
N_TEST = 100

batch_size = 4
n_epochs = 100
lr = 1e-3
weight_decay = 1e-4

patch_size = 4
stride = 4

num_output_channels = 4
L2_NORMALIZE_HYBRID_FEATURES = False

CACHE_DIR = Path("feature_cache_hybrid_progressive_entanglement_experiments")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Progressive entanglement experiments
# -----------------------------
ENTANGLEMENT_EXPERIMENTS = [
    {"name": "0_ent", "measurement_type": "Z", "entanglement_layers": 0},
    {"name": "1_ent", "measurement_type": "Z", "entanglement_layers": 1},
    {"name": "2_ent", "measurement_type": "Z", "entanglement_layers": 2},
    {"name": "3_ent", "measurement_type": "Z", "entanglement_layers": 3},
    {"name": "4_ent", "measurement_type": "Z", "entanglement_layers": 4},
]

# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# -----------------------------
# Helpers
# -----------------------------
def get_num_output_channels(kernel_size: int) -> int:
    return kernel_size**2 if num_output_channels == -1 else num_output_channels

def get_features_per_circuit(patch_len: int, measurement_type: str) -> Tuple[int, int]:
    n_qubits = int(math.ceil(math.log2(patch_len)))

    if measurement_type in {"Z", "X", "Y", "ZZ", "XXZZ"}:
        feature_len = n_qubits
    elif measurement_type == "XZ":
        feature_len = 2 * n_qubits
    else:
        raise ValueError(f"Unknown measurement_type: {measurement_type}")

    return n_qubits, feature_len

def mean_std(values):
    arr = np.array(values, dtype=np.float64)
    return float(arr.mean()), float(arr.std(ddof=0))

def output_shape_for_patching(image_size: int, patch_size: int, stride: int) -> Tuple[int, int]:
    out_h = (image_size - patch_size) // stride + 1
    out_w = (image_size - patch_size) // stride + 1
    return out_h, out_w

def validate_patch_config(image_size: int, patch_size: int, stride: int):
    if (image_size - patch_size) % stride != 0:
        raise ValueError(
            f"Invalid config: image_size={image_size}, patch_size={patch_size}, stride={stride}"
        )

def config_tag(exp_cfg: Dict) -> str:
    return (
        f"{exp_cfg['name']}"
        f"_meas{exp_cfg['measurement_type']}"
        f"_entlayers{exp_cfg['entanglement_layers']}"
        f"_p{patch_size}_s{stride}"
        f"_c{get_num_output_channels(patch_size)}"
        f"_tr{N_TRAIN}_val{N_VAL}_te{N_TEST}"
    )

# -----------------------------
# PCA helpers
# -----------------------------
def flatten_feature_maps(x: np.ndarray) -> np.ndarray:
    return x.reshape(x.shape[0], -1).astype(np.float64)

def compute_pca_metrics(
    x: np.ndarray,
    max_components: int = 20,
    variance_threshold: float = 0.90
) -> Dict[str, float]:
    X = flatten_feature_maps(x)
    X = X - X.mean(axis=0, keepdims=True)

    if X.shape[0] < 2 or np.allclose(X, 0.0):
        return {
            "pca_top5_var": 0.0,
            "pca_top10_var": 0.0,
            "pca_top20_var": 0.0,
            "pca_num_for_90": 0.0,
            "pca_effective_rank": 0.0,
        }

    _, s, _ = np.linalg.svd(X, full_matrices=False)
    eigvals = (s ** 2) / max(X.shape[0] - 1, 1)
    total_var = eigvals.sum()

    if total_var <= 1e-12:
        return {
            "pca_top5_var": 0.0,
            "pca_top10_var": 0.0,
            "pca_top20_var": 0.0,
            "pca_num_for_90": 0.0,
            "pca_effective_rank": 0.0,
        }

    explained = eigvals / total_var
    cumsum = np.cumsum(explained)

    def topk_var(k: int) -> float:
        k = min(k, len(explained))
        return float(explained[:k].sum())

    num_for_threshold = int(np.searchsorted(cumsum, variance_threshold) + 1)

    eps = 1e-12
    p = explained[explained > eps]
    entropy = -np.sum(p * np.log(p))
    effective_rank = float(np.exp(entropy))

    return {
        "pca_top5_var": topk_var(5),
        "pca_top10_var": topk_var(10),
        "pca_top20_var": topk_var(max_components),
        "pca_num_for_90": float(num_for_threshold),
        "pca_effective_rank": effective_rank,
    }

# ============================================================
# HYBRID QUANTUM FEATURE EXTRACTOR
# ============================================================
_QNODE_CACHE: Dict[Tuple[int, str, int], qml.QNode] = {}

def apply_progressive_entanglement(n_qubits: int, entanglement_layers: int):
    """
    Progressively add CZ pairs.

    For n_qubits = 4:
        0 -> []
        1 -> [(0,1)]
        2 -> [(0,1), (1,2)]
        3 -> [(0,1), (1,2), (2,3)]
        4 -> [(0,1), (1,2), (2,3), (3,0)]

    For general n_qubits:
        max progressive pairs = n_qubits
    """
    if n_qubits <= 1 or entanglement_layers <= 0:
        return

    pairs = [(i, i + 1) for i in range(n_qubits - 1)] + [(n_qubits - 1, 0)]

    if entanglement_layers > len(pairs):
        raise ValueError(
            f"entanglement_layers={entanglement_layers} exceeds max allowed "
            f"{len(pairs)} for n_qubits={n_qubits}"
        )

    for a, b in pairs[:entanglement_layers]:
        qml.CZ(wires=[a, b])

def get_hybrid_qnode(n_qubits: int, measurement_type: str, entanglement_layers: int):
    key = (n_qubits, measurement_type, entanglement_layers)
    if key in _QNODE_CACHE:
        return _QNODE_CACHE[key]

    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch")
    def qnode(state_vec, thetas):
        wires = list(range(n_qubits))

        qml.AmplitudeEmbedding(state_vec, wires=wires, normalize=True)

        for i in range(n_qubits):
            qml.RY(thetas[i], wires=i)

        apply_progressive_entanglement(n_qubits, entanglement_layers)

        if measurement_type == "Z":
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
        elif measurement_type == "X":
            return [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]
        elif measurement_type == "Y":
            return [qml.expval(qml.PauliY(i)) for i in range(n_qubits)]
        elif measurement_type == "ZZ":
            if n_qubits == 1:
                return [qml.expval(qml.PauliZ(0))]
            return [
                qml.expval(qml.PauliZ(i) @ qml.PauliZ((i + 1) % n_qubits))
                for i in range(n_qubits)
            ]
        elif measurement_type == "XXZZ":
            half = n_qubits // 2
            x_part = [qml.expval(qml.PauliX(i)) for i in range(half)]
            z_part = [qml.expval(qml.PauliZ(i)) for i in range(half, n_qubits)]
            return x_part + z_part
        elif measurement_type == "XZ":
            x_vals = [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]
            z_vals = [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
            return x_vals + z_vals
        else:
            raise ValueError(f"Unknown measurement_type: {measurement_type}")

    _QNODE_CACHE[key] = qnode
    return qnode

_HYBRID_THETA_BANK: Optional[np.ndarray] = None
_HYBRID_THETA_META: Optional[Tuple[int, int, int]] = None
# meta = (seed, num_circuits, n_qubits)

def get_hybrid_theta_bank(num_circuits: int, n_qubits: int, seed: int) -> np.ndarray:
    global _HYBRID_THETA_BANK, _HYBRID_THETA_META

    if _HYBRID_THETA_BANK is not None and _HYBRID_THETA_META == (seed, num_circuits, n_qubits):
        return _HYBRID_THETA_BANK

    rng = np.random.default_rng(seed)
    bank = rng.uniform(0.0, 2 * np.pi, size=(num_circuits, n_qubits)).astype(np.float32)

    _HYBRID_THETA_BANK = bank
    _HYBRID_THETA_META = (seed, num_circuits, n_qubits)
    return bank

def hybrid_connector(
    vector: np.ndarray,
    thetas: np.ndarray,
    measurement_type: str,
    entanglement_layers: int
) -> np.ndarray:
    vec = vector.astype(np.float32)

    n_qubits = int(math.ceil(np.log2(vec.shape[0])))
    target_len = 2 ** n_qubits

    if vec.shape[0] < target_len:
        vec = np.concatenate(
            [vec, np.zeros(target_len - vec.shape[0], dtype=np.float32)],
            axis=0
        )

    if thetas.shape[0] != n_qubits:
        raise ValueError(f"thetas has length {thetas.shape[0]} but n_qubits={n_qubits}.")

    qnode = get_hybrid_qnode(n_qubits, measurement_type, entanglement_layers)
    feats_t = qnode(torch.tensor(vec), torch.tensor(thetas))
    feats = np.asarray(feats_t, dtype=np.float32)

    if L2_NORMALIZE_HYBRID_FEATURES:
        norm = np.linalg.norm(feats)
        if norm > 1e-12:
            feats = feats / norm

    return feats

def hybrid_patch_features(
    image: np.ndarray,
    seed: int,
    patch_size: int,
    stride: int,
    measurement_type: str,
    entanglement_layers: int
) -> np.ndarray:
    img = np.squeeze(image).astype(np.float32)
    image_size = img.shape[0]

    validate_patch_config(image_size=image_size, patch_size=patch_size, stride=stride)

    num_deep = get_num_output_channels(patch_size)
    patch_len = patch_size * patch_size
    n_qubits, feature_len = get_features_per_circuit(patch_len, measurement_type)
    num_circuits = int(math.ceil(num_deep / feature_len))

    max_allowed_entanglement_layers = n_qubits
    if entanglement_layers > max_allowed_entanglement_layers:
        raise ValueError(
            f"entanglement_layers={entanglement_layers} exceeds max allowed "
            f"{max_allowed_entanglement_layers} for patch_size={patch_size} "
            f"(n_qubits={n_qubits})"
        )

    theta_bank = get_hybrid_theta_bank(
        num_circuits=num_circuits,
        n_qubits=n_qubits,
        seed=seed
    )

    out_h, out_w = output_shape_for_patching(image_size, patch_size, stride)
    out = np.zeros((out_h, out_w, num_deep), dtype=np.float32)

    out_i = 0
    for i in range(0, image_size - patch_size + 1, stride):
        out_j = 0
        for j in range(0, image_size - patch_size + 1, stride):
            sub = img[i:i + patch_size, j:j + patch_size].copy()

            if np.all(sub == 0):
                sub.flat[0] = 1.0

            flat = sub.flatten()
            pnorm = np.linalg.norm(flat)
            if pnorm < 1e-12:
                pnorm = 1.0
            flat = flat / pnorm

            feats = []
            for c in range(num_circuits):
                feats.append(
                    hybrid_connector(
                        flat,
                        theta_bank[c],
                        measurement_type=measurement_type,
                        entanglement_layers=entanglement_layers
                    )
                )

            all_feats = np.concatenate(feats, axis=0)
            out[out_i, out_j, :num_deep] = all_feats[:num_deep]
            out_j += 1
        out_i += 1

    return out

def hybrid_converter(
    data: np.ndarray,
    seed: int,
    measurement_type: str,
    entanglement_layers: int
) -> np.ndarray:
    out = []
    N = len(data)
    print(
        f"\n=== Hybrid preprocessing started | "
        f"measurement={measurement_type} | ent_layers={entanglement_layers} ==="
    )
    for idx, x in enumerate(data):
        print(f"Processing image {idx+1}/{N}", end="\r")
        out.append(
            hybrid_patch_features(
                x,
                seed=seed,
                patch_size=patch_size,
                stride=stride,
                measurement_type=measurement_type,
                entanglement_layers=entanglement_layers
            )
        )
    print(
        f"\n=== Hybrid preprocessing finished | "
        f"measurement={measurement_type} | ent_layers={entanglement_layers} ===\n"
    )
    return np.array(out, dtype=np.float32)

# ============================================================
# DATA LOADING WITH SHARED RAW SPLITS
# ============================================================
def load_fashion_mnist_shared_pool(seed: int, n_train: int, n_val: int, n_test: int):
    rng = random.Random(seed)
    tfm = T.Compose([T.ToTensor()])

    train_ds = torchvision.datasets.FashionMNIST(
        root="data",
        train=True,
        download=True,
        transform=tfm
    )
    test_ds = torchvision.datasets.FashionMNIST(
        root="data",
        train=False,
        download=True,
        transform=tfm
    )

    train_idx = rng.sample(range(len(train_ds)), n_train + n_val)
    test_idx = rng.sample(range(len(test_ds)), n_test)

    x_train = np.asarray(
        [train_ds[i][0].numpy().transpose(1, 2, 0) for i in train_idx[:n_train]],
        dtype=np.float32
    )
    y_train = np.asarray(
        [train_ds[i][1] for i in train_idx[:n_train]],
        dtype=np.int64
    )

    x_val = np.asarray(
        [train_ds[i][0].numpy().transpose(1, 2, 0) for i in train_idx[n_train:n_train + n_val]],
        dtype=np.float32
    )
    y_val = np.asarray(
        [train_ds[i][1] for i in train_idx[n_train:n_train + n_val]],
        dtype=np.int64
    )

    x_test = np.asarray(
        [test_ds[i][0].numpy().transpose(1, 2, 0) for i in test_idx],
        dtype=np.float32
    )
    y_test = np.asarray(
        [test_ds[i][1] for i in test_idx],
        dtype=np.int64
    )

    return x_train, y_train, x_val, y_val, x_test, y_test

# ============================================================
# CACHE HELPERS
# ============================================================
def raw_cache_path(seed: int) -> Path:
    return CACHE_DIR / f"raw_seed{seed}_tr{N_TRAIN}_val{N_VAL}_te{N_TEST}.npz"

def feature_cache_path(seed: int, exp_cfg: Dict) -> Path:
    return CACHE_DIR / f"hybrid_seed{seed}_{config_tag(exp_cfg)}.npz"

def save_npz(path: Path, **arrays):
    np.savez_compressed(path, **arrays)

def load_or_build_raw_pool(seed: int):
    path = raw_cache_path(seed)
    if path.exists():
        data = np.load(path)
        return (
            data["x_train"], data["y_train"],
            data["x_val"], data["y_val"],
            data["x_test"], data["y_test"],
        )

    x_train, y_train, x_val, y_val, x_test, y_test = load_fashion_mnist_shared_pool(
        seed=seed, n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST
    )
    save_npz(
        path,
        x_train=x_train, y_train=y_train,
        x_val=x_val, y_val=y_val,
        x_test=x_test, y_test=y_test
    )
    return x_train, y_train, x_val, y_val, x_test, y_test

def load_or_build_feature_cache(seed: int, exp_cfg: Dict):
    path = feature_cache_path(seed, exp_cfg)
    if path.exists():
        data = np.load(path, allow_pickle=True)
        return {
            "x_train": data["x_train"],
            "y_train": data["y_train"],
            "x_val": data["x_val"],
            "y_val": data["y_val"],
            "x_test": data["x_test"],
            "y_test": data["y_test"],
            "preprocess_time_sec": float(data["preprocess_time_sec"]),
            "feature_shape": tuple(data["feature_shape"]),
        }

    x_train, y_train, x_val, y_val, x_test, y_test = load_or_build_raw_pool(seed)

    start = time.perf_counter()
    fx_train = hybrid_converter(
        x_train,
        seed=seed,
        measurement_type=exp_cfg["measurement_type"],
        entanglement_layers=exp_cfg["entanglement_layers"]
    )
    fx_val = hybrid_converter(
        x_val,
        seed=seed,
        measurement_type=exp_cfg["measurement_type"],
        entanglement_layers=exp_cfg["entanglement_layers"]
    )
    fx_test = hybrid_converter(
        x_test,
        seed=seed,
        measurement_type=exp_cfg["measurement_type"],
        entanglement_layers=exp_cfg["entanglement_layers"]
    )
    preprocess_time = time.perf_counter() - start

    feature_shape = np.array(fx_train.shape[1:], dtype=np.int64)

    save_npz(
        path,
        x_train=fx_train, y_train=y_train,
        x_val=fx_val, y_val=y_val,
        x_test=fx_test, y_test=y_test,
        preprocess_time_sec=np.array(preprocess_time, dtype=np.float64),
        feature_shape=feature_shape,
    )

    return {
        "x_train": fx_train,
        "y_train": y_train,
        "x_val": fx_val,
        "y_val": y_val,
        "x_test": fx_test,
        "y_test": y_test,
        "preprocess_time_sec": preprocess_time,
        "feature_shape": tuple(fx_train.shape[1:]),
    }

# ============================================================
# SHARED MLP HEAD
# ============================================================
class HybridModel(nn.Module):
    def __init__(self, in_dim: int, num_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        return self.net(x)

def eval_loader(model, loader, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)
    return loss_sum / total, correct / total

def train_model(Xtr, Ytr, Xva, Yva, Xte, Yte):
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(Xva, Yva), batch_size=64, shuffle=False)
    test_loader = DataLoader(TensorDataset(Xte, Yte), batch_size=64, shuffle=False)

    in_dim = int(np.prod(Xtr.shape[1:]))
    model = HybridModel(in_dim=in_dim).to(device_torch)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_acc = -1.0
    best_state = None
    best_train_acc = 0.0
    best_train_loss = 0.0
    best_val_loss = 0.0

    train_start = time.perf_counter()

    for epoch in range(1, n_epochs + 1):
        model.train()
        total, correct, loss_sum = 0, 0, 0.0

        for xb, yb in train_loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)

        train_loss = loss_sum / total
        train_acc = correct / total
        val_loss, val_acc = eval_loader(model, val_loader, criterion)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_train_loss = train_loss
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:03d}/{n_epochs} | "
            f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.3f}"
        )

    training_time = time.perf_counter() - train_start

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_acc = eval_loader(model, test_loader, criterion)

    return {
        "best_train_loss": best_train_loss,
        "best_train_acc": best_train_acc,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "training_time_sec": training_time,
    }

# ============================================================
# EXPERIMENT RUNNERS
# ============================================================
def run_one_from_cache(seed: int, exp_cfg: Dict):
    set_seed(seed)

    cache = load_or_build_feature_cache(seed=seed, exp_cfg=exp_cfg)

    Xtr_np = cache["x_train"]
    Ytr_np = cache["y_train"]
    Xva_np = cache["x_val"]
    Yva_np = cache["y_val"]
    Xte_np = cache["x_test"]
    Yte_np = cache["y_test"]

    pca_metrics = compute_pca_metrics(Xtr_np, max_components=20, variance_threshold=0.90)

    Xtr = torch.tensor(Xtr_np, dtype=torch.float32)
    Ytr = torch.tensor(Ytr_np, dtype=torch.long)
    Xva = torch.tensor(Xva_np, dtype=torch.float32)
    Yva = torch.tensor(Yva_np, dtype=torch.long)
    Xte = torch.tensor(Xte_np, dtype=torch.float32)
    Yte = torch.tensor(Yte_np, dtype=torch.long)

    metrics = train_model(Xtr, Ytr, Xva, Yva, Xte, Yte)
    metrics.update(pca_metrics)

    metrics["preprocess_time_sec"] = cache["preprocess_time_sec"]
    metrics["feature_shape"] = cache["feature_shape"]
    metrics["measurement_type"] = exp_cfg["measurement_type"]
    metrics["entanglement_layers"] = exp_cfg["entanglement_layers"]

    image_size = 28
    patch_len = patch_size * patch_size
    n_qubits, features_per_circuit = get_features_per_circuit(patch_len, exp_cfg["measurement_type"])
    num_circuits = int(math.ceil(get_num_output_channels(patch_size) / features_per_circuit))
    out_h, out_w = output_shape_for_patching(image_size, patch_size, stride)

    metrics["patch_size"] = patch_size
    metrics["stride"] = stride
    metrics["n_qubits"] = n_qubits
    metrics["features_per_circuit"] = features_per_circuit
    metrics["num_circuits"] = num_circuits
    metrics["patches_per_image"] = out_h * out_w

    return metrics

def summarize_results(results, exp_cfg):
    print(f"\n===== SUMMARY | {exp_cfg['name']} =====")

    keys = [
        "best_train_acc",
        "best_val_acc",
        "test_acc",
        "best_train_loss",
        "best_val_loss",
        "test_loss",
        "preprocess_time_sec",
        "training_time_sec",
        "pca_top5_var",
        "pca_top10_var",
        "pca_top20_var",
        "pca_num_for_90",
        "pca_effective_rank",
    ]

    for k in keys:
        mu, sd = mean_std([r[k] for r in results])
        print(f"{k:20s}: {mu:.6f} ± {sd:.6f}")

    print(f"measurement_type     : {results[0]['measurement_type']}")
    print(f"entanglement_layers  : {results[0]['entanglement_layers']}")
    print(f"feature_shape        : {results[0]['feature_shape']}")
    print(f"patch_size           : {results[0]['patch_size']}")
    print(f"stride               : {results[0]['stride']}")
    print(f"n_qubits             : {results[0]['n_qubits']}")
    print(f"features_per_circuit : {results[0]['features_per_circuit']}")
    print(f"num_circuits         : {results[0]['num_circuits']}")
    print(f"patches_per_image    : {results[0]['patches_per_image']}")

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    print(f"Using device: {device_torch}")
    print(f"Seeds: {EXPERIMENT_SEEDS}")
    print(f"n_train: {N_TRAIN}")
    print(f"Patch size: {patch_size}")
    print(f"Stride: {stride}")
    print(f"Output channels: {num_output_channels}")
    print(f"Cache dir: {CACHE_DIR.resolve()}")

    validate_patch_config(28, patch_size, stride)

    out_h, out_w = output_shape_for_patching(28, patch_size, stride)
    patch_len = patch_size * patch_size
    print(f"\nFixed config: out=({out_h},{out_w}), patches/image={out_h*out_w}")

    print("\nProgressive entanglement experiments:")
    for exp_cfg in ENTANGLEMENT_EXPERIMENTS:
        n_qubits, features_per_circuit = get_features_per_circuit(
            patch_len, exp_cfg["measurement_type"]
        )
        num_circuits = int(math.ceil(get_num_output_channels(patch_size) / features_per_circuit))
        print(
            f"  {exp_cfg['name']}: "
            f"measurement={exp_cfg['measurement_type']}, "
            f"entanglement_layers={exp_cfg['entanglement_layers']}, "
            f"n_qubits={n_qubits}, "
            f"features/circuit={features_per_circuit}, "
            f"circuits/patch={num_circuits}"
        )

    for seed in EXPERIMENT_SEEDS:
        print(f"\nPreparing shared raw pool for seed={seed}")
        load_or_build_raw_pool(seed)

        for exp_cfg in ENTANGLEMENT_EXPERIMENTS:
            print(
                f"Preparing/loading hybrid cache for seed={seed}, "
                f"measurement={exp_cfg['measurement_type']}, "
                f"entanglement_layers={exp_cfg['entanglement_layers']}"
            )
            load_or_build_feature_cache(seed=seed, exp_cfg=exp_cfg)

    all_results = {}

    for exp_cfg in ENTANGLEMENT_EXPERIMENTS:
        exp_results = []

        for seed in EXPERIMENT_SEEDS:
            print(f"\n\n########## {exp_cfg['name']} | seed={seed} ##########")
            result = run_one_from_cache(seed=seed, exp_cfg=exp_cfg)
            exp_results.append(result)

            print("\n--- Run result ---")
            for k, v in result.items():
                print(f"{k}: {v}")

        all_results[exp_cfg["name"]] = exp_results
        summarize_results(exp_results, exp_cfg)

    print("\n\n================ FINAL AGGREGATED SUMMARY ================")
    for exp_name, runs in all_results.items():
        test_mu, test_sd = mean_std([r["test_acc"] for r in runs])
        val_mu, val_sd = mean_std([r["best_val_acc"] for r in runs])
        prep_mu, prep_sd = mean_std([r["preprocess_time_sec"] for r in runs])
        train_mu, train_sd = mean_std([r["training_time_sec"] for r in runs])
        pca_rank_mu, pca_rank_sd = mean_std([r["pca_effective_rank"] for r in runs])
        pca90_mu, pca90_sd = mean_std([r["pca_num_for_90"] for r in runs])

        meta = runs[0]
        print(
            f"{exp_name:8s} | "
            f"meas={meta['measurement_type']} | "
            f"ent_layers={meta['entanglement_layers']} | "
            f"val_acc={val_mu:.4f}±{val_sd:.4f} | "
            f"test_acc={test_mu:.4f}±{test_sd:.4f} | "
            f"pca_rank={pca_rank_mu:.2f}±{pca_rank_sd:.2f} | "
            f"pca90={pca90_mu:.2f}±{pca90_sd:.2f} | "
            f"prep_time={prep_mu:.2f}±{prep_sd:.2f}s | "
            f"train_time={train_mu:.2f}±{train_sd:.2f}s"
        )

Using device: cpu
Seeds: [246, 247, 248]
n_train: 500
Patch size: 4
Stride: 4
Output channels: 4
Cache dir: C:\Users\Asus\qml\coursework\feature_cache_hybrid_progressive_entanglement_experiments

Fixed config: out=(7,7), patches/image=49

Progressive entanglement experiments:
  0_ent: measurement=Z, entanglement_layers=0, n_qubits=4, features/circuit=4, circuits/patch=1
  1_ent: measurement=Z, entanglement_layers=1, n_qubits=4, features/circuit=4, circuits/patch=1
  2_ent: measurement=Z, entanglement_layers=2, n_qubits=4, features/circuit=4, circuits/patch=1
  3_ent: measurement=Z, entanglement_layers=3, n_qubits=4, features/circuit=4, circuits/patch=1
  4_ent: measurement=Z, entanglement_layers=4, n_qubits=4, features/circuit=4, circuits/patch=1

Preparing shared raw pool for seed=246
Preparing/loading hybrid cache for seed=246, measurement=Z, entanglement_layers=0

=== Hybrid preprocessing started | measurement=Z | ent_layers=0 ===
Processing image 500/500
=== Hybrid preprocessing fi

## D2 — Single-Entanglement Reference

Reference experiment with a fixed single-entanglement configuration for controlled comparison.


In [25]:
import math
import random
import time
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as T
import pennylane as qml

# ============================================================
# HYBRID SINGLE-PAIR ENTANGLEMENT EXPERIMENT
#
# Fixed circuit:
#   AmplitudeEmbedding
#   -> random RY layer
#   -> optional single CZ pair
#   -> Z measurement
#
# What changes:
#   either no entanglement, or exactly one entangling pair
#
# For 4 qubits:
#   0_ent   : no entanglement
#   ent_01  : CZ(0,1)
#   ent_12  : CZ(1,2)
#   ent_23  : CZ(2,3)
#   ent_30  : CZ(3,0)
#
# Everything else is fixed so you isolate the effect
# of WHICH qubit pair is entangled.
# ============================================================

# -----------------------------
# Global config
# -----------------------------
device_torch = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_SEED = 246
EXPERIMENT_SEEDS = [246, 247, 248]

N_TRAIN = 500
N_VAL = 100
N_TEST = 100

batch_size = 4
n_epochs = 100
lr = 1e-3
weight_decay = 1e-4

patch_size = 4
stride = 4

num_output_channels = 4
L2_NORMALIZE_HYBRID_FEATURES = False

CACHE_DIR = Path("feature_cache_hybrid_single_pair_entanglement_experiments")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Single-pair entanglement experiments
# -----------------------------
ENTANGLEMENT_EXPERIMENTS = [
    {"name": "0_ent", "measurement_type": "Z", "entangled_pair": None},
    {"name": "ent_01", "measurement_type": "Z", "entangled_pair": (0, 1)},
    {"name": "ent_12", "measurement_type": "Z", "entangled_pair": (1, 2)},
    {"name": "ent_23", "measurement_type": "Z", "entangled_pair": (2, 3)},
    {"name": "ent_30", "measurement_type": "Z", "entangled_pair": (3, 0)},
]

# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# -----------------------------
# Helpers
# -----------------------------
def get_num_output_channels(kernel_size: int) -> int:
    return kernel_size**2 if num_output_channels == -1 else num_output_channels

def get_features_per_circuit(patch_len: int, measurement_type: str) -> Tuple[int, int]:
    n_qubits = int(math.ceil(math.log2(patch_len)))

    if measurement_type in {"Z", "X", "Y", "ZZ", "XXZZ"}:
        feature_len = n_qubits
    elif measurement_type == "XZ":
        feature_len = 2 * n_qubits
    else:
        raise ValueError(f"Unknown measurement_type: {measurement_type}")

    return n_qubits, feature_len

def mean_std(values):
    arr = np.array(values, dtype=np.float64)
    return float(arr.mean()), float(arr.std(ddof=0))

def output_shape_for_patching(image_size: int, patch_size: int, stride: int) -> Tuple[int, int]:
    out_h = (image_size - patch_size) // stride + 1
    out_w = (image_size - patch_size) // stride + 1
    return out_h, out_w

def validate_patch_config(image_size: int, patch_size: int, stride: int):
    if (image_size - patch_size) % stride != 0:
        raise ValueError(
            f"Invalid config: image_size={image_size}, patch_size={patch_size}, stride={stride}"
        )

def pair_tag(pair: Optional[Tuple[int, int]]) -> str:
    if pair is None:
        return "none"
    return f"{pair[0]}{pair[1]}"

def config_tag(exp_cfg: Dict) -> str:
    return (
        f"{exp_cfg['name']}"
        f"_meas{exp_cfg['measurement_type']}"
        f"_pair{pair_tag(exp_cfg['entangled_pair'])}"
        f"_p{patch_size}_s{stride}"
        f"_c{get_num_output_channels(patch_size)}"
        f"_tr{N_TRAIN}_val{N_VAL}_te{N_TEST}"
    )

# -----------------------------
# PCA helpers
# -----------------------------
def flatten_feature_maps(x: np.ndarray) -> np.ndarray:
    return x.reshape(x.shape[0], -1).astype(np.float64)

def compute_pca_metrics(
    x: np.ndarray,
    max_components: int = 20,
    variance_threshold: float = 0.90
) -> Dict[str, float]:
    X = flatten_feature_maps(x)
    X = X - X.mean(axis=0, keepdims=True)

    if X.shape[0] < 2 or np.allclose(X, 0.0):
        return {
            "pca_top5_var": 0.0,
            "pca_top10_var": 0.0,
            "pca_top20_var": 0.0,
            "pca_num_for_90": 0.0,
            "pca_effective_rank": 0.0,
        }

    _, s, _ = np.linalg.svd(X, full_matrices=False)
    eigvals = (s ** 2) / max(X.shape[0] - 1, 1)
    total_var = eigvals.sum()

    if total_var <= 1e-12:
        return {
            "pca_top5_var": 0.0,
            "pca_top10_var": 0.0,
            "pca_top20_var": 0.0,
            "pca_num_for_90": 0.0,
            "pca_effective_rank": 0.0,
        }

    explained = eigvals / total_var
    cumsum = np.cumsum(explained)

    def topk_var(k: int) -> float:
        k = min(k, len(explained))
        return float(explained[:k].sum())

    num_for_threshold = int(np.searchsorted(cumsum, variance_threshold) + 1)

    eps = 1e-12
    p = explained[explained > eps]
    entropy = -np.sum(p * np.log(p))
    effective_rank = float(np.exp(entropy))

    return {
        "pca_top5_var": topk_var(5),
        "pca_top10_var": topk_var(10),
        "pca_top20_var": topk_var(max_components),
        "pca_num_for_90": float(num_for_threshold),
        "pca_effective_rank": effective_rank,
    }

# ============================================================
# HYBRID QUANTUM FEATURE EXTRACTOR
# ============================================================
_QNODE_CACHE: Dict[Tuple[int, str, Optional[Tuple[int, int]]], qml.QNode] = {}

def apply_single_pair_entanglement(
    n_qubits: int,
    entangled_pair: Optional[Tuple[int, int]]
):
    if entangled_pair is None:
        return

    a, b = entangled_pair

    if not (0 <= a < n_qubits and 0 <= b < n_qubits):
        raise ValueError(
            f"Invalid entangled_pair={entangled_pair} for n_qubits={n_qubits}"
        )
    if a == b:
        raise ValueError(f"Entangled pair must use two distinct qubits, got {entangled_pair}")

    qml.CZ(wires=[a, b])

def get_hybrid_qnode(
    n_qubits: int,
    measurement_type: str,
    entangled_pair: Optional[Tuple[int, int]]
):
    key = (n_qubits, measurement_type, entangled_pair)
    if key in _QNODE_CACHE:
        return _QNODE_CACHE[key]

    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch")
    def qnode(state_vec, thetas):
        wires = list(range(n_qubits))

        qml.AmplitudeEmbedding(state_vec, wires=wires, normalize=True)

        for i in range(n_qubits):
            qml.RY(thetas[i], wires=i)

        apply_single_pair_entanglement(n_qubits, entangled_pair)

        if measurement_type == "Z":
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
        elif measurement_type == "X":
            return [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]
        elif measurement_type == "Y":
            return [qml.expval(qml.PauliY(i)) for i in range(n_qubits)]
        elif measurement_type == "ZZ":
            if n_qubits == 1:
                return [qml.expval(qml.PauliZ(0))]
            return [
                qml.expval(qml.PauliZ(i) @ qml.PauliZ((i + 1) % n_qubits))
                for i in range(n_qubits)
            ]
        elif measurement_type == "XXZZ":
            half = n_qubits // 2
            x_part = [qml.expval(qml.PauliX(i)) for i in range(half)]
            z_part = [qml.expval(qml.PauliZ(i)) for i in range(half, n_qubits)]
            return x_part + z_part
        elif measurement_type == "XZ":
            x_vals = [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]
            z_vals = [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
            return x_vals + z_vals
        else:
            raise ValueError(f"Unknown measurement_type: {measurement_type}")

    _QNODE_CACHE[key] = qnode
    return qnode

_HYBRID_THETA_BANK: Optional[np.ndarray] = None
_HYBRID_THETA_META: Optional[Tuple[int, int, int]] = None
# meta = (seed, num_circuits, n_qubits)

def get_hybrid_theta_bank(num_circuits: int, n_qubits: int, seed: int) -> np.ndarray:
    global _HYBRID_THETA_BANK, _HYBRID_THETA_META

    if _HYBRID_THETA_BANK is not None and _HYBRID_THETA_META == (seed, num_circuits, n_qubits):
        return _HYBRID_THETA_BANK

    rng = np.random.default_rng(seed)
    bank = rng.uniform(0.0, 2 * np.pi, size=(num_circuits, n_qubits)).astype(np.float32)

    _HYBRID_THETA_BANK = bank
    _HYBRID_THETA_META = (seed, num_circuits, n_qubits)
    return bank

def hybrid_connector(
    vector: np.ndarray,
    thetas: np.ndarray,
    measurement_type: str,
    entangled_pair: Optional[Tuple[int, int]]
) -> np.ndarray:
    vec = vector.astype(np.float32)

    n_qubits = int(math.ceil(np.log2(vec.shape[0])))
    target_len = 2 ** n_qubits

    if vec.shape[0] < target_len:
        vec = np.concatenate(
            [vec, np.zeros(target_len - vec.shape[0], dtype=np.float32)],
            axis=0
        )

    if thetas.shape[0] != n_qubits:
        raise ValueError(f"thetas has length {thetas.shape[0]} but n_qubits={n_qubits}.")

    qnode = get_hybrid_qnode(n_qubits, measurement_type, entangled_pair)
    feats_t = qnode(torch.tensor(vec), torch.tensor(thetas))
    feats = np.asarray(feats_t, dtype=np.float32)

    if L2_NORMALIZE_HYBRID_FEATURES:
        norm = np.linalg.norm(feats)
        if norm > 1e-12:
            feats = feats / norm

    return feats

def hybrid_patch_features(
    image: np.ndarray,
    seed: int,
    patch_size: int,
    stride: int,
    measurement_type: str,
    entangled_pair: Optional[Tuple[int, int]]
) -> np.ndarray:
    img = np.squeeze(image).astype(np.float32)
    image_size = img.shape[0]

    validate_patch_config(image_size=image_size, patch_size=patch_size, stride=stride)

    num_deep = get_num_output_channels(patch_size)
    patch_len = patch_size * patch_size
    n_qubits, feature_len = get_features_per_circuit(patch_len, measurement_type)
    num_circuits = int(math.ceil(num_deep / feature_len))

    if entangled_pair is not None:
        a, b = entangled_pair
        if not (0 <= a < n_qubits and 0 <= b < n_qubits):
            raise ValueError(
                f"entangled_pair={entangled_pair} invalid for n_qubits={n_qubits}"
            )

    theta_bank = get_hybrid_theta_bank(
        num_circuits=num_circuits,
        n_qubits=n_qubits,
        seed=seed
    )

    out_h, out_w = output_shape_for_patching(image_size, patch_size, stride)
    out = np.zeros((out_h, out_w, num_deep), dtype=np.float32)

    out_i = 0
    for i in range(0, image_size - patch_size + 1, stride):
        out_j = 0
        for j in range(0, image_size - patch_size + 1, stride):
            sub = img[i:i + patch_size, j:j + patch_size].copy()

            if np.all(sub == 0):
                sub.flat[0] = 1.0

            flat = sub.flatten()
            pnorm = np.linalg.norm(flat)
            if pnorm < 1e-12:
                pnorm = 1.0
            flat = flat / pnorm

            feats = []
            for c in range(num_circuits):
                feats.append(
                    hybrid_connector(
                        flat,
                        theta_bank[c],
                        measurement_type=measurement_type,
                        entangled_pair=entangled_pair
                    )
                )

            all_feats = np.concatenate(feats, axis=0)
            out[out_i, out_j, :num_deep] = all_feats[:num_deep]
            out_j += 1
        out_i += 1

    return out

def hybrid_converter(
    data: np.ndarray,
    seed: int,
    measurement_type: str,
    entangled_pair: Optional[Tuple[int, int]]
) -> np.ndarray:
    out = []
    N = len(data)
    print(
        f"\n=== Hybrid preprocessing started | "
        f"measurement={measurement_type} | pair={entangled_pair} ==="
    )
    for idx, x in enumerate(data):
        print(f"Processing image {idx+1}/{N}", end="\r")
        out.append(
            hybrid_patch_features(
                x,
                seed=seed,
                patch_size=patch_size,
                stride=stride,
                measurement_type=measurement_type,
                entangled_pair=entangled_pair
            )
        )
    print(
        f"\n=== Hybrid preprocessing finished | "
        f"measurement={measurement_type} | pair={entangled_pair} ===\n"
    )
    return np.array(out, dtype=np.float32)

# ============================================================
# DATA LOADING WITH SHARED RAW SPLITS
# ============================================================
def load_fashion_mnist_shared_pool(seed: int, n_train: int, n_val: int, n_test: int):
    rng = random.Random(seed)
    tfm = T.Compose([T.ToTensor()])

    train_ds = torchvision.datasets.FashionMNIST(
        root="data",
        train=True,
        download=True,
        transform=tfm
    )
    test_ds = torchvision.datasets.FashionMNIST(
        root="data",
        train=False,
        download=True,
        transform=tfm
    )

    train_idx = rng.sample(range(len(train_ds)), n_train + n_val)
    test_idx = rng.sample(range(len(test_ds)), n_test)

    x_train = np.asarray(
        [train_ds[i][0].numpy().transpose(1, 2, 0) for i in train_idx[:n_train]],
        dtype=np.float32
    )
    y_train = np.asarray(
        [train_ds[i][1] for i in train_idx[:n_train]],
        dtype=np.int64
    )

    x_val = np.asarray(
        [train_ds[i][0].numpy().transpose(1, 2, 0) for i in train_idx[n_train:n_train + n_val]],
        dtype=np.float32
    )
    y_val = np.asarray(
        [train_ds[i][1] for i in train_idx[n_train:n_train + n_val]],
        dtype=np.int64
    )

    x_test = np.asarray(
        [test_ds[i][0].numpy().transpose(1, 2, 0) for i in test_idx],
        dtype=np.float32
    )
    y_test = np.asarray(
        [test_ds[i][1] for i in test_idx],
        dtype=np.int64
    )

    return x_train, y_train, x_val, y_val, x_test, y_test

# ============================================================
# CACHE HELPERS
# ============================================================
def raw_cache_path(seed: int) -> Path:
    return CACHE_DIR / f"raw_seed{seed}_tr{N_TRAIN}_val{N_VAL}_te{N_TEST}.npz"

def feature_cache_path(seed: int, exp_cfg: Dict) -> Path:
    return CACHE_DIR / f"hybrid_seed{seed}_{config_tag(exp_cfg)}.npz"

def save_npz(path: Path, **arrays):
    np.savez_compressed(path, **arrays)

def load_or_build_raw_pool(seed: int):
    path = raw_cache_path(seed)
    if path.exists():
        data = np.load(path)
        return (
            data["x_train"], data["y_train"],
            data["x_val"], data["y_val"],
            data["x_test"], data["y_test"],
        )

    x_train, y_train, x_val, y_val, x_test, y_test = load_fashion_mnist_shared_pool(
        seed=seed, n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST
    )
    save_npz(
        path,
        x_train=x_train, y_train=y_train,
        x_val=x_val, y_val=y_val,
        x_test=x_test, y_test=y_test
    )
    return x_train, y_train, x_val, y_val, x_test, y_test

def load_or_build_feature_cache(seed: int, exp_cfg: Dict):
    path = feature_cache_path(seed, exp_cfg)
    if path.exists():
        data = np.load(path, allow_pickle=True)
        return {
            "x_train": data["x_train"],
            "y_train": data["y_train"],
            "x_val": data["x_val"],
            "y_val": data["y_val"],
            "x_test": data["x_test"],
            "y_test": data["y_test"],
            "preprocess_time_sec": float(data["preprocess_time_sec"]),
            "feature_shape": tuple(data["feature_shape"]),
        }

    x_train, y_train, x_val, y_val, x_test, y_test = load_or_build_raw_pool(seed)

    start = time.perf_counter()
    fx_train = hybrid_converter(
        x_train,
        seed=seed,
        measurement_type=exp_cfg["measurement_type"],
        entangled_pair=exp_cfg["entangled_pair"]
    )
    fx_val = hybrid_converter(
        x_val,
        seed=seed,
        measurement_type=exp_cfg["measurement_type"],
        entangled_pair=exp_cfg["entangled_pair"]
    )
    fx_test = hybrid_converter(
        x_test,
        seed=seed,
        measurement_type=exp_cfg["measurement_type"],
        entangled_pair=exp_cfg["entangled_pair"]
    )
    preprocess_time = time.perf_counter() - start

    feature_shape = np.array(fx_train.shape[1:], dtype=np.int64)

    save_npz(
        path,
        x_train=fx_train, y_train=y_train,
        x_val=fx_val, y_val=y_val,
        x_test=fx_test, y_test=y_test,
        preprocess_time_sec=np.array(preprocess_time, dtype=np.float64),
        feature_shape=feature_shape,
    )

    return {
        "x_train": fx_train,
        "y_train": y_train,
        "x_val": fx_val,
        "y_val": y_val,
        "x_test": fx_test,
        "y_test": y_test,
        "preprocess_time_sec": preprocess_time,
        "feature_shape": tuple(fx_train.shape[1:]),
    }

# ============================================================
# SHARED MLP HEAD
# ============================================================
class HybridModel(nn.Module):
    def __init__(self, in_dim: int, num_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        return self.net(x)

def eval_loader(model, loader, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)
    return loss_sum / total, correct / total

def train_model(Xtr, Ytr, Xva, Yva, Xte, Yte):
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(Xva, Yva), batch_size=64, shuffle=False)
    test_loader = DataLoader(TensorDataset(Xte, Yte), batch_size=64, shuffle=False)

    in_dim = int(np.prod(Xtr.shape[1:]))
    model = HybridModel(in_dim=in_dim).to(device_torch)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_acc = -1.0
    best_state = None
    best_train_acc = 0.0
    best_train_loss = 0.0
    best_val_loss = 0.0

    train_start = time.perf_counter()

    for epoch in range(1, n_epochs + 1):
        model.train()
        total, correct, loss_sum = 0, 0, 0.0

        for xb, yb in train_loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)

        train_loss = loss_sum / total
        train_acc = correct / total
        val_loss, val_acc = eval_loader(model, val_loader, criterion)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_train_loss = train_loss
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:03d}/{n_epochs} | "
            f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.3f}"
        )

    training_time = time.perf_counter() - train_start

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_acc = eval_loader(model, test_loader, criterion)

    return {
        "best_train_loss": best_train_loss,
        "best_train_acc": best_train_acc,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "training_time_sec": training_time,
    }

# ============================================================
# EXPERIMENT RUNNERS
# ============================================================
def run_one_from_cache(seed: int, exp_cfg: Dict):
    set_seed(seed)

    cache = load_or_build_feature_cache(seed=seed, exp_cfg=exp_cfg)

    Xtr_np = cache["x_train"]
    Ytr_np = cache["y_train"]
    Xva_np = cache["x_val"]
    Yva_np = cache["y_val"]
    Xte_np = cache["x_test"]
    Yte_np = cache["y_test"]

    pca_metrics = compute_pca_metrics(Xtr_np, max_components=20, variance_threshold=0.90)

    Xtr = torch.tensor(Xtr_np, dtype=torch.float32)
    Ytr = torch.tensor(Ytr_np, dtype=torch.long)
    Xva = torch.tensor(Xva_np, dtype=torch.float32)
    Yva = torch.tensor(Yva_np, dtype=torch.long)
    Xte = torch.tensor(Xte_np, dtype=torch.float32)
    Yte = torch.tensor(Yte_np, dtype=torch.long)

    metrics = train_model(Xtr, Ytr, Xva, Yva, Xte, Yte)
    metrics.update(pca_metrics)

    metrics["preprocess_time_sec"] = cache["preprocess_time_sec"]
    metrics["feature_shape"] = cache["feature_shape"]
    metrics["measurement_type"] = exp_cfg["measurement_type"]
    metrics["entangled_pair"] = exp_cfg["entangled_pair"]

    image_size = 28
    patch_len = patch_size * patch_size
    n_qubits, features_per_circuit = get_features_per_circuit(patch_len, exp_cfg["measurement_type"])
    num_circuits = int(math.ceil(get_num_output_channels(patch_size) / features_per_circuit))
    out_h, out_w = output_shape_for_patching(image_size, patch_size, stride)

    metrics["patch_size"] = patch_size
    metrics["stride"] = stride
    metrics["n_qubits"] = n_qubits
    metrics["features_per_circuit"] = features_per_circuit
    metrics["num_circuits"] = num_circuits
    metrics["patches_per_image"] = out_h * out_w

    return metrics

def summarize_results(results, exp_cfg):
    print(f"\n===== SUMMARY | {exp_cfg['name']} =====")

    keys = [
        "best_train_acc",
        "best_val_acc",
        "test_acc",
        "best_train_loss",
        "best_val_loss",
        "test_loss",
        "preprocess_time_sec",
        "training_time_sec",
        "pca_top5_var",
        "pca_top10_var",
        "pca_top20_var",
        "pca_num_for_90",
        "pca_effective_rank",
    ]

    for k in keys:
        mu, sd = mean_std([r[k] for r in results])
        print(f"{k:20s}: {mu:.6f} ± {sd:.6f}")

    print(f"measurement_type     : {results[0]['measurement_type']}")
    print(f"entangled_pair       : {results[0]['entangled_pair']}")
    print(f"feature_shape        : {results[0]['feature_shape']}")
    print(f"patch_size           : {results[0]['patch_size']}")
    print(f"stride               : {results[0]['stride']}")
    print(f"n_qubits             : {results[0]['n_qubits']}")
    print(f"features_per_circuit : {results[0]['features_per_circuit']}")
    print(f"num_circuits         : {results[0]['num_circuits']}")
    print(f"patches_per_image    : {results[0]['patches_per_image']}")

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    print(f"Using device: {device_torch}")
    print(f"Seeds: {EXPERIMENT_SEEDS}")
    print(f"n_train: {N_TRAIN}")
    print(f"Patch size: {patch_size}")
    print(f"Stride: {stride}")
    print(f"Output channels: {num_output_channels}")
    print(f"Cache dir: {CACHE_DIR.resolve()}")

    validate_patch_config(28, patch_size, stride)

    out_h, out_w = output_shape_for_patching(28, patch_size, stride)
    patch_len = patch_size * patch_size
    print(f"\nFixed config: out=({out_h},{out_w}), patches/image={out_h*out_w}")

    print("\nSingle-pair entanglement experiments:")
    for exp_cfg in ENTANGLEMENT_EXPERIMENTS:
        n_qubits, features_per_circuit = get_features_per_circuit(
            patch_len, exp_cfg["measurement_type"]
        )
        num_circuits = int(math.ceil(get_num_output_channels(patch_size) / features_per_circuit))
        print(
            f"  {exp_cfg['name']}: "
            f"measurement={exp_cfg['measurement_type']}, "
            f"entangled_pair={exp_cfg['entangled_pair']}, "
            f"n_qubits={n_qubits}, "
            f"features/circuit={features_per_circuit}, "
            f"circuits/patch={num_circuits}"
        )

    for seed in EXPERIMENT_SEEDS:
        print(f"\nPreparing shared raw pool for seed={seed}")
        load_or_build_raw_pool(seed)

        for exp_cfg in ENTANGLEMENT_EXPERIMENTS:
            print(
                f"Preparing/loading hybrid cache for seed={seed}, "
                f"measurement={exp_cfg['measurement_type']}, "
                f"entangled_pair={exp_cfg['entangled_pair']}"
            )
            load_or_build_feature_cache(seed=seed, exp_cfg=exp_cfg)

    all_results = {}

    for exp_cfg in ENTANGLEMENT_EXPERIMENTS:
        exp_results = []

        for seed in EXPERIMENT_SEEDS:
            print(f"\n\n########## {exp_cfg['name']} | seed={seed} ##########")
            result = run_one_from_cache(seed=seed, exp_cfg=exp_cfg)
            exp_results.append(result)

            print("\n--- Run result ---")
            for k, v in result.items():
                print(f"{k}: {v}")

        all_results[exp_cfg["name"]] = exp_results
        summarize_results(exp_results, exp_cfg)

    print("\n\n================ FINAL AGGREGATED SUMMARY ================")
    for exp_name, runs in all_results.items():
        test_mu, test_sd = mean_std([r["test_acc"] for r in runs])
        val_mu, val_sd = mean_std([r["best_val_acc"] for r in runs])
        prep_mu, prep_sd = mean_std([r["preprocess_time_sec"] for r in runs])
        train_mu, train_sd = mean_std([r["training_time_sec"] for r in runs])
        pca_rank_mu, pca_rank_sd = mean_std([r["pca_effective_rank"] for r in runs])
        pca90_mu, pca90_sd = mean_std([r["pca_num_for_90"] for r in runs])

        meta = runs[0]
        print(
            f"{exp_name:8s} | "
            f"meas={meta['measurement_type']} | "
            f"pair={meta['entangled_pair']} | "
            f"val_acc={val_mu:.4f}±{val_sd:.4f} | "
            f"test_acc={test_mu:.4f}±{test_sd:.4f} | "
            f"pca_rank={pca_rank_mu:.2f}±{pca_rank_sd:.2f} | "
            f"pca90={pca90_mu:.2f}±{pca90_sd:.2f} | "
            f"prep_time={prep_mu:.2f}±{prep_sd:.2f}s | "
            f"train_time={train_mu:.2f}±{train_sd:.2f}s"
        )

Using device: cpu
Seeds: [246, 247, 248]
n_train: 500
Patch size: 4
Stride: 4
Output channels: 4
Cache dir: C:\Users\Asus\qml\coursework\feature_cache_hybrid_single_pair_entanglement_experiments

Fixed config: out=(7,7), patches/image=49

Single-pair entanglement experiments:
  0_ent: measurement=Z, entangled_pair=None, n_qubits=4, features/circuit=4, circuits/patch=1
  ent_01: measurement=Z, entangled_pair=(0, 1), n_qubits=4, features/circuit=4, circuits/patch=1
  ent_12: measurement=Z, entangled_pair=(1, 2), n_qubits=4, features/circuit=4, circuits/patch=1
  ent_23: measurement=Z, entangled_pair=(2, 3), n_qubits=4, features/circuit=4, circuits/patch=1
  ent_30: measurement=Z, entangled_pair=(3, 0), n_qubits=4, features/circuit=4, circuits/patch=1

Preparing shared raw pool for seed=246
Preparing/loading hybrid cache for seed=246, measurement=Z, entangled_pair=None

=== Hybrid preprocessing started | measurement=Z | pair=None ===
Processing image 500/500
=== Hybrid preprocessing finis

# Sanity Check / Test Cell

Quick verification run used to validate pipeline integrity after configuration or code changes.


In [27]:
import math
import random
import time
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as T
import pennylane as qml

# ============================================================
# HYBRID ENTANGLEMENT EXPERIMENT
#
# New fixed circuit:
#   AmplitudeEmbedding
#   -> random RY layer
#   -> entangling-X pattern (CNOT-based)
#   -> second random RY layer
#   -> Z measurement
#
# Entanglement experiments:
#   NONE  : no entanglement
#   CHAIN : CNOT(0,1), CNOT(1,2), ..., CNOT(n-2,n-1)
#   RING  : CHAIN + CNOT(n-1,0)
#   FULL  : CNOT on all ordered nearest-free pairs i<j
#
# Same number of qubits as before.
# ============================================================

# -----------------------------
# Global config
# -----------------------------
device_torch = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_SEED = 246
EXPERIMENT_SEEDS = [246, 247, 248]

N_TRAIN = 500
N_VAL = 100
N_TEST = 100

batch_size = 4
n_epochs = 100
lr = 1e-3
weight_decay = 1e-4

# Fixed patch setup
patch_size = 4
stride = 4

num_output_channels = 4
L2_NORMALIZE_HYBRID_FEATURES = False

CACHE_DIR = Path("feature_cache_hybrid_entanglement_experiments_ryxry")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Entanglement experiments
# -----------------------------
ENTANGLEMENT_EXPERIMENTS = [
    {"name": "ent_none", "measurement_type": "Z", "entanglement": "NONE"},
    {"name": "ent_chain", "measurement_type": "Z", "entanglement": "CHAIN"},
    {"name": "ent_ring", "measurement_type": "Z", "entanglement": "RING"},
    {"name": "ent_full", "measurement_type": "Z", "entanglement": "FULL"},
]

# -----------------------------
# Reproducibility
# -----------------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# -----------------------------
# Helpers
# -----------------------------
def get_num_output_channels(kernel_size: int) -> int:
    return kernel_size**2 if num_output_channels == -1 else num_output_channels

def get_features_per_circuit(patch_len: int, measurement_type: str) -> Tuple[int, int]:
    n_qubits = int(math.ceil(math.log2(patch_len)))

    if measurement_type in {"Z", "X", "Y", "ZZ", "XXZZ"}:
        feature_len = n_qubits
    elif measurement_type == "XZ":
        feature_len = 2 * n_qubits
    else:
        raise ValueError(f"Unknown measurement_type: {measurement_type}")

    return n_qubits, feature_len

def mean_std(values):
    arr = np.array(values, dtype=np.float64)
    return float(arr.mean()), float(arr.std(ddof=0))

def output_shape_for_patching(image_size: int, patch_size: int, stride: int) -> Tuple[int, int]:
    out_h = (image_size - patch_size) // stride + 1
    out_w = (image_size - patch_size) // stride + 1
    return out_h, out_w

def validate_patch_config(image_size: int, patch_size: int, stride: int):
    if (image_size - patch_size) % stride != 0:
        raise ValueError(
            f"Invalid config: image_size={image_size}, patch_size={patch_size}, stride={stride}"
        )

def config_tag(exp_cfg: Dict) -> str:
    return (
        f"{exp_cfg['name']}"
        f"_meas{exp_cfg['measurement_type']}"
        f"_ent{exp_cfg['entanglement']}"
        f"_ansatzRYXRY"
        f"_p{patch_size}_s{stride}"
        f"_c{get_num_output_channels(patch_size)}"
        f"_tr{N_TRAIN}_val{N_VAL}_te{N_TEST}"
    )

# -----------------------------
# PCA helpers
# -----------------------------
def flatten_feature_maps(x: np.ndarray) -> np.ndarray:
    return x.reshape(x.shape[0], -1).astype(np.float64)

def compute_pca_metrics(
    x: np.ndarray,
    max_components: int = 20,
    variance_threshold: float = 0.90
) -> Dict[str, float]:
    X = flatten_feature_maps(x)
    X = X - X.mean(axis=0, keepdims=True)

    if X.shape[0] < 2 or np.allclose(X, 0.0):
        return {
            "pca_top5_var": 0.0,
            "pca_top10_var": 0.0,
            "pca_top20_var": 0.0,
            "pca_num_for_90": 0.0,
            "pca_effective_rank": 0.0,
        }

    _, s, _ = np.linalg.svd(X, full_matrices=False)
    eigvals = (s ** 2) / max(X.shape[0] - 1, 1)
    total_var = eigvals.sum()

    if total_var <= 1e-12:
        return {
            "pca_top5_var": 0.0,
            "pca_top10_var": 0.0,
            "pca_top20_var": 0.0,
            "pca_num_for_90": 0.0,
            "pca_effective_rank": 0.0,
        }

    explained = eigvals / total_var
    cumsum = np.cumsum(explained)

    def topk_var(k: int) -> float:
        k = min(k, len(explained))
        return float(explained[:k].sum())

    num_for_threshold = int(np.searchsorted(cumsum, variance_threshold) + 1)

    eps = 1e-12
    p = explained[explained > eps]
    entropy = -np.sum(p * np.log(p))
    effective_rank = float(np.exp(entropy))

    return {
        "pca_top5_var": topk_var(5),
        "pca_top10_var": topk_var(10),
        "pca_top20_var": topk_var(max_components),
        "pca_num_for_90": float(num_for_threshold),
        "pca_effective_rank": effective_rank,
    }

# ============================================================
# HYBRID QUANTUM FEATURE EXTRACTOR
# ============================================================
_QNODE_CACHE: Dict[Tuple[int, str, str], qml.QNode] = {}

def apply_entanglement_x(n_qubits: int, entanglement: str):
    if n_qubits <= 1 or entanglement == "NONE":
        return

    if entanglement == "CHAIN":
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])

    elif entanglement == "RING":
        for i in range(n_qubits):
            qml.CNOT(wires=[i, (i + 1) % n_qubits])

    elif entanglement == "FULL":
        for i in range(n_qubits):
            for j in range(i + 1, n_qubits):
                qml.CNOT(wires=[i, j])

    else:
        raise ValueError(f"Unknown entanglement: {entanglement}")

def get_hybrid_qnode(n_qubits: int, measurement_type: str, entanglement: str):
    key = (n_qubits, measurement_type, entanglement)
    if key in _QNODE_CACHE:
        return _QNODE_CACHE[key]

    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch")
    def qnode(state_vec, theta_1, theta_2):
        wires = list(range(n_qubits))

        qml.AmplitudeEmbedding(state_vec, wires=wires, normalize=True)

        # First RY layer
        for i in range(n_qubits):
            qml.RY(theta_1[i], wires=i)

        # Entangling-X block
        apply_entanglement_x(n_qubits, entanglement)

        # Second RY layer
        for i in range(n_qubits):
            qml.RY(theta_2[i], wires=i)

        if measurement_type == "Z":
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
        elif measurement_type == "X":
            return [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]
        elif measurement_type == "Y":
            return [qml.expval(qml.PauliY(i)) for i in range(n_qubits)]
        elif measurement_type == "ZZ":
            if n_qubits == 1:
                return [qml.expval(qml.PauliZ(0))]
            return [
                qml.expval(qml.PauliZ(i) @ qml.PauliZ((i + 1) % n_qubits))
                for i in range(n_qubits)
            ]
        elif measurement_type == "XXZZ":
            half = n_qubits // 2
            x_part = [qml.expval(qml.PauliX(i)) for i in range(half)]
            z_part = [qml.expval(qml.PauliZ(i)) for i in range(half, n_qubits)]
            return x_part + z_part
        elif measurement_type == "XZ":
            x_vals = [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]
            z_vals = [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
            return x_vals + z_vals
        else:
            raise ValueError(f"Unknown measurement_type: {measurement_type}")

    _QNODE_CACHE[key] = qnode
    return qnode

_HYBRID_THETA_BANK: Optional[np.ndarray] = None
_HYBRID_THETA_META: Optional[Tuple[int, int, int]] = None
# meta = (seed, num_circuits, n_qubits)

def get_hybrid_theta_bank(num_circuits: int, n_qubits: int, seed: int) -> np.ndarray:
    """
    Returns theta bank of shape:
        (num_circuits, 2, n_qubits)

    Two RY layers per circuit.
    """
    global _HYBRID_THETA_BANK, _HYBRID_THETA_META

    if _HYBRID_THETA_BANK is not None and _HYBRID_THETA_META == (seed, num_circuits, n_qubits):
        return _HYBRID_THETA_BANK

    rng = np.random.default_rng(seed)
    bank = rng.uniform(
        0.0, 2 * np.pi, size=(num_circuits, 2, n_qubits)
    ).astype(np.float32)

    _HYBRID_THETA_BANK = bank
    _HYBRID_THETA_META = (seed, num_circuits, n_qubits)
    return bank

def hybrid_connector(
    vector: np.ndarray,
    thetas: np.ndarray,
    measurement_type: str,
    entanglement: str
) -> np.ndarray:
    vec = vector.astype(np.float32)

    n_qubits = int(math.ceil(np.log2(vec.shape[0])))
    target_len = 2 ** n_qubits

    if vec.shape[0] < target_len:
        vec = np.concatenate(
            [vec, np.zeros(target_len - vec.shape[0], dtype=np.float32)],
            axis=0
        )

    if thetas.shape != (2, n_qubits):
        raise ValueError(
            f"Expected thetas shape (2, {n_qubits}), got {thetas.shape}"
        )

    qnode = get_hybrid_qnode(n_qubits, measurement_type, entanglement)
    feats_t = qnode(
        torch.tensor(vec),
        torch.tensor(thetas[0]),
        torch.tensor(thetas[1]),
    )
    feats = np.asarray(feats_t, dtype=np.float32)

    if L2_NORMALIZE_HYBRID_FEATURES:
        norm = np.linalg.norm(feats)
        if norm > 1e-12:
            feats = feats / norm

    return feats

def hybrid_patch_features(
    image: np.ndarray,
    seed: int,
    patch_size: int,
    stride: int,
    measurement_type: str,
    entanglement: str
) -> np.ndarray:
    img = np.squeeze(image).astype(np.float32)
    image_size = img.shape[0]

    validate_patch_config(image_size=image_size, patch_size=patch_size, stride=stride)

    num_deep = get_num_output_channels(patch_size)
    patch_len = patch_size * patch_size
    n_qubits, feature_len = get_features_per_circuit(patch_len, measurement_type)
    num_circuits = int(math.ceil(num_deep / feature_len))

    theta_bank = get_hybrid_theta_bank(
        num_circuits=num_circuits,
        n_qubits=n_qubits,
        seed=seed
    )

    out_h, out_w = output_shape_for_patching(image_size, patch_size, stride)
    out = np.zeros((out_h, out_w, num_deep), dtype=np.float32)

    out_i = 0
    for i in range(0, image_size - patch_size + 1, stride):
        out_j = 0
        for j in range(0, image_size - patch_size + 1, stride):
            sub = img[i:i + patch_size, j:j + patch_size].copy()

            if np.all(sub == 0):
                sub.flat[0] = 1.0

            flat = sub.flatten()
            pnorm = np.linalg.norm(flat)
            if pnorm < 1e-12:
                pnorm = 1.0
            flat = flat / pnorm

            feats = []
            for c in range(num_circuits):
                feats.append(
                    hybrid_connector(
                        flat,
                        theta_bank[c],
                        measurement_type=measurement_type,
                        entanglement=entanglement
                    )
                )

            all_feats = np.concatenate(feats, axis=0)
            out[out_i, out_j, :num_deep] = all_feats[:num_deep]
            out_j += 1
        out_i += 1

    return out

def hybrid_converter(data: np.ndarray, seed: int, measurement_type: str, entanglement: str) -> np.ndarray:
    out = []
    N = len(data)
    print(
        f"\n=== Hybrid preprocessing started | "
        f"measurement={measurement_type} | entanglement={entanglement} ==="
    )
    for idx, x in enumerate(data):
        print(f"Processing image {idx+1}/{N}", end="\r")
        out.append(
            hybrid_patch_features(
                x,
                seed=seed,
                patch_size=patch_size,
                stride=stride,
                measurement_type=measurement_type,
                entanglement=entanglement
            )
        )
    print(
        f"\n=== Hybrid preprocessing finished | "
        f"measurement={measurement_type} | entanglement={entanglement} ===\n"
    )
    return np.array(out, dtype=np.float32)

# ============================================================
# DATA LOADING WITH SHARED RAW SPLITS
# ============================================================
def load_fashion_mnist_shared_pool(seed: int, n_train: int, n_val: int, n_test: int):
    rng = random.Random(seed)
    tfm = T.Compose([T.ToTensor()])

    train_ds = torchvision.datasets.FashionMNIST(
        root="data",
        train=True,
        download=True,
        transform=tfm
    )
    test_ds = torchvision.datasets.FashionMNIST(
        root="data",
        train=False,
        download=True,
        transform=tfm
    )

    train_idx = rng.sample(range(len(train_ds)), n_train + n_val)
    test_idx = rng.sample(range(len(test_ds)), n_test)

    x_train = np.asarray(
        [train_ds[i][0].numpy().transpose(1, 2, 0) for i in train_idx[:n_train]],
        dtype=np.float32
    )
    y_train = np.asarray(
        [train_ds[i][1] for i in train_idx[:n_train]],
        dtype=np.int64
    )

    x_val = np.asarray(
        [train_ds[i][0].numpy().transpose(1, 2, 0) for i in train_idx[n_train:n_train + n_val]],
        dtype=np.float32
    )
    y_val = np.asarray(
        [train_ds[i][1] for i in train_idx[n_train:n_train + n_val]],
        dtype=np.int64
    )

    x_test = np.asarray(
        [test_ds[i][0].numpy().transpose(1, 2, 0) for i in test_idx],
        dtype=np.float32
    )
    y_test = np.asarray(
        [test_ds[i][1] for i in test_idx],
        dtype=np.int64
    )

    return x_train, y_train, x_val, y_val, x_test, y_test

# ============================================================
# CACHE HELPERS
# ============================================================
def raw_cache_path(seed: int) -> Path:
    return CACHE_DIR / f"raw_seed{seed}_tr{N_TRAIN}_val{N_VAL}_te{N_TEST}.npz"

def feature_cache_path(seed: int, exp_cfg: Dict) -> Path:
    return CACHE_DIR / f"hybrid_seed{seed}_{config_tag(exp_cfg)}.npz"

def save_npz(path: Path, **arrays):
    np.savez_compressed(path, **arrays)

def load_or_build_raw_pool(seed: int):
    path = raw_cache_path(seed)
    if path.exists():
        data = np.load(path)
        return (
            data["x_train"], data["y_train"],
            data["x_val"], data["y_val"],
            data["x_test"], data["y_test"],
        )

    x_train, y_train, x_val, y_val, x_test, y_test = load_fashion_mnist_shared_pool(
        seed=seed, n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST
    )
    save_npz(
        path,
        x_train=x_train, y_train=y_train,
        x_val=x_val, y_val=y_val,
        x_test=x_test, y_test=y_test
    )
    return x_train, y_train, x_val, y_val, x_test, y_test

def load_or_build_feature_cache(seed: int, exp_cfg: Dict):
    path = feature_cache_path(seed, exp_cfg)
    if path.exists():
        data = np.load(path, allow_pickle=True)
        return {
            "x_train": data["x_train"],
            "y_train": data["y_train"],
            "x_val": data["x_val"],
            "y_val": data["y_val"],
            "x_test": data["x_test"],
            "y_test": data["y_test"],
            "preprocess_time_sec": float(data["preprocess_time_sec"]),
            "feature_shape": tuple(data["feature_shape"]),
        }

    x_train, y_train, x_val, y_val, x_test, y_test = load_or_build_raw_pool(seed)

    start = time.perf_counter()
    fx_train = hybrid_converter(
        x_train,
        seed=seed,
        measurement_type=exp_cfg["measurement_type"],
        entanglement=exp_cfg["entanglement"]
    )
    fx_val = hybrid_converter(
        x_val,
        seed=seed,
        measurement_type=exp_cfg["measurement_type"],
        entanglement=exp_cfg["entanglement"]
    )
    fx_test = hybrid_converter(
        x_test,
        seed=seed,
        measurement_type=exp_cfg["measurement_type"],
        entanglement=exp_cfg["entanglement"]
    )
    preprocess_time = time.perf_counter() - start

    feature_shape = np.array(fx_train.shape[1:], dtype=np.int64)

    save_npz(
        path,
        x_train=fx_train, y_train=y_train,
        x_val=fx_val, y_val=y_val,
        x_test=fx_test, y_test=y_test,
        preprocess_time_sec=np.array(preprocess_time, dtype=np.float64),
        feature_shape=feature_shape,
    )

    return {
        "x_train": fx_train,
        "y_train": y_train,
        "x_val": fx_val,
        "y_val": y_val,
        "x_test": fx_test,
        "y_test": y_test,
        "preprocess_time_sec": preprocess_time,
        "feature_shape": tuple(fx_train.shape[1:]),
    }

# ============================================================
# SHARED MLP HEAD
# ============================================================
class HybridModel(nn.Module):
    def __init__(self, in_dim: int, num_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, num_classes),
        )

    def forward(self, x):
        return self.net(x)

def eval_loader(model, loader, criterion):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)
    return loss_sum / total, correct / total

def train_model(Xtr, Ytr, Xva, Yva, Xte, Yte):
    train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(Xva, Yva), batch_size=64, shuffle=False)
    test_loader = DataLoader(TensorDataset(Xte, Yte), batch_size=64, shuffle=False)

    in_dim = int(np.prod(Xtr.shape[1:]))
    model = HybridModel(in_dim=in_dim).to(device_torch)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_acc = -1.0
    best_state = None
    best_train_acc = 0.0
    best_train_loss = 0.0
    best_val_loss = 0.0

    train_start = time.perf_counter()

    for epoch in range(1, n_epochs + 1):
        model.train()
        total, correct, loss_sum = 0, 0, 0.0

        for xb, yb in train_loader:
            xb = xb.to(device_torch)
            yb = yb.to(device_torch)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * yb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += yb.size(0)

        train_loss = loss_sum / total
        train_acc = correct / total
        val_loss, val_acc = eval_loader(model, val_loader, criterion)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_train_loss = train_loss
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:03d}/{n_epochs} | "
            f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.3f}"
        )

    training_time = time.perf_counter() - train_start

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_acc = eval_loader(model, test_loader, criterion)

    return {
        "best_train_loss": best_train_loss,
        "best_train_acc": best_train_acc,
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "training_time_sec": training_time,
    }

# ============================================================
# EXPERIMENT RUNNERS
# ============================================================
def run_one_from_cache(seed: int, exp_cfg: Dict):
    set_seed(seed)

    cache = load_or_build_feature_cache(seed=seed, exp_cfg=exp_cfg)

    Xtr_np = cache["x_train"]
    Ytr_np = cache["y_train"]
    Xva_np = cache["x_val"]
    Yva_np = cache["y_val"]
    Xte_np = cache["x_test"]
    Yte_np = cache["y_test"]

    pca_metrics = compute_pca_metrics(Xtr_np, max_components=20, variance_threshold=0.90)

    Xtr = torch.tensor(Xtr_np, dtype=torch.float32)
    Ytr = torch.tensor(Ytr_np, dtype=torch.long)
    Xva = torch.tensor(Xva_np, dtype=torch.float32)
    Yva = torch.tensor(Yva_np, dtype=torch.long)
    Xte = torch.tensor(Xte_np, dtype=torch.float32)
    Yte = torch.tensor(Yte_np, dtype=torch.long)

    metrics = train_model(Xtr, Ytr, Xva, Yva, Xte, Yte)
    metrics.update(pca_metrics)

    metrics["preprocess_time_sec"] = cache["preprocess_time_sec"]
    metrics["feature_shape"] = cache["feature_shape"]
    metrics["measurement_type"] = exp_cfg["measurement_type"]
    metrics["entanglement"] = exp_cfg["entanglement"]

    image_size = 28
    patch_len = patch_size * patch_size
    n_qubits, features_per_circuit = get_features_per_circuit(patch_len, exp_cfg["measurement_type"])
    num_circuits = int(math.ceil(get_num_output_channels(patch_size) / features_per_circuit))
    out_h, out_w = output_shape_for_patching(image_size, patch_size, stride)

    metrics["patch_size"] = patch_size
    metrics["stride"] = stride
    metrics["n_qubits"] = n_qubits
    metrics["features_per_circuit"] = features_per_circuit
    metrics["num_circuits"] = num_circuits
    metrics["patches_per_image"] = out_h * out_w

    return metrics

def summarize_results(results, exp_cfg):
    print(f"\n===== SUMMARY | {exp_cfg['name']} =====")

    keys = [
        "best_train_acc",
        "best_val_acc",
        "test_acc",
        "best_train_loss",
        "best_val_loss",
        "test_loss",
        "preprocess_time_sec",
        "training_time_sec",
        "pca_top5_var",
        "pca_top10_var",
        "pca_top20_var",
        "pca_num_for_90",
        "pca_effective_rank",
    ]

    for k in keys:
        mu, sd = mean_std([r[k] for r in results])
        print(f"{k:20s}: {mu:.6f} ± {sd:.6f}")

    print(f"measurement_type     : {results[0]['measurement_type']}")
    print(f"entanglement         : {results[0]['entanglement']}")
    print(f"feature_shape        : {results[0]['feature_shape']}")
    print(f"patch_size           : {results[0]['patch_size']}")
    print(f"stride               : {results[0]['stride']}")
    print(f"n_qubits             : {results[0]['n_qubits']}")
    print(f"features_per_circuit : {results[0]['features_per_circuit']}")
    print(f"num_circuits         : {results[0]['num_circuits']}")
    print(f"patches_per_image    : {results[0]['patches_per_image']}")

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    print(f"Using device: {device_torch}")
    print(f"Seeds: {EXPERIMENT_SEEDS}")
    print(f"n_train: {N_TRAIN}")
    print(f"Patch size: {patch_size}")
    print(f"Stride: {stride}")
    print(f"Output channels: {num_output_channels}")
    print(f"Cache dir: {CACHE_DIR.resolve()}")

    validate_patch_config(28, patch_size, stride)

    out_h, out_w = output_shape_for_patching(28, patch_size, stride)
    patch_len = patch_size * patch_size
    print(f"\nFixed config: out=({out_h},{out_w}), patches/image={out_h*out_w}")

    print("\nEntanglement experiments:")
    for exp_cfg in ENTANGLEMENT_EXPERIMENTS:
        n_qubits, features_per_circuit = get_features_per_circuit(
            patch_len, exp_cfg["measurement_type"]
        )
        num_circuits = int(math.ceil(get_num_output_channels(patch_size) / features_per_circuit))
        print(
            f"  {exp_cfg['name']}: "
            f"measurement={exp_cfg['measurement_type']}, "
            f"entanglement={exp_cfg['entanglement']}, "
            f"n_qubits={n_qubits}, "
            f"features/circuit={features_per_circuit}, "
            f"circuits/patch={num_circuits}"
        )

    # Build raw cache once per seed
    for seed in EXPERIMENT_SEEDS:
        print(f"\nPreparing shared raw pool for seed={seed}")
        load_or_build_raw_pool(seed)

        for exp_cfg in ENTANGLEMENT_EXPERIMENTS:
            print(
                f"Preparing/loading hybrid cache for seed={seed}, "
                f"measurement={exp_cfg['measurement_type']}, "
                f"entanglement={exp_cfg['entanglement']}"
            )
            load_or_build_feature_cache(seed=seed, exp_cfg=exp_cfg)

    all_results = {}

    for exp_cfg in ENTANGLEMENT_EXPERIMENTS:
        exp_results = []

        for seed in EXPERIMENT_SEEDS:
            print(f"\n\n########## {exp_cfg['name']} | seed={seed} ##########")
            result = run_one_from_cache(seed=seed, exp_cfg=exp_cfg)
            exp_results.append(result)

            print("\n--- Run result ---")
            for k, v in result.items():
                print(f"{k}: {v}")

        all_results[exp_cfg["name"]] = exp_results
        summarize_results(exp_results, exp_cfg)

    print("\n\n================ FINAL AGGREGATED SUMMARY ================")
    for exp_name, runs in all_results.items():
        test_mu, test_sd = mean_std([r["test_acc"] for r in runs])
        val_mu, val_sd = mean_std([r["best_val_acc"] for r in runs])
        prep_mu, prep_sd = mean_std([r["preprocess_time_sec"] for r in runs])
        train_mu, train_sd = mean_std([r["training_time_sec"] for r in runs])
        pca_rank_mu, pca_rank_sd = mean_std([r["pca_effective_rank"] for r in runs])
        pca90_mu, pca90_sd = mean_std([r["pca_num_for_90"] for r in runs])

        meta = runs[0]
        print(
            f"{exp_name:12s} | "
            f"meas={meta['measurement_type']} | "
            f"ent={meta['entanglement']} | "
            f"val_acc={val_mu:.4f}±{val_sd:.4f} | "
            f"test_acc={test_mu:.4f}±{test_sd:.4f} | "
            f"pca_rank={pca_rank_mu:.2f}±{pca_rank_sd:.2f} | "
            f"pca90={pca90_mu:.2f}±{pca90_sd:.2f} | "
            f"prep_time={prep_mu:.2f}±{prep_sd:.2f}s | "
            f"train_time={train_mu:.2f}±{train_sd:.2f}s"
        )

Using device: cpu
Seeds: [246, 247, 248]
n_train: 500
Patch size: 4
Stride: 4
Output channels: 4
Cache dir: C:\Users\Asus\qml\coursework\feature_cache_hybrid_entanglement_experiments_ryxry

Fixed config: out=(7,7), patches/image=49

Entanglement experiments:
  ent_none: measurement=Z, entanglement=NONE, n_qubits=4, features/circuit=4, circuits/patch=1
  ent_chain: measurement=Z, entanglement=CHAIN, n_qubits=4, features/circuit=4, circuits/patch=1
  ent_ring: measurement=Z, entanglement=RING, n_qubits=4, features/circuit=4, circuits/patch=1
  ent_full: measurement=Z, entanglement=FULL, n_qubits=4, features/circuit=4, circuits/patch=1

Preparing shared raw pool for seed=246
Preparing/loading hybrid cache for seed=246, measurement=Z, entanglement=NONE

=== Hybrid preprocessing started | measurement=Z | entanglement=NONE ===
Processing image 500/500
=== Hybrid preprocessing finished | measurement=Z | entanglement=NONE ===


=== Hybrid preprocessing started | measurement=Z | entanglement=NO